In [ ]:
# ============================================================
# 03 — CALIBRATION AREA
# Neural SDM — Sapajus libidinosus
# ============================================================

from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt


# ------------------------------------------------------------
# Project directories
# ------------------------------------------------------------

PROJECT_DIR = Path.cwd().parent

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

print("Project directory:")
print(PROJECT_DIR)

print("\nProcessed data directory:")
print(PROCESSED_DIR)

In [ ]:
# ============================================================
# LOAD FINAL MODELING OCCURRENCES
# ============================================================

presence_path = (
    PROCESSED_DIR /
    "sapajus_occurrences_modeling.gpkg"
)

if not presence_path.exists():
    raise FileNotFoundError(
        f"Modeling occurrence file not found: {presence_path}"
    )

occ = gpd.read_file(presence_path)

print("Occurrences loaded:", len(occ))
print("CRS:", occ.crs)

display(
    occ[
        [
            "gbifID",
            "decimalLongitude",
            "decimalLatitude",
            "BIO1",
            "BIO4",
            "BIO12",
            "BIO15"
        ]
    ].head()
)

In [ ]:
# ============================================================
# INPUT INTEGRITY CHECK
# ============================================================

required_columns = [
    "gbifID",
    "decimalLongitude",
    "decimalLatitude",
    "BIO1",
    "BIO4",
    "BIO12",
    "BIO15",
    "raster_row",
    "raster_col"
]

missing_columns = [
    col
    for col in required_columns
    if col not in occ.columns
]

if missing_columns:
    raise ValueError(
        f"Missing required columns: {missing_columns}"
    )

if len(occ) != 349:
    raise ValueError(
        f"Expected 349 modeling occurrences, found {len(occ)}."
    )

duplicated_cells = occ.duplicated(
    subset=["raster_row", "raster_col"]
).sum()

if duplicated_cells != 0:
    raise ValueError(
        f"Found {duplicated_cells} duplicated WorldClim cells."
    )

if occ[
    ["BIO1", "BIO4", "BIO12", "BIO15"]
].isna().any().any():
    raise ValueError(
        "Missing environmental values detected."
    )

print("Input integrity: OK")
print("Occurrences:", len(occ))
print("Duplicated WorldClim cells:", duplicated_cells)
print("Environmental NoData: 0")

In [ ]:
# ============================================================
# GEOGRAPHIC EXTENT OF MODELING OCCURRENCES
# ============================================================

extent = {
    "min_longitude": occ["decimalLongitude"].min(),
    "max_longitude": occ["decimalLongitude"].max(),
    "min_latitude": occ["decimalLatitude"].min(),
    "max_latitude": occ["decimalLatitude"].max()
}

print("=== MODELING OCCURRENCE EXTENT ===")

for key, value in extent.items():
    print(f"{key}: {value:.4f}")

In [ ]:
# ============================================================
# FINAL PRESENCE DISTRIBUTION
# ============================================================

fig, ax = plt.subplots(figsize=(8, 9))

ax.scatter(
    occ["decimalLongitude"],
    occ["decimalLatitude"],
    s=20,
    alpha=0.65
)

ax.set_title(
    "Sapajus libidinosus — modeling occurrences"
)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

ax.grid(alpha=0.2)

plt.show()

In [ ]:
# ============================================================
# LOCATE RESOLVE ECOREGIONS DATA
# ============================================================

import zipfile

RAW_DIR = DATA_DIR / "raw"

ecoregions_zip = (
    RAW_DIR /
    "Ecoregions2017.zip"
)

if not ecoregions_zip.exists():
    raise FileNotFoundError(
        f"Ecoregions archive not found: {ecoregions_zip}"
    )

with zipfile.ZipFile(ecoregions_zip, "r") as z:
    files = z.namelist()

print("Archive:")
print(ecoregions_zip)

print("\nFiles inside archive:")
for file in files:
    print(file)

In [ ]:
# ============================================================
# LOAD RESOLVE ECOREGIONS
# ============================================================

ecoregions = gpd.read_file(
    f"zip://{ecoregions_zip}"
)

print("Ecoregions loaded:", len(ecoregions))
print("CRS:", ecoregions.crs)

print("\nAvailable columns:")
for col in ecoregions.columns:
    print(col)

print("\nFirst records:")
display(ecoregions.head())

In [ ]:
# ============================================================
# ECOREGION GEOMETRY CHECK
# ============================================================

print("Geometry types:")
print(ecoregions.geometry.geom_type.value_counts())

print("\nMissing geometries:")
print(ecoregions.geometry.isna().sum())

print("\nInvalid geometries:")
print((~ecoregions.geometry.is_valid).sum())

print("\nBounds:")
print(ecoregions.total_bounds)

In [ ]:
# ============================================================
# CRS COMPATIBILITY
# ============================================================

print("Occurrence CRS:", occ.crs)
print("Ecoregion CRS:", ecoregions.crs)

if occ.crs == ecoregions.crs:
    print("\nCRS match: OK")
else:
    print("\nCRS differ — reprojection will be required.")

In [ ]:
# ============================================================
# REPAIR INVALID ECOREGION GEOMETRIES
# ============================================================

ecoregions_clean = ecoregions.copy()

n_invalid_before = (~ecoregions_clean.geometry.is_valid).sum()

ecoregions_clean["geometry"] = (
    ecoregions_clean.geometry.make_valid()
)

n_invalid_after = (~ecoregions_clean.geometry.is_valid).sum()

print("Invalid geometries before:", n_invalid_before)
print("Invalid geometries after:", n_invalid_after)

print("\nGeometry types after repair:")
print(
    ecoregions_clean.geometry
    .geom_type
    .value_counts()
)

In [ ]:
# ============================================================
# ASSIGN ECOREGIONS TO MODELING OCCURRENCES
# ============================================================

occ_ecoregions = gpd.sjoin(
    occ,
    ecoregions_clean[
        [
            "ECO_NAME",
            "BIOME_NAME",
            "REALM",
            "ECO_ID",
            "geometry"
        ]
    ],
    how="left",
    predicate="intersects"
)

print("Input occurrences:", len(occ))
print("Rows after spatial join:", len(occ_ecoregions))

print(
    "Occurrences without ecoregion:",
    occ_ecoregions["ECO_NAME"].isna().sum()
)

print(
    "Unique ecoregions represented:",
    occ_ecoregions["ECO_NAME"].nunique()
)

In [ ]:
# ============================================================
# OCCURRENCES BY ECOREGION
# ============================================================

ecoregion_summary = (
    occ_ecoregions
    .groupby(
        ["ECO_NAME", "BIOME_NAME"],
        dropna=False
    )
    .size()
    .reset_index(name="n_occurrences")
    .sort_values(
        "n_occurrences",
        ascending=False
    )
    .reset_index(drop=True)
)

display(ecoregion_summary)

In [ ]:
# ============================================================
# MAP ECOREGIONS REPRESENTED BY MODELING OCCURRENCES
# ============================================================

occupied_names = (
    occ_ecoregions["ECO_NAME"]
    .dropna()
    .unique()
)

occupied_ecoregions = (
    ecoregions_clean[
        ecoregions_clean["ECO_NAME"].isin(occupied_names)
    ]
    .copy()
)

print("Occupied ecoregions:", len(occupied_ecoregions))

fig, ax = plt.subplots(figsize=(11, 11))

occupied_ecoregions.plot(
    ax=ax,
    alpha=0.35,
    edgecolor="black",
    linewidth=0.6
)

occ.plot(
    ax=ax,
    markersize=10,
    alpha=0.7
)

# Zoom around the occurrence extent
margin = 2

ax.set_xlim(
    occ["decimalLongitude"].min() - margin,
    occ["decimalLongitude"].max() + margin
)

ax.set_ylim(
    occ["decimalLatitude"].min() - margin,
    occ["decimalLatitude"].max() + margin
)

ax.set_title(
    "Ecoregions represented by Sapajus libidinosus occurrences"
)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

plt.show()

In [ ]:
# ============================================================
# ECOREGION AREA VS OCCURRENCE SUPPORT
# ============================================================

# South America Albers Equal Area Conic
AREA_CRS = "ESRI:102033"

occupied_area = occupied_ecoregions.to_crs(AREA_CRS).copy()

occupied_area["area_km2"] = (
    occupied_area.geometry.area / 1_000_000
)

# Occurrence counts already calculated
ecoregion_diagnostic = (
    ecoregion_summary
    .merge(
        occupied_area[
            ["ECO_NAME", "area_km2"]
        ],
        on="ECO_NAME",
        how="left"
    )
)

ecoregion_diagnostic["pct_occurrences"] = (
    ecoregion_diagnostic["n_occurrences"]
    / len(occ)
    * 100
)

ecoregion_diagnostic["occurrences_per_10000_km2"] = (
    ecoregion_diagnostic["n_occurrences"]
    / ecoregion_diagnostic["area_km2"]
    * 10_000
)

ecoregion_diagnostic = (
    ecoregion_diagnostic
    .sort_values(
        "n_occurrences",
        ascending=False
    )
    .reset_index(drop=True)
)

display(
    ecoregion_diagnostic[
        [
            "ECO_NAME",
            "BIOME_NAME",
            "n_occurrences",
            "pct_occurrences",
            "area_km2",
            "occurrences_per_10000_km2"
        ]
    ]
)

In [ ]:
# ============================================================
# TOTAL AREA OF OCCUPIED ECOREGIONS
# ============================================================

occupied_union = occupied_area.geometry.union_all()

total_area_km2 = occupied_union.area / 1_000_000

print(
    "Total area represented by the 17 ecoregions:",
    f"{total_area_km2:,.0f} km²"
)

print(
    "Occurrences represented:",
    len(occ)
)

In [ ]:
# ============================================================
# CANDIDATE ACCESSIBILITY BUFFERS
# ============================================================

# Equal-area projection for South America
occ_projected = occ.to_crs("ESRI:102033")

buffer_distances_km = [
    100,
    200,
    300,
    500
]

buffer_results = []

for distance_km in buffer_distances_km:

    distance_m = distance_km * 1000

    buffered_geometry = (
        occ_projected.geometry
        .buffer(distance_m)
        .union_all()
    )

    area_km2 = (
        buffered_geometry.area /
        1_000_000
    )

    buffer_results.append({
        "buffer_km": distance_km,
        "area_km2": area_km2
    })


buffer_summary = pd.DataFrame(buffer_results)

display(buffer_summary)

In [ ]:
# ============================================================
# VISUALIZE CANDIDATE BUFFERS
# ============================================================

for distance_km in buffer_distances_km:

    distance_m = distance_km * 1000

    candidate_M = (
        occ_projected.geometry
        .buffer(distance_m)
        .union_all()
    )

    candidate_gdf = gpd.GeoDataFrame(
        geometry=[candidate_M],
        crs=occ_projected.crs
    ).to_crs("EPSG:4326")

    fig, ax = plt.subplots(
        figsize=(9, 10)
    )

    candidate_gdf.plot(
        ax=ax,
        alpha=0.25,
        edgecolor="black"
    )

    occ.plot(
        ax=ax,
        markersize=8,
        alpha=0.65
    )

    ax.set_title(
        f"Candidate accessibility area — {distance_km} km"
    )

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

    plt.show()

In [ ]:
# ============================================================
# INTERSECT CANDIDATE BUFFERS WITH OCCUPIED ECOREGIONS
# ============================================================

candidate_results = []

for distance_km in [200, 300]:

    # Create buffer in projected CRS
    buffer_geom = (
        occ_projected.geometry
        .buffer(distance_km * 1000)
        .union_all()
    )

    buffer_gdf = gpd.GeoDataFrame(
        {"buffer_km": [distance_km]},
        geometry=[buffer_geom],
        crs=occ_projected.crs
    )

    # Occupied ecoregions are already in the same projected CRS
    eco_intersection = gpd.overlay(
        occupied_area[
            ["ECO_NAME", "BIOME_NAME", "geometry"]
        ],
        buffer_gdf,
        how="intersection"
    )

    # Calculate intersected area
    eco_intersection["intersection_area_km2"] = (
        eco_intersection.geometry.area / 1_000_000
    )

    candidate_results.append(
        eco_intersection
    )

    print(
        f"{distance_km} km:",
        f"{eco_intersection.geometry.union_all().area / 1_000_000:,.0f} km²"
    )

In [ ]:
# ============================================================
# VISUALIZE BIOGEOGRAPHICALLY CONSTRAINED CANDIDATES
# ============================================================

for eco_intersection in candidate_results:

    distance_km = int(
        eco_intersection["buffer_km"].iloc[0]
    )

    candidate_wgs84 = (
        eco_intersection
        .to_crs("EPSG:4326")
    )

    fig, ax = plt.subplots(
        figsize=(9, 10)
    )

    candidate_wgs84.plot(
        ax=ax,
        alpha=0.30,
        edgecolor="black",
        linewidth=0.5
    )

    occ.plot(
        ax=ax,
        markersize=8,
        alpha=0.7
    )

    ax.set_title(
        f"Candidate M — {distance_km} km × occupied ecoregions"
    )

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

    plt.show()

In [ ]:
# ============================================================
# BIOGEOGRAPHIC AUDIT BY STATE
# ============================================================

state_summary = (
    occ
    .groupby("stateProvince", dropna=False)
    .size()
    .reset_index(name="n_occurrences")
    .sort_values("n_occurrences", ascending=False)
    .reset_index(drop=True)
)

state_summary["pct_occurrences"] = (
    state_summary["n_occurrences"]
    / len(occ)
    * 100
)

display(state_summary)

In [ ]:
# ============================================================
# STATE × ECOREGION AUDIT
# ============================================================

state_ecoregion = (
    occ_ecoregions
    .groupby(
        ["stateProvince", "ECO_NAME"],
        dropna=False
    )
    .size()
    .reset_index(name="n_occurrences")
    .sort_values(
        "n_occurrences",
        ascending=False
    )
    .reset_index(drop=True)
)

display(state_ecoregion)

In [ ]:
# ============================================================
# FLAG RECORDS FOR BIOGEOGRAPHIC / TAXONOMIC REVIEW
# ============================================================

review_ecoregions = [
    "Pantanal",
    "Serra do Mar coastal forests",
    "Tocantins/Pindare moist forests",
    "Xingu-Tocantins-Araguaia moist forests"
]

review_states = [
    "Mato Grosso",
    "Mato Grosso do Sul",
    "Pará",
    "São Paulo"
]

occ_review = occ_ecoregions[
    occ_ecoregions["stateProvince"].isin(review_states)
    |
    occ_ecoregions["ECO_NAME"].isin(review_ecoregions)
].copy()

review_table = occ_review[
    [
        "gbifID",
        "stateProvince",
        "ECO_NAME",
        "decimalLongitude",
        "decimalLatitude",
        "year",
        "institutionCode",
        "basisOfRecord",
        "coordinateUncertaintyInMeters",
        "issue"
    ]
].sort_values(
    ["stateProvince", "ECO_NAME"]
)

print(
    "Records selected for biogeographic review:",
    len(review_table)
)

display(review_table)

In [ ]:
# ============================================================
# PRIORITY BIOGEOGRAPHIC REVIEW
# ============================================================

priority_review = occ_ecoregions[
    (
        occ_ecoregions["ECO_NAME"].eq("Pantanal")
    )
    |
    (
        (
            occ_ecoregions["stateProvince"].eq("Pará")
        )
        &
        (
            occ_ecoregions["ECO_NAME"]
            .eq("Tocantins/Pindare moist forests")
        )
    )
    |
    (
        occ_ecoregions["ECO_NAME"]
        .eq("Serra do Mar coastal forests")
    )
    |
    (
        occ_ecoregions["ECO_NAME"]
        .eq("Xingu-Tocantins-Araguaia moist forests")
    )
].copy()


priority_review["qc_biogeographic_status"] = "pending"

priority_review["qc_biogeographic_reason"] = ""

priority_review["qc_manual_review"] = True


print(
    "Priority records:",
    len(priority_review)
)

display(
    priority_review[
        [
            "gbifID",
            "species",
            "stateProvince",
            "ECO_NAME",
            "decimalLongitude",
            "decimalLatitude",
            "year",
            "institutionCode",
            "basisOfRecord",
            "coordinateUncertaintyInMeters",
            "qc_biogeographic_status"
        ]
    ]
)

In [ ]:
# ============================================================
# QUERY ORIGINAL GBIF RECORDS — PRIORITY REVIEW
# ============================================================

import requests
import pandas as pd

GBIF_OCCURRENCE_API = (
    "https://api.gbif.org/v1/occurrence/"
)

priority_ids = (
    priority_review["gbifID"]
    .astype(str)
    .tolist()
)

gbif_review_records = []

for gbif_id in priority_ids:

    url = GBIF_OCCURRENCE_API + gbif_id

    response = requests.get(
        url,
        timeout=30
    )

    if response.status_code != 200:
        print(
            f"Failed: {gbif_id} "
            f"(HTTP {response.status_code})"
        )
        continue

    record = response.json()

    gbif_review_records.append({

        "gbifID": gbif_id,

        "scientificName":
            record.get("scientificName"),

        "acceptedScientificName":
            record.get("acceptedScientificName"),

        "taxonKey":
            record.get("taxonKey"),

        "identifiedBy":
            record.get("identifiedBy"),

        "identificationVerificationStatus":
            record.get(
                "identificationVerificationStatus"
            ),

        "occurrenceStatus":
            record.get("occurrenceStatus"),

        "establishmentMeans":
            record.get("establishmentMeans"),

        "degreeOfEstablishment":
            record.get("degreeOfEstablishment"),

        "locality":
            record.get("locality"),

        "stateProvince":
            record.get("stateProvince"),

        "country":
            record.get("country"),

        "decimalLongitude":
            record.get("decimalLongitude"),

        "decimalLatitude":
            record.get("decimalLatitude"),

        "coordinateUncertaintyInMeters":
            record.get(
                "coordinateUncertaintyInMeters"
            ),

        "eventDate":
            record.get("eventDate"),

        "basisOfRecord":
            record.get("basisOfRecord"),

        "institutionCode":
            record.get("institutionCode"),

        "datasetTitle":
            record.get("datasetTitle"),

        "occurrenceRemarks":
            record.get("occurrenceRemarks"),

        "identificationRemarks":
            record.get("identificationRemarks"),

        "references":
            record.get("references"),

        "issues":
            ";".join(record.get("issues", []))
    })


gbif_review = pd.DataFrame(
    gbif_review_records
)

print(
    "Records retrieved:",
    len(gbif_review),
    "/",
    len(priority_ids)
)

display(gbif_review)

In [ ]:
# ============================================================
# ORIGINAL SOURCE — SERRA DO MAR TEST RECORD
# ============================================================

test_id = "6452331056"

record = gbif_review.loc[
    gbif_review["gbifID"].eq(test_id)
].iloc[0]

print("GBIF ID:", test_id)
print("Scientific name:", record["scientificName"])
print("Event date:", record["eventDate"])
print("Coordinates:")
print(
    record["decimalLongitude"],
    record["decimalLatitude"]
)

print("\nOriginal observation:")
print(record["references"])

In [ ]:
# GBIF 6452331056 — Jaraguá, São Paulo

identification_status = "supported"
occurrence_status = "supported"
native_status = "non_native_or_introduced"
qc_biogeographic_status = "exclude_from_native_sdm"

qc_reason = (
    "Research-grade photographic occurrence of Sapajus libidinosus "
    "in Jaragua, metropolitan Sao Paulo. The species' documented "
    "native range in Sao Paulo is restricted primarily to the "
    "northern/northeastern portion of the state, while introduced "
    "populations are documented elsewhere in Sao Paulo. Excluded "
    "from native-range SDM calibration."
)

In [ ]:
# ============================================================
# SERRA DO MAR / METROPOLITAN SÃO PAULO — ORIGINAL SOURCES
# ============================================================

sp_ids = [
    "4952384761",
    "5840466626",
    "6452331056",
    "5839802298",
    "4987668403"
]

sp_review = gbif_review[
    gbif_review["gbifID"].isin(sp_ids)
].copy()

for _, row in sp_review.iterrows():

    print("=" * 70)
    print("GBIF ID:", row["gbifID"])
    print("Species:", row["scientificName"])
    print(
        "Coordinates:",
        row["decimalLongitude"],
        row["decimalLatitude"]
    )
    print("Date:", row["eventDate"])
    print(
        "Uncertainty:",
        row["coordinateUncertaintyInMeters"]
    )
    print("Identified by:", row["identifiedBy"])
    print("Original observation:")
    print(row["references"])
    print()

In [ ]:
# ============================================================
# ORIGINAL iNATURALIST LINKS — SÃO PAULO CLUSTER
# ============================================================

sp_links = (
    sp_review[
        [
            "gbifID",
            "decimalLongitude",
            "decimalLatitude",
            "eventDate",
            "references"
        ]
    ]
    .sort_values("gbifID")
    .reset_index(drop=True)
)

display(sp_links)

In [ ]:
record_id = "5839802298"

url = gbif_review.loc[
    gbif_review["gbifID"].astype(str).eq(record_id),
    "references"
].iloc[0]

print(url)

In [ ]:
for record_id in [
    "5839802298",
    "5840466626"
]:
    url = gbif_review.loc[
        gbif_review["gbifID"].astype(str).eq(record_id),
        "references"
    ].iloc[0]

    print(record_id, "->", url)

In [ ]:
# ============================================================
# RELOAD FINAL THINNED OCCURRENCE DATASET
# ============================================================

import geopandas as gpd

occurrence_path = (
    PROCESSED_DIR /
    "sapajus_occurrences_modeling.gpkg"
)

occ_thinned = gpd.read_file(occurrence_path)

print("Dataset reloaded successfully.")
print("Occurrences:", len(occ_thinned))
print("CRS:", occ_thinned.crs)

display(occ_thinned.head())

In [ ]:
# Check biogeographic QC columns

qc_columns = [
    "qc_biogeographic_status",
    "qc_biogeographic_reason"
]

for col in qc_columns:
    print(
        f"{col}:",
        "FOUND" if col in occ_thinned.columns else "NOT FOUND"
    )

In [ ]:
if "qc_biogeographic_status" not in occ_thinned.columns:
    occ_thinned["qc_biogeographic_status"] = "pending"

if "qc_biogeographic_reason" not in occ_thinned.columns:
    occ_thinned["qc_biogeographic_reason"] = None

print(
    occ_thinned["qc_biogeographic_status"]
    .value_counts(dropna=False)
)

In [ ]:
# ============================================================
# BIOGEOGRAPHIC QC — SÃO PAULO RECORDS
# ============================================================

# GBIF records manually inspected through the original
# iNaturalist observations.
sp_non_native_ids = [
    "4952384761",
    "4987668403",
    "5839802298",
    "5840466626",
    "6452331056"
]

# ------------------------------------------------------------
# 1. Identify the records
# ------------------------------------------------------------

mask_sp = (
    occ_thinned["gbifID"]
    .astype(str)
    .isin(sp_non_native_ids)
)

print(
    "São Paulo records identified:",
    mask_sp.sum()
)

# ------------------------------------------------------------
# 2. Assign biogeographic QC status
# ------------------------------------------------------------

occ_thinned.loc[
    mask_sp,
    "qc_biogeographic_status"
] = "exclude_non_native"

occ_thinned.loc[
    mask_sp,
    "qc_biogeographic_reason"
] = (
    "Manually verified occurrence of Sapajus libidinosus "
    "in São Paulo, but treated as outside the native "
    "distribution used for SDM calibration."
)

# ------------------------------------------------------------
# 3. Inspect flagged records
# ------------------------------------------------------------

columns_sp = [
    "gbifID",
    "species",
    "stateProvince",
    "decimalLongitude",
    "decimalLatitude",
    "year",
    "institutionCode",
    "coordinateUncertaintyInMeters",
    "qc_biogeographic_status",
    "qc_biogeographic_reason"
]

display(
    occ_thinned.loc[
        mask_sp,
        columns_sp
    ].sort_values("gbifID")
)

# ------------------------------------------------------------
# 4. QC summary
# ------------------------------------------------------------

print("\nBiogeographic QC status:")
print(
    occ_thinned[
        "qc_biogeographic_status"
    ].value_counts(dropna=False)
)

In [ ]:
qc_biogeographic_status = "needs_taxonomic_review"

In [ ]:
# ============================================================
# TRACE ORIGINAL SOURCE — PANTANAL RECORDS
# ============================================================

import requests
import pandas as pd

pantanal_ids = [
    "2835474141",
    "2842438415",
    "2843307795",
    "2832301198",
    "2832428892"
]

pantanal_source = []

for gbif_id in pantanal_ids:

    url = f"https://api.gbif.org/v1/occurrence/{gbif_id}"

    response = requests.get(url, timeout=30)
    response.raise_for_status()

    r = response.json()

    pantanal_source.append({
        "gbifID": gbif_id,
        "scientificName": r.get("scientificName"),
        "verbatimScientificName": r.get("verbatimScientificName"),
        "acceptedScientificName": r.get("acceptedScientificName"),

        "datasetKey": r.get("datasetKey"),
        "datasetTitle": r.get("datasetTitle"),
        "publishingOrgKey": r.get("publishingOrgKey"),
        "institutionCode": r.get("institutionCode"),
        "collectionCode": r.get("collectionCode"),

        "recordedBy": r.get("recordedBy"),
        "identifiedBy": r.get("identifiedBy"),
        "identificationVerificationStatus":
            r.get("identificationVerificationStatus"),

        "basisOfRecord": r.get("basisOfRecord"),
        "occurrenceStatus": r.get("occurrenceStatus"),

        "locality": r.get("locality"),
        "stateProvince": r.get("stateProvince"),
        "decimalLongitude": r.get("decimalLongitude"),
        "decimalLatitude": r.get("decimalLatitude"),

        "eventDate": r.get("eventDate"),

        "references": r.get("references"),
        "occurrenceRemarks": r.get("occurrenceRemarks"),
        "identificationRemarks": r.get("identificationRemarks"),

        "issues": ";".join(r.get("issues", []))
    })

pantanal_source = pd.DataFrame(pantanal_source)

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

display(pantanal_source)

In [ ]:
for _, r in pantanal_source.iterrows():

    print("=" * 70)

    print("GBIF:", r["gbifID"])
    print("Species:", r["scientificName"])
    print("Verbatim species:", r["verbatimScientificName"])

    print("\nDataset:")
    print(r["datasetTitle"])
    print("Dataset key:", r["datasetKey"])

    print("\nSource:")
    print("Recorded by:", r["recordedBy"])
    print("Identified by:", r["identifiedBy"])
    print(
        "Verification:",
        r["identificationVerificationStatus"]
    )

    print("\nLocation:")
    print(r["locality"])
    print(
        r["decimalLongitude"],
        r["decimalLatitude"]
    )

    print("\nReference:")
    print(r["references"])

    print("\nRemarks:")
    print(r["occurrenceRemarks"])
    print(r["identificationRemarks"])

    print()

In [ ]:
# ============================================================
# INSPECT COMPLETE RAW GBIF RECORD
# ============================================================

test_id = "2842438415"

url = (
    f"https://api.gbif.org/v1/occurrence/{test_id}"
)

response = requests.get(
    url,
    timeout=30
)

response.raise_for_status()

raw_record = response.json()

# Show every available field
for key in sorted(raw_record.keys()):

    value = raw_record[key]

    if value not in [
        None,
        "",
        [],
        {}
    ]:
        print(
            f"{key}: {value}"
        )

In [ ]:
# ============================================================
# SEARCH FOR SOURCE IDENTIFIERS
# ============================================================

search_terms = [
    "id",
    "identifier",
    "occurrence",
    "catalog",
    "record",
    "reference",
    "source"
]

for key, value in raw_record.items():

    key_lower = key.lower()

    if any(
        term in key_lower
        for term in search_terms
    ):
        print(
            f"{key}: {value}"
        )

In [ ]:
# ============================================================
# TRACE ORIGINAL OBSERVATION.ORG IDENTIFIER
# ============================================================

import requests

test_id = "2842438415"

response = requests.get(
    f"https://api.gbif.org/v1/occurrence/{test_id}",
    timeout=30
)

response.raise_for_status()

raw_record = response.json()

# Fields that may allow us to trace the original observation
fields = [
    "key",
    "gbifID",
    "occurrenceID",
    "catalogNumber",
    "recordNumber",
    "identifier",
    "references",
    "source",
    "datasetID",
    "datasetKey",
    "datasetTitle",
    "institutionCode",
    "collectionCode",
    "recordedBy",
    "identifiedBy",
    "identificationVerificationStatus"
]

for field in fields:
    print(
        f"{field}:",
        raw_record.get(field)
    )

In [ ]:
print("\n=== ALL ID / SOURCE-RELATED FIELDS ===")

keywords = [
    "id",
    "catalog",
    "record",
    "reference",
    "source",
    "dataset"
]

for key in sorted(raw_record):

    if any(
        word in key.lower()
        for word in keywords
    ):
        print(
            f"{key}: {raw_record[key]}"
        )

In [ ]:
# ============================================================
# RECOVER OBSERVATION.ORG LINKS — PANTANAL RECORDS
# ============================================================

import requests
import pandas as pd

pantanal_ids = [
    "2835474141",
    "2842438415",
    "2843307795",
    "2832301198",
    "2832428892"
]

source_links = []

for gbif_id in pantanal_ids:

    response = requests.get(
        f"https://api.gbif.org/v1/occurrence/{gbif_id}",
        timeout=30
    )

    response.raise_for_status()

    r = response.json()

    source_links.append({
        "gbifID": gbif_id,
        "identifier": r.get("identifier"),
        "occurrenceID": r.get("occurrenceID"),
        "recordedBy": r.get("recordedBy"),
        "eventDate": r.get("eventDate"),
        "locality": r.get("locality"),
        "longitude": r.get("decimalLongitude"),
        "latitude": r.get("decimalLatitude"),
        "uncertainty_m": r.get(
            "coordinateUncertaintyInMeters"
        ),
        "verification": r.get(
            "identificationVerificationStatus"
        )
    })

pantanal_links = pd.DataFrame(source_links)

display(pantanal_links)

In [ ]:
# ============================================================
# BIOGEOGRAPHIC QC — PANTANAL TAXONOMIC CONFLICT
# ============================================================

pantanal_conflict_ids = [
    "2835474141",
    "2842438415",
    "2843307795",
    "2832301198",
    "2832428892"
]

mask_pantanal = (
    occ_thinned["gbifID"]
    .astype(str)
    .isin(pantanal_conflict_ids)
)

occ_thinned.loc[
    mask_pantanal,
    "qc_biogeographic_status"
] = "exclude_taxonomic_conflict"

occ_thinned.loc[
    mask_pantanal,
    "qc_biogeographic_reason"
] = (
    "Occurrence attributed to Sapajus libidinosus by the "
    "source platform but located within the documented "
    "native distribution of Sapajus cay in the Pantanal; "
    "excluded from S. libidinosus SDM calibration because "
    "of geographic/taxonomic conflict."
)

print(
    occ_thinned[
        "qc_biogeographic_status"
    ].value_counts(dropna=False)
)

display(
    occ_thinned.loc[
        mask_pantanal,
        [
            "gbifID",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "qc_biogeographic_status",
            "qc_biogeographic_reason"
        ]
    ]
)

In [ ]:
# ============================================================
# INSPECT NORTHERN PARA RECORDS
# ============================================================

para_ids = [
    "6510789448",
    "6510500833"
]

para_records = occ_thinned[
    occ_thinned["gbifID"].astype(str).isin(para_ids)
].copy()

cols = [
    "gbifID",
    "scientificName",
    "stateProvince",
    "decimalLongitude",
    "decimalLatitude",
    "coordinateUncertaintyInMeters",
    "year",
    "basisOfRecord",
    "institutionCode"
]

display(para_records[cols])

In [ ]:
import requests
import pandas as pd

para_ids = [
    "6510789448",
    "6510500833"
]

records = []

for gbif_id in para_ids:

    url = f"https://api.gbif.org/v1/occurrence/{gbif_id}"
    r = requests.get(url, timeout=30)
    r.raise_for_status()
    data = r.json()

    records.append({
        "gbifID": data.get("gbifID"),
        "scientificName": data.get("scientificName"),
        "acceptedScientificName": data.get("acceptedScientificName"),
        "taxonKey": data.get("taxonKey"),
        "occurrenceID": data.get("occurrenceID"),
        "identifier": data.get("identifier"),
        "datasetKey": data.get("datasetKey"),
        "datasetTitle": data.get("datasetTitle"),
        "institutionCode": data.get("institutionCode"),
        "collectionCode": data.get("collectionCode"),
        "recordedBy": data.get("recordedBy"),
        "identifiedBy": data.get("identifiedBy"),
        "identificationVerificationStatus":
            data.get("identificationVerificationStatus"),
        "basisOfRecord": data.get("basisOfRecord"),
        "occurrenceStatus": data.get("occurrenceStatus"),
        "locality": data.get("locality"),
        "stateProvince": data.get("stateProvince"),
        "decimalLongitude": data.get("decimalLongitude"),
        "decimalLatitude": data.get("decimalLatitude"),
        "coordinateUncertaintyInMeters":
            data.get("coordinateUncertaintyInMeters"),
        "eventDate": data.get("eventDate"),
        "references": data.get("references"),
        "occurrenceRemarks": data.get("occurrenceRemarks"),
        "identificationRemarks": data.get("identificationRemarks"),
        "issues": ";".join(data.get("issues", []))
    })

para_metadata = pd.DataFrame(records)

display(para_metadata.T)

In [ ]:
for gbif_id in para_ids:

    print("\n" + "=" * 70)
    print("GBIF:", gbif_id)

    data = requests.get(
        f"https://api.gbif.org/v1/occurrence/{gbif_id}",
        timeout=30
    ).json()

    for key, value in data.items():

        key_lower = key.lower()

        if any(term in key_lower for term in [
            "identifier",
            "occurrence",
            "record",
            "reference",
            "dataset",
            "institution",
            "collection",
            "identified",
            "verification",
            "remark",
            "source"
        ]):
            print(f"{key}: {value}")

In [ ]:
# ============================================================
# BIOGEOGRAPHIC QC — PARÁ RECORDS
# ============================================================

para_uncertain_ids = [
    "6510789448",
    "6510500833"
]

mask_para = (
    occ_thinned["gbifID"]
    .astype(str)
    .isin(para_uncertain_ids)
)

# ------------------------------------------------------------
# Assign QC status
# ------------------------------------------------------------

occ_thinned.loc[
    mask_para,
    "qc_biogeographic_status"
] = "exclude_taxonomic_uncertainty"

occ_thinned.loc[
    mask_para,
    "qc_biogeographic_reason"
] = (
    "SISBio/ICMBio occurrence from northeastern Pará near "
    "the northern biogeographic limit of Sapajus libidinosus. "
    "The record lacks a traceable taxonomic identification "
    "or verification, and published distribution sources are "
    "not concordant for this region. Excluded conservatively "
    "from the primary SDM calibration."
)

# ------------------------------------------------------------
# Inspect
# ------------------------------------------------------------

display(
    occ_thinned.loc[
        mask_para,
        [
            "gbifID",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "year",
            "institutionCode",
            "qc_biogeographic_status",
            "qc_biogeographic_reason"
        ]
    ]
)

print("\nBiogeographic QC status:")

print(
    occ_thinned[
        "qc_biogeographic_status"
    ].value_counts(dropna=False)
)

In [ ]:
# ============================================================
# BIOGEOGRAPHIC QC — TOCANTINS RECORD
# TRACE ORIGINAL iNATURALIST OBSERVATION
# ============================================================

import requests
import pandas as pd

tocantins_id = "6129889104"

# ------------------------------------------------------------
# 1. Retrieve complete GBIF record
# ------------------------------------------------------------

response = requests.get(
    f"https://api.gbif.org/v1/occurrence/{tocantins_id}",
    timeout=30
)

response.raise_for_status()

toc_raw = response.json()


# ------------------------------------------------------------
# 2. Extract relevant metadata
# ------------------------------------------------------------

toc_metadata = {
    "gbifID": toc_raw.get("gbifID"),
    "scientificName": toc_raw.get("scientificName"),
    "acceptedScientificName":
        toc_raw.get("acceptedScientificName"),
    "occurrenceID": toc_raw.get("occurrenceID"),
    "identifier": toc_raw.get("identifier"),
    "references": toc_raw.get("references"),
    "datasetKey": toc_raw.get("datasetKey"),
    "datasetName": toc_raw.get("datasetName"),
    "recordedBy": toc_raw.get("recordedBy"),
    "identifiedBy": toc_raw.get("identifiedBy"),
    "identificationVerificationStatus":
        toc_raw.get("identificationVerificationStatus"),
    "basisOfRecord": toc_raw.get("basisOfRecord"),
    "occurrenceStatus": toc_raw.get("occurrenceStatus"),
    "stateProvince": toc_raw.get("stateProvince"),
    "locality": toc_raw.get("locality"),
    "decimalLongitude": toc_raw.get("decimalLongitude"),
    "decimalLatitude": toc_raw.get("decimalLatitude"),
    "coordinateUncertaintyInMeters":
        toc_raw.get("coordinateUncertaintyInMeters"),
    "eventDate": toc_raw.get("eventDate"),
    "occurrenceRemarks":
        toc_raw.get("occurrenceRemarks"),
    "identificationRemarks":
        toc_raw.get("identificationRemarks"),
    "issues": ";".join(toc_raw.get("issues", []))
}

display(
    pd.DataFrame(
        toc_metadata.items(),
        columns=["field", "value"]
    )
)


# ------------------------------------------------------------
# 3. Print possible source identifiers
# ------------------------------------------------------------

print("\n=== SOURCE / IDENTIFICATION FIELDS ===")

keywords = [
    "identifier",
    "occurrence",
    "reference",
    "dataset",
    "record",
    "identified",
    "verification",
    "remark"
]

for key in sorted(toc_raw):

    if any(
        word in key.lower()
        for word in keywords
    ):
        print(f"{key}: {toc_raw[key]}")

In [ ]:
# ============================================================
# BIOGEOGRAPHIC QC — TOCANTINS RECORD
# ============================================================

tocantins_id = "6129889104"

mask_tocantins = (
    occ_thinned["gbifID"]
    .astype(str)
    .eq(tocantins_id)
)

occ_thinned.loc[
    mask_tocantins,
    "qc_biogeographic_status"
] = "retain_native"

occ_thinned.loc[
    mask_tocantins,
    "qc_biogeographic_reason"
] = (
    "Occurrence from Araguaina, Tocantins, within the documented "
    "native distribution of Sapajus libidinosus. Original iNaturalist "
    "observation contains photographic evidence and reached Research "
    "Grade with community identification as S. libidinosus. Retained "
    "for SDM calibration."
)

display(
    occ_thinned.loc[
        mask_tocantins,
        [
            "gbifID",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "year",
            "institutionCode",
            "qc_biogeographic_status",
            "qc_biogeographic_reason"
        ]
    ]
)

print(
    "\nBiogeographic QC status:\n",
    occ_thinned["qc_biogeographic_status"]
    .value_counts(dropna=False)
)

In [ ]:
print(pending.columns.tolist())

In [ ]:
# ============================================================
# FINAL BIOGEOGRAPHIC QC — STATE DIAGNOSTIC
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# 1. Select pending occurrences
# ------------------------------------------------------------

pending = occ_thinned.loc[
    occ_thinned["qc_biogeographic_status"].eq("pending")
].copy()

print("Pending occurrences:", len(pending))


# ------------------------------------------------------------
# 2. Create standardized state variable
# ------------------------------------------------------------

state_fix = {
    "Goias": "Goiás",
    "Maranhao": "Maranhão"
}

pending["state_qc"] = (
    pending["stateProvince"]
    .replace(state_fix)
    .fillna("Unknown")
)

print(
    "\nStandardized states:",
    pending["state_qc"].nunique()
)


# ------------------------------------------------------------
# 3. State summary
# ------------------------------------------------------------

state_summary = (
    pending
    .groupby(
        "state_qc",
        dropna=False
    )
    .agg(
        n_occurrences=(
            "gbifID",
            "size"
        ),
        median_nn_km=(
            "nearest_neighbor_km",
            "median"
        ),
        max_nn_km=(
            "nearest_neighbor_km",
            "max"
        ),
        median_uncertainty_m=(
            "coordinateUncertaintyInMeters",
            "median"
        )
    )
    .reset_index()
    .sort_values(
        "n_occurrences",
        ascending=False
    )
)

print("\n=== STATE SUMMARY ===")

display(state_summary)


# ------------------------------------------------------------
# 4. Spatially unusual pending records
# ------------------------------------------------------------

pending_suspicious = (
    pending.loc[
        (
            pending["qc_spatial_isolation"].eq(True)
            |
            pending["nearest_neighbor_km"].gt(50)
        ),
        [
            "gbifID",
            "scientificName",
            "state_qc",
            "decimalLongitude",
            "decimalLatitude",
            "nearest_neighbor_km",
            "coordinateUncertaintyInMeters",
            "year",
            "institutionCode",
            "basisOfRecord"
        ]
    ]
    .sort_values(
        "nearest_neighbor_km",
        ascending=False
    )
)

print(
    "\nSpatially unusual pending records:",
    len(pending_suspicious)
)

display(pending_suspicious)

In [ ]:
# ============================================================
# FINAL BIOGEOGRAPHIC QC — PRIORITY SCREENING
# ============================================================

import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Work only with pending records
# ------------------------------------------------------------

qc_review = pending.copy()


# ------------------------------------------------------------
# 2. Diagnostic flags
# ------------------------------------------------------------

# Strong spatial isolation
qc_review["flag_high_isolation"] = (
    qc_review["nearest_neighbor_km"] > 100
)

# Moderate spatial isolation
qc_review["flag_moderate_isolation"] = (
    qc_review["nearest_neighbor_km"] > 50
)

# Unknown coordinate uncertainty
qc_review["flag_unknown_uncertainty"] = (
    qc_review["coordinateUncertaintyInMeters"].isna()
)

# Relatively high uncertainty among records that survived
# the previous uncertainty QC
qc_review["flag_high_uncertainty"] = (
    qc_review["coordinateUncertaintyInMeters"] > 5000
)

# Missing state information
qc_review["flag_unknown_state"] = (
    qc_review["state_qc"].eq("Unknown")
)


# ------------------------------------------------------------
# 3. Build diagnostic score
#
# IMPORTANT:
# This score prioritizes manual inspection.
# It is NOT an exclusion score.
# ------------------------------------------------------------

qc_review["review_score"] = (
    qc_review["flag_high_isolation"].astype(int) * 2
    +
    qc_review["flag_moderate_isolation"].astype(int)
    +
    qc_review["flag_unknown_uncertainty"].astype(int)
    +
    qc_review["flag_high_uncertainty"].astype(int)
    +
    qc_review["flag_unknown_state"].astype(int)
)


# ------------------------------------------------------------
# 4. Assign review priority
# ------------------------------------------------------------

qc_review["review_priority"] = np.select(
    [
        qc_review["review_score"] >= 3,
        qc_review["review_score"] == 2,
        qc_review["review_score"] == 1
    ],
    [
        "high",
        "medium",
        "low"
    ],
    default="routine"
)


# ------------------------------------------------------------
# 5. Summary
# ------------------------------------------------------------

print("=== REVIEW PRIORITY ===")

print(
    qc_review["review_priority"]
    .value_counts()
)

print("\n=== REVIEW SCORE ===")

print(
    qc_review["review_score"]
    .value_counts()
    .sort_index()
)


# ------------------------------------------------------------
# 6. High-priority records only
# ------------------------------------------------------------

high_priority = (
    qc_review[
        qc_review["review_priority"].eq("high")
    ]
    [
        [
            "gbifID",
            "scientificName",
            "state_qc",
            "decimalLongitude",
            "decimalLatitude",
            "nearest_neighbor_km",
            "coordinateUncertaintyInMeters",
            "year",
            "institutionCode",
            "basisOfRecord",
            "flag_high_isolation",
            "flag_unknown_uncertainty",
            "flag_high_uncertainty",
            "flag_unknown_state",
            "review_score"
        ]
    ]
    .sort_values(
        [
            "review_score",
            "nearest_neighbor_km"
        ],
        ascending=[False, False]
    )
)

print(
    "\nHigh-priority records:",
    len(high_priority)
)

display(high_priority)

In [ ]:
# ============================================================
# HIGH-PRIORITY RECORDS — SOURCE TRACEABILITY
# ============================================================

import requests
import pandas as pd
import time

priority_ids = (
    high_priority["gbifID"]
    .astype(str)
    .tolist()
)

records = []

for gbif_id in priority_ids:

    url = (
        "https://api.gbif.org/v1/occurrence/"
        f"{gbif_id}"
    )

    try:
        response = requests.get(
            url,
            timeout=30
        )

        response.raise_for_status()
        r = response.json()

        records.append({
            "gbifID": str(r.get("gbifID")),

            "scientificName":
                r.get("scientificName"),

            "acceptedScientificName":
                r.get("acceptedScientificName"),

            "stateProvince":
                r.get("stateProvince"),

            "locality":
                r.get("locality"),

            "decimalLongitude":
                r.get("decimalLongitude"),

            "decimalLatitude":
                r.get("decimalLatitude"),

            "coordinateUncertaintyInMeters":
                r.get(
                    "coordinateUncertaintyInMeters"
                ),

            "year":
                r.get("year"),

            "basisOfRecord":
                r.get("basisOfRecord"),

            "institutionCode":
                r.get("institutionCode"),

            "datasetName":
                r.get("datasetName"),

            "recordedBy":
                r.get("recordedBy"),

            "identifiedBy":
                r.get("identifiedBy"),

            "identificationVerificationStatus":
                r.get(
                    "identificationVerificationStatus"
                ),

            "occurrenceID":
                r.get("occurrenceID"),

            "identifier":
                r.get("identifier"),

            "references":
                r.get("references"),

            "catalogNumber":
                r.get("catalogNumber"),

            "occurrenceRemarks":
                r.get("occurrenceRemarks"),

            "identificationRemarks":
                r.get("identificationRemarks")
        })

    except Exception as e:

        print(
            f"ERROR {gbif_id}: {e}"
        )

    time.sleep(0.1)


priority_metadata = pd.DataFrame(records)

print(
    "Records retrieved:",
    len(priority_metadata)
)

display(priority_metadata)

In [ ]:
# ============================================================
# TRACEABILITY SUMMARY
# ============================================================

traceability = (
    priority_metadata[
        [
            "gbifID",
            "stateProvince",
            "year",
            "basisOfRecord",
            "institutionCode",
            "datasetName",
            "recordedBy",
            "identifiedBy",
            "identificationVerificationStatus",
            "coordinateUncertaintyInMeters",
            "catalogNumber",
            "references"
        ]
    ]
    .copy()
)

display(traceability)

In [ ]:
# ============================================================
# HIGH-PRIORITY QC — RECOVER ECOREGION INFORMATION
# ============================================================

import geopandas as gpd
import pandas as pd

# ------------------------------------------------------------
# 1. Load ecoregions
# ------------------------------------------------------------

ecoregions_path = (
    RAW_DIR /
    "Ecoregions2017.zip"
)

ecoregions = gpd.read_file(
    f"zip://{ecoregions_path}"
)

# Repair geometries if necessary
ecoregions["geometry"] = (
    ecoregions.geometry.make_valid()
)

# Ensure same CRS
ecoregions = ecoregions.to_crs(
    occ_thinned.crs
)


# ------------------------------------------------------------
# 2. Select high-priority occurrences
# ------------------------------------------------------------

high_ids = (
    high_priority["gbifID"]
    .astype(str)
)

high_geo = occ_thinned[
    occ_thinned["gbifID"]
    .astype(str)
    .isin(high_ids)
].copy()


# ------------------------------------------------------------
# 3. Spatial join
# ------------------------------------------------------------

high_eco = gpd.sjoin(
    high_geo,
    ecoregions[
        [
            "ECO_NAME",
            "BIOME_NAME",
            "geometry"
        ]
    ],
    how="left",
    predicate="intersects"
)


# ------------------------------------------------------------
# 4. Merge diagnostic score
# ------------------------------------------------------------

high_eco["gbifID"] = (
    high_eco["gbifID"].astype(str)
)

score_table = high_priority[
    [
        "gbifID",
        "review_score"
    ]
].copy()

score_table["gbifID"] = (
    score_table["gbifID"].astype(str)
)

high_eco = high_eco.merge(
    score_table,
    on="gbifID",
    how="left"
)


# ------------------------------------------------------------
# 5. Display diagnostic table
# ------------------------------------------------------------

cols = [
    "gbifID",
    "stateProvince",
    "ECO_NAME",
    "BIOME_NAME",
    "decimalLongitude",
    "decimalLatitude",
    "nearest_neighbor_km",
    "coordinateUncertaintyInMeters",
    "year",
    "institutionCode",
    "basisOfRecord",
    "review_score"
]

high_eco_table = (
    high_eco[cols]
    .sort_values(
        [
            "review_score",
            "nearest_neighbor_km"
        ],
        ascending=[False, False]
    )
)

display(high_eco_table)

In [ ]:
boundary_ids = [
    "6320291511",  # Querência
    "6185940309",  # Tabaporã
    "3925438343",  # Rondonópolis
]

mask = (
    occ_thinned["gbifID"]
    .astype(str)
    .isin(boundary_ids)
)

occ_thinned.loc[
    mask,
    "qc_biogeographic_status"
] = "review_taxonomic_boundary"

occ_thinned.loc[
    mask,
    "qc_biogeographic_reason"
] = (
    "Research-grade photographic occurrence attributed to "
    "Sapajus libidinosus, but located in Mato Grosso near or beyond "
    "published taxonomic/biogeographic boundaries involving S. cay "
    "and/or S. apella. Retained for manual taxonomic review before "
    "SDM calibration."
)

In [ ]:
# ============================================================
# INSPECT FULL GBIF RECORD — 6510070188
# ============================================================

import requests

gbif_id = "6510070188"

url = f"https://api.gbif.org/v1/occurrence/{gbif_id}"

response = requests.get(
    url,
    timeout=30
)

response.raise_for_status()

record = response.json()

print("=" * 70)
print("GBIF:", gbif_id)
print("=" * 70)

source_terms = [
    "scientificName",
    "acceptedScientificName",
    "verbatimScientificName",
    "taxonKey",
    "gbifID",
    "occurrenceID",
    "identifier",
    "identifiers",
    "datasetKey",
    "datasetName",
    "datasetTitle",
    "institutionCode",
    "collectionCode",
    "ownerInstitutionCode",
    "recordedBy",
    "recordedByIDs",
    "identifiedBy",
    "identifiedByIDs",
    "identificationVerificationStatus",
    "basisOfRecord",
    "occurrenceStatus",
    "locality",
    "stateProvince",
    "decimalLongitude",
    "decimalLatitude",
    "coordinateUncertaintyInMeters",
    "eventDate",
    "year",
    "georeferenceRemarks",
    "locationRemarks",
    "occurrenceRemarks",
    "identificationRemarks",
    "references",
    "issues"
]

for field in source_terms:
    value = record.get(field)

    if value is not None:
        print(f"{field}: {value}")

In [ ]:
# Search for potentially useful provenance / identification fields

keywords = [
    "ident",
    "record",
    "taxon",
    "source",
    "reference",
    "remark",
    "local",
    "location",
    "institution",
    "collection",
    "dataset",
    "georef"
]

print("\n=== POSSIBLY RELEVANT FIELDS ===")

for key in sorted(record.keys()):
    if any(word in key.lower() for word in keywords):
        print(f"{key}: {record[key]}")

In [ ]:
# ============================================================
# INSPECT FULL GBIF RECORD — 3397816338
# ============================================================

import requests

gbif_id = "3397816338"

url = f"https://api.gbif.org/v1/occurrence/{gbif_id}"

response = requests.get(
    url,
    timeout=30
)

response.raise_for_status()

record_pb = response.json()

print("=" * 70)
print("GBIF:", gbif_id)
print("=" * 70)

fields = [
    "scientificName",
    "acceptedScientificName",
    "verbatimScientificName",
    "taxonKey",
    "gbifID",
    "occurrenceID",
    "identifier",
    "identifiers",
    "catalogNumber",
    "datasetKey",
    "datasetName",
    "datasetTitle",
    "institutionCode",
    "collectionCode",
    "ownerInstitutionCode",
    "recordedBy",
    "recordedByIDs",
    "identifiedBy",
    "identifiedByIDs",
    "identificationVerificationStatus",
    "basisOfRecord",
    "occurrenceStatus",
    "locality",
    "municipality",
    "stateProvince",
    "decimalLongitude",
    "decimalLatitude",
    "coordinateUncertaintyInMeters",
    "eventDate",
    "year",
    "georeferenceRemarks",
    "locationRemarks",
    "occurrenceRemarks",
    "identificationRemarks",
    "references",
    "issues"
]

for field in fields:
    value = record_pb.get(field)

    if value is not None:
        print(f"{field}: {value}")

In [ ]:
# ============================================================
# SEARCH ADDITIONAL PROVENANCE / IDENTIFICATION FIELDS
# ============================================================

keywords = [
    "ident",
    "record",
    "taxon",
    "source",
    "reference",
    "remark",
    "local",
    "location",
    "institution",
    "collection",
    "dataset",
    "georef",
    "catalog",
    "media"
]

print("\n=== POSSIBLY RELEVANT FIELDS ===")

for key in sorted(record_pb.keys()):
    if any(word in key.lower() for word in keywords):
        print(f"{key}: {record_pb[key]}")

In [ ]:
# ============================================================
# MANUAL BIOGEOGRAPHIC QC
# GBIF 3397816338 — FNJV 0050472
# ============================================================

target_id = "3397816338"

mask = occ_thinned["gbifID"].astype(str).eq(target_id)

occ_thinned.loc[
    mask, "qc_biogeographic_status"
] = "retain_native_verified_collection"

occ_thinned.loc[
    mask, "qc_biogeographic_reason"
] = (
    "Primary-source verification in the Fonoteca Neotropical "
    "Jacques Vielliard (FNJV) confirmed an archived acoustic "
    "record of Sapajus libidinosus (FNJV 0050472), recorded by "
    "Eduardo Marques Santos Júnior on 12 June 2017 at Floresta "
    "Nacional de Cabedelo, Paraíba. Locality and coordinates "
    "match the GBIF occurrence. Retained despite spatial "
    "isolation and mangrove ecoregion assignment."
)

# Conferência
occ_thinned.loc[
    mask,
    [
        "gbifID",
        "stateProvince",
        "decimalLongitude",
        "decimalLatitude",
        "qc_biogeographic_status",
        "qc_biogeographic_reason",
    ],
]

In [ ]:
occ_thinned["qc_biogeographic_status"].value_counts(dropna=False)

In [ ]:
target_id = "1321834077"

rec = occ_thinned[
    occ_thinned["gbifID"].astype(str).eq(target_id)
].copy()

rec.T

In [ ]:
import requests

gbif_id = "1321834077"

url = f"https://api.gbif.org/v1/occurrence/{gbif_id}"
r = requests.get(url, timeout=30)
r.raise_for_status()

rec_usnm = r.json()

fields = [
    "key",
    "gbifID",
    "scientificName",
    "acceptedScientificName",
    "verbatimScientificName",
    "taxonKey",
    "acceptedTaxonKey",
    "taxonomicStatus",
    "basisOfRecord",
    "occurrenceStatus",
    "institutionCode",
    "collectionCode",
    "catalogNumber",
    "recordNumber",
    "occurrenceID",
    "type",
    "recordedBy",
    "identifiedBy",
    "dateIdentified",
    "eventDate",
    "year",
    "locality",
    "verbatimLocality",
    "municipality",
    "county",
    "stateProvince",
    "decimalLongitude",
    "decimalLatitude",
    "coordinateUncertaintyInMeters",
    "verbatimLongitude",
    "verbatimLatitude",
    "geodeticDatum",
    "georeferencedBy",
    "georeferencedDate",
    "georeferenceProtocol",
    "georeferenceSources",
    "georeferenceRemarks",
    "datasetName",
    "datasetTitle",
    "references",
    "issues",
]

for field in fields:
    if field in rec_usnm:
        print(f"{field}: {rec_usnm[field]}")

In [ ]:
print("\n=== ALL POTENTIALLY RELEVANT FIELDS ===")

keywords = (
    "tax",
    "ident",
    "catalog",
    "record",
    "local",
    "geo",
    "coord",
    "verbatim",
    "collect",
    "type",
    "remark",
    "reference",
)

for key in sorted(rec_usnm):
    if any(word in key.lower() for word in keywords):
        print(f"{key}: {rec_usnm[key]}")

In [ ]:
target_id = "1321834077"

mask = occ_thinned["gbifID"].astype(str).eq(target_id)

occ_thinned.loc[
    mask, "qc_biogeographic_status"
] = "retain_native_verified_specimen"

occ_thinned.loc[
    mask, "qc_biogeographic_reason"
] = (
    "Primary-source verification in the Smithsonian National Museum "
    "of Natural History (NMNH) confirmed preserved specimen USNM 518270, "
    "currently identified as Sapajus libidinosus libidinosus. "
    "The specimen was collected on 25 July 1947 at Carinhanha, Cocos, "
    "Mata do Boi, Fazenda da Tolda, Bahia. Smithsonian coordinates "
    "(-14.17, -44.55) match the GBIF occurrence. The collection also "
    "records the historical identification Sapajus apella libidinosus. "
    "Retained despite high spatial isolation because the occurrence is "
    "supported by a traceable preserved museum specimen."
)

occ_thinned.loc[
    mask,
    [
        "gbifID",
        "stateProvince",
        "decimalLongitude",
        "decimalLatitude",
        "nearest_neighbor_km",
        "qc_biogeographic_status",
        "qc_biogeographic_reason",
    ]
]

In [ ]:
occ_thinned["qc_biogeographic_status"].value_counts(dropna=False)

In [ ]:
occ_thinned.to_csv(
    "../data/processed/occurrences_biogeographic_qc_checkpoint.csv",
    index=False
)

In [ ]:
# IDs dos 20 registros originalmente classificados como HIGH PRIORITY
high_ids = [
    1321834077,
    6510070188,
    1317493471,
    1320820255,
    3397816338,
    6320291511,
    6273800014,
    6185940309,
    5292552768,
    5892444353,
    6133042386,
    4440816529,
    3925438343,
    6163007459,
    3902295584,
    6159632702,
    4171964336,
    6236293553,
    6235417760,
    5862715833
]

high_remaining = occ_thinned[
    occ_thinned["gbifID"].astype(str).isin(
        [str(x) for x in high_ids]
    )
    &
    (occ_thinned["qc_biogeographic_status"] == "pending")
].copy()

print(
    f"High-priority pendentes: {len(high_remaining)} / {len(high_ids)}"
)

high_remaining[
    [
        "gbifID",
        "stateProvince",
        "decimalLongitude",
        "decimalLatitude",
        "nearest_neighbor_km",
        "institutionCode",
        "basisOfRecord"
    ]
].sort_values(
    "nearest_neighbor_km",
    ascending=False
)

In [ ]:
occ_thinned.to_csv(
    "../data/processed/occurrences_biogeographic_qc_checkpoint.csv",
    index=False
)

print("Checkpoint salvo!")
print("Pending total:", (occ_thinned["qc_biogeographic_status"] == "pending").sum())
print("High-priority pending:", len(high_remaining))

In [ ]:
import requests

gbif_id = "1317493471"

url = f"https://api.gbif.org/v1/occurrence/{gbif_id}"
r = requests.get(url, timeout=30)
r.raise_for_status()

rec_usnm = r.json()

fields = [
    "gbifID",
    "scientificName",
    "acceptedScientificName",
    "verbatimScientificName",
    "taxonKey",
    "acceptedTaxonKey",
    "taxonomicStatus",
    "basisOfRecord",
    "institutionCode",
    "collectionCode",
    "catalogNumber",
    "recordNumber",
    "occurrenceID",
    "recordedBy",
    "identifiedBy",
    "eventDate",
    "locality",
    "stateProvince",
    "decimalLongitude",
    "decimalLatitude",
    "coordinateUncertaintyInMeters",
    "geodeticDatum",
    "datasetName",
    "references",
    "issues",
]

for field in fields:
    if field in rec_usnm:
        print(f"{field}: {rec_usnm[field]}")

In [ ]:
import pandas as pd

occ_thinned = pd.read_csv(
    "../data/processed/occurrences_biogeographic_qc_checkpoint.csv"
)

print("Dataset carregado:", occ_thinned.shape)

print("\nBiogeographic QC status:")
print(
    occ_thinned["qc_biogeographic_status"]
    .value_counts(dropna=False)
)

In [ ]:
target_id = "1317493471"

mask = occ_thinned["gbifID"].astype(str) == target_id

occ_thinned.loc[
    mask,
    "qc_biogeographic_status"
] = "retain_native_verified_specimen"

occ_thinned.loc[
    mask,
    "qc_biogeographic_reason"
] = (
    "Primary-source verification in the Smithsonian National Museum "
    "of Natural History (USNM 518347) confirmed a preserved female "
    "specimen currently identified as Sapajus libidinosus libidinosus, "
    "collected on 29 December 1948 at Pedro Afonso, Fazenda Mirador, "
    "Mata Mirador, Brazil. Museum coordinates (-48.18, -8.98) match "
    "the GBIF occurrence. Retained despite spatial isolation because "
    "the occurrence is supported by a traceable museum voucher."
)

print(
    occ_thinned.loc[
        mask,
        [
            "gbifID",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "qc_biogeographic_status",
            "qc_biogeographic_reason"
        ]
    ]
)

print("\nBiogeographic QC status:")
print(
    occ_thinned["qc_biogeographic_status"]
    .value_counts(dropna=False)
)

In [ ]:
occ_thinned.to_csv(
    "../data/processed/occurrences_biogeographic_qc_checkpoint.csv",
    index=False
)

print("Checkpoint atualizado.")

In [ ]:
import requests

gbif_id = "1320820255"

url = f"https://api.gbif.org/v1/occurrence/{gbif_id}"
r = requests.get(url, timeout=30)
r.raise_for_status()

rec_usnm = r.json()

fields = [
    "gbifID",
    "scientificName",
    "acceptedScientificName",
    "verbatimScientificName",
    "taxonKey",
    "acceptedTaxonKey",
    "taxonomicStatus",
    "basisOfRecord",
    "institutionCode",
    "collectionCode",
    "catalogNumber",
    "recordNumber",
    "occurrenceID",
    "recordedBy",
    "identifiedBy",
    "eventDate",
    "locality",
    "stateProvince",
    "decimalLongitude",
    "decimalLatitude",
    "coordinateUncertaintyInMeters",
    "geodeticDatum",
    "datasetName",
    "references",
    "issues",
]

for field in fields:
    if field in rec_usnm:
        print(f"{field}: {rec_usnm[field]}")

In [ ]:
target_id = "1320820255"

mask = occ_thinned["gbifID"].astype(str) == target_id

occ_thinned.loc[
    mask,
    "qc_biogeographic_status"
] = "retain_native_verified_specimen"

occ_thinned.loc[
    mask,
    "qc_biogeographic_reason"
] = (
    "Primary-source verification in the Smithsonian National Museum "
    "of Natural History (USNM 518278) confirmed a preserved female "
    "specimen currently identified as Sapajus libidinosus libidinosus, "
    "collected on 5 August 1949 at Itajui, Fazenda Aldeia, Bahia, Brazil. "
    "Museum coordinates (-45.2, -11.05) match the GBIF occurrence. "
    "Retained despite spatial isolation because the occurrence is "
    "supported by a traceable museum voucher."
)

print(
    occ_thinned.loc[
        mask,
        [
            "gbifID",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "qc_biogeographic_status",
            "qc_biogeographic_reason"
        ]
    ]
)

print("\nBiogeographic QC status:")
print(
    occ_thinned["qc_biogeographic_status"]
    .value_counts(dropna=False)
)

# salva imediatamente
occ_thinned.to_csv(
    "../data/processed/occurrences_biogeographic_qc_checkpoint.csv",
    index=False
)

print("\nCheckpoint atualizado.")

In [ ]:
import requests

gbif_id = "6510070188"

r = requests.get(
    f"https://api.gbif.org/v1/occurrence/{gbif_id}",
    timeout=30
)
r.raise_for_status()

rec = r.json()

fields = [
    "gbifID",
    "scientificName",
    "acceptedScientificName",
    "taxonKey",
    "acceptedTaxonKey",
    "basisOfRecord",
    "occurrenceID",
    "identifier",
    "datasetKey",
    "datasetName",
    "institutionCode",
    "collectionCode",
    "recordedBy",
    "identifiedBy",
    "identificationVerificationStatus",
    "eventDate",
    "locality",
    "municipality",
    "stateProvince",
    "decimalLongitude",
    "decimalLatitude",
    "coordinateUncertaintyInMeters",
    "georeferenceRemarks",
    "locationRemarks",
    "references",
    "issues",
]

for field in fields:
    if field in rec:
        print(f"{field}: {rec[field]}")

In [ ]:
target_id = "6510070188"

mask = occ_thinned["gbifID"].astype(str) == target_id

occ_thinned.loc[
    mask,
    "qc_biogeographic_status"
] = "review_taxonomic_boundary"

occ_thinned.loc[
    mask,
    "qc_biogeographic_reason"
] = (
    "SISBio/ICMBio human observation from Araçatuba, São Paulo, "
    "a region of taxonomic and biogeographic complexity involving "
    "Sapajus libidinosus and S. nigritus. The record lacks a traceable "
    "individual-level taxonomic identification, voucher, photograph, "
    "or other verification evidence, and the reported coordinates are "
    "described as a reference coordinate for the sampled area. "
    "Retained for taxonomic review but not treated as a confirmed "
    "S. libidinosus presence for SDM calibration."
)

print(
    occ_thinned.loc[
        mask,
        [
            "gbifID",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "qc_biogeographic_status",
            "qc_biogeographic_reason"
        ]
    ]
)

occ_thinned.to_csv(
    "../data/processed/occurrences_biogeographic_qc_checkpoint.csv",
    index=False
)

print("\nCheckpoint atualizado.")
print(
    occ_thinned["qc_biogeographic_status"]
    .value_counts(dropna=False)
)

In [ ]:
target_id = "6273800014"

mask = occ_thinned["gbifID"].astype(str) == target_id

# Atualizar status
occ_thinned.loc[
    mask,
    "qc_biogeographic_status"
] = "retain_native_verified_observation"

# Registrar justificativa
occ_thinned.loc[
    mask,
    "qc_biogeographic_reason"
] = (
    "Photographically documented iNaturalist Research Grade observation "
    "of Sapajus libidinosus from Auriflama, northwestern São Paulo. "
    "The identification is supported by community consensus, and the locality "
    "is compatible with the documented native occurrence of S. libidinosus "
    "in northern São Paulo. The record was manually reviewed because of its "
    "high spatial isolation from other occurrences."
)

# Conferir
display(
    occ_thinned.loc[
        mask,
        [
            "gbifID",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "qc_biogeographic_status",
            "qc_biogeographic_reason",
        ],
    ]
)

print("\nBiogeographic QC status:")
print(
    occ_thinned["qc_biogeographic_status"]
    .value_counts(dropna=False)
)

# Salvar NO MESMO checkpoint que estamos usando
occ_thinned.to_csv(
    "../data/processed/occurrences_biogeographic_qc_checkpoint.csv",
    index=False
)

print("\nCheckpoint atualizado.")

In [ ]:
target_id = "5292552768"

mask = occ_thinned["gbifID"].astype(str) == target_id

occ_thinned.loc[
    mask,
    "qc_biogeographic_status"
] = "exclude_suspected_captive"

occ_thinned.loc[
    mask,
    "qc_biogeographic_reason"
] = (
    "Photographically documented iNaturalist observation from São Luís, "
    "Maranhão. Although identified as Sapajus libidinosus and classified "
    "as Research Grade, the photographic context raises concern that the "
    "individual may have been captive, habituated, or otherwise associated "
    "with human management. The locality is also biogeographically "
    "problematic for a wild native occurrence. Because provenance as a "
    "free-ranging wild individual cannot be established, the record was "
    "excluded from SDM calibration."
)

# Conferir a alteração
display(
    occ_thinned.loc[
        mask,
        [
            "gbifID",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "qc_biogeographic_status",
            "qc_biogeographic_reason",
        ],
    ]
)

print("\nBiogeographic QC status:")
print(
    occ_thinned["qc_biogeographic_status"]
    .value_counts(dropna=False)
)

# Salvar checkpoint
occ_thinned.to_csv(
    "../data/processed/occurrences_biogeographic_qc_checkpoint.csv",
    index=False
)

print("\nCheckpoint atualizado.")

In [ ]:
target_id = "5892444353"

mask = occ_thinned["gbifID"].astype(str) == target_id

# Atualizar status
occ_thinned.loc[
    mask,
    "qc_biogeographic_status"
] = "retain_native_verified_observation"

# Registrar justificativa
occ_thinned.loc[
    mask,
    "qc_biogeographic_reason"
] = (
    "Photographically documented iNaturalist Research Grade observation "
    "of Sapajus libidinosus from Bom Jardim, Nobres, Mato Grosso. "
    "The observation shows a free-ranging individual in a natural setting, "
    "and the identification is supported by photographic evidence and "
    "iNaturalist community identification. Independent photographic evidence "
    "also documents wild S. libidinosus from the same locality. "
    "The record was manually reviewed because of its high spatial isolation "
    "and its location near a biogeographically relevant distributional boundary."
)

# Conferir
display(
    occ_thinned.loc[
        mask,
        [
            "gbifID",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "qc_biogeographic_status",
            "qc_biogeographic_reason",
        ],
    ]
)

print("\nBiogeographic QC status:")
print(
    occ_thinned["qc_biogeographic_status"]
    .value_counts(dropna=False)
)

# Salvar NO checkpoint correto
occ_thinned.to_csv(
    "../data/processed/occurrences_biogeographic_qc_checkpoint.csv",
    index=False
)

print("\nCheckpoint atualizado.")

In [ ]:
target_id = "6133042386"

mask = occ_thinned["gbifID"].astype(str) == target_id

occ_thinned.loc[
    mask,
    "qc_biogeographic_status"
] = "retain_native_verified_observation"

occ_thinned.loc[
    mask,
    "qc_biogeographic_reason"
] = (
    "Photographically documented iNaturalist Research Grade observation "
    "of Sapajus libidinosus from Ilha do Urubu, Paulo Afonso, Bahia. "
    "The photograph shows an individual moving freely in vegetation, "
    "with no visible evidence of captivity or restraint. The identification "
    "is supported by photographic evidence and iNaturalist community "
    "identification. The record was manually reviewed because of its high "
    "spatial isolation from other occurrence records."
)

display(
    occ_thinned.loc[
        mask,
        [
            "gbifID",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "qc_biogeographic_status",
            "qc_biogeographic_reason",
        ],
    ]
)

print("\nBiogeographic QC status:")
print(
    occ_thinned["qc_biogeographic_status"]
    .value_counts(dropna=False)
)

occ_thinned.to_csv(
    "../data/processed/occurrences_biogeographic_qc_checkpoint.csv",
    index=False
)

print("\nCheckpoint atualizado.")

In [ ]:
target_id = "4440816529"

mask = occ_thinned["gbifID"].astype(str) == target_id

# Atualizar status
occ_thinned.loc[
    mask,
    "qc_biogeographic_status"
] = "review_taxonomic_boundary"

# Registrar justificativa
occ_thinned.loc[
    mask,
    "qc_biogeographic_reason"
] = (
    "Photographically documented iNaturalist Research Grade observation "
    "identified as Sapajus libidinosus from Marliéria, Minas Gerais. "
    "The photograph supports identification as Sapajus, but the locality "
    "occurs in a biogeographically and taxonomically sensitive region for "
    "species-level identification within the genus. The record is therefore "
    "retained for taxonomic review rather than accepted as a confirmed "
    "presence of S. libidinosus for SDM calibration."
)

# Conferir registro
display(
    occ_thinned.loc[
        mask,
        [
            "gbifID",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "qc_biogeographic_status",
            "qc_biogeographic_reason",
        ],
    ]
)

print("\nBiogeographic QC status:")
print(
    occ_thinned["qc_biogeographic_status"]
    .value_counts(dropna=False)
)

# Salvar checkpoint
occ_thinned.to_csv(
    "../data/processed/occurrences_biogeographic_qc_checkpoint.csv",
    index=False
)

print("\nCheckpoint atualizado.")

In [ ]:
target_id = "6163007459"

mask = occ_thinned["gbifID"].astype(str) == target_id

# Atualizar status
occ_thinned.loc[
    mask,
    "qc_biogeographic_status"
] = "retain_native_verified_observation"

# Registrar justificativa
occ_thinned.loc[
    mask,
    "qc_biogeographic_reason"
] = (
    "Photographically documented iNaturalist Research Grade observation "
    "of Sapajus libidinosus from Talismã, Tocantins. "
    "The photographs show an individual moving freely in vegetation, "
    "with no visible evidence of captivity or restraint. "
    "The identification is supported by photographic evidence and "
    "iNaturalist community identification. The locality is biogeographically "
    "compatible with native occurrence of S. libidinosus. "
    "The record was manually reviewed because of its high spatial isolation "
    "from other occurrence records."
)

# Conferir
display(
    occ_thinned.loc[
        mask,
        [
            "gbifID",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "qc_biogeographic_status",
            "qc_biogeographic_reason",
        ],
    ]
)

print("\nBiogeographic QC status:")
print(
    occ_thinned["qc_biogeographic_status"]
    .value_counts(dropna=False)
)

# Salvar checkpoint
occ_thinned.to_csv(
    "../data/processed/occurrences_biogeographic_qc_checkpoint.csv",
    index=False
)

print("\nCheckpoint atualizado.")

In [ ]:
target_id = "6159632702"

mask = occ_thinned["gbifID"].astype(str) == target_id

# Atualizar status
occ_thinned.loc[
    mask,
    "qc_biogeographic_status"
] = "retain_native_verified_observation"

# Registrar justificativa
occ_thinned.loc[
    mask,
    "qc_biogeographic_reason"
] = (
    "Photographically documented iNaturalist Research Grade observation "
    "of Sapajus libidinosus from São Félix do Araguaia, Mato Grosso. "
    "The photograph shows an individual moving freely in vegetation, "
    "with no visible evidence of captivity or restraint. "
    "The identification is supported by photographic evidence and "
    "iNaturalist community identification. The record was manually reviewed "
    "because of its high spatial isolation from other occurrence records."
)

# Conferir
display(
    occ_thinned.loc[
        mask,
        [
            "gbifID",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "qc_biogeographic_status",
            "qc_biogeographic_reason",
        ],
    ]
)

print("\nBiogeographic QC status:")
print(
    occ_thinned["qc_biogeographic_status"]
    .value_counts(dropna=False)
)

# Salvar checkpoint
occ_thinned.to_csv(
    "../data/processed/occurrences_biogeographic_qc_checkpoint.csv",
    index=False
)

print("\nCheckpoint atualizado.")

In [ ]:
target_id = "3902295584"

mask = occ_thinned["gbifID"].astype(str) == target_id

# Atualizar status
occ_thinned.loc[
    mask,
    "qc_biogeographic_status"
] = "retain_native_verified_observation"

# Registrar justificativa
occ_thinned.loc[
    mask,
    "qc_biogeographic_reason"
] = (
    "Photographically documented iNaturalist Research Grade observation "
    "of Sapajus libidinosus from Dueré, Tocantins. "
    "The photograph shows an individual occurring freely in natural vegetation, "
    "with no visible evidence of captivity or restraint. "
    "The identification is supported by photographic evidence and iNaturalist "
    "community identification. The locality is consistent with native occurrence "
    "of the species. The record was manually reviewed because of its high "
    "spatial isolation from other occurrence records."
)

# Conferir registro
display(
    occ_thinned.loc[
        mask,
        [
            "gbifID",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "qc_biogeographic_status",
            "qc_biogeographic_reason",
        ],
    ]
)

print("\nBiogeographic QC status:")
print(
    occ_thinned["qc_biogeographic_status"]
    .value_counts(dropna=False)
)

# Salvar checkpoint
occ_thinned.to_csv(
    "../data/processed/occurrences_biogeographic_qc_checkpoint.csv",
    index=False
)

print("\nCheckpoint atualizado.")

In [ ]:
target_id = "4171964336"

mask = occ_thinned["gbifID"].astype(str) == target_id

occ_thinned.loc[
    mask,
    "qc_biogeographic_status"
] = "retain_native_verified_observation"

occ_thinned.loc[
    mask,
    "qc_biogeographic_reason"
] = (
    "Photographically documented iNaturalist Research Grade observation "
    "of Sapajus libidinosus from Tocantins. "
    "The photograph shows an individual occurring freely in vegetation, "
    "with no visible evidence of captivity or restraint. "
    "The identification is supported by photographic documentation and "
    "the iNaturalist Research Grade record. The locality is consistent "
    "with native occurrence of the species. The record was manually reviewed "
    "because of its high spatial isolation from other occurrence records."
)

display(
    occ_thinned.loc[
        mask,
        [
            "gbifID",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "qc_biogeographic_status",
            "qc_biogeographic_reason",
        ],
    ]
)

print("\nBiogeographic QC status:")
print(
    occ_thinned["qc_biogeographic_status"]
    .value_counts(dropna=False)
)

occ_thinned.to_csv(
    "../data/processed/occurrences_biogeographic_qc_checkpoint.csv",
    index=False
)

print("\nCheckpoint atualizado.")

In [ ]:
target_id = "6236293553"

mask = occ_thinned["gbifID"].astype(str) == target_id

occ_thinned.loc[
    mask,
    "qc_biogeographic_status"
] = "retain_native_verified_observation"

occ_thinned.loc[
    mask,
    "qc_biogeographic_reason"
] = (
    "Photographically documented iNaturalist observation identified as "
    "Sapajus libidinosus from northern Minas Gerais. The record contains "
    "multiple photographs showing individuals occurring freely in natural "
    "vegetation, with no visible evidence of captivity or restraint. "
    "The photographic documentation supports the occurrence of Sapajus "
    "at the locality, which is biogeographically consistent with the native "
    "distribution of S. libidinosus. The record was manually reviewed "
    "because of its high spatial isolation from other occurrence records."
)

display(
    occ_thinned.loc[
        mask,
        [
            "gbifID",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "qc_biogeographic_status",
            "qc_biogeographic_reason",
        ],
    ]
)

print("\nBiogeographic QC status:")
print(
    occ_thinned["qc_biogeographic_status"]
    .value_counts(dropna=False)
)

occ_thinned.to_csv(
    "../data/processed/occurrences_biogeographic_qc_checkpoint.csv",
    index=False
)

print("\nCheckpoint atualizado.")

In [ ]:
target_id = "6235417760"

mask = occ_thinned["gbifID"].astype(str) == target_id

occ_thinned.loc[
    mask,
    "qc_biogeographic_status"
] = "review_taxonomic_boundary"

occ_thinned.loc[
    mask,
    "qc_biogeographic_reason"
] = (
    "Photographically documented iNaturalist Research Grade observation "
    "identified as Sapajus libidinosus from Minas Gerais. The photograph "
    "supports identification as Sapajus and is consistent with a free-ranging "
    "individual, with no visible evidence of captivity. However, the locality "
    "occurs in a biogeographically and taxonomically sensitive region for "
    "species-level identification within Sapajus. The record is therefore "
    "retained for taxonomic review rather than accepted as a confirmed "
    "presence of S. libidinosus for SDM calibration."
)

display(
    occ_thinned.loc[
        mask,
        [
            "gbifID",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "qc_biogeographic_status",
            "qc_biogeographic_reason",
        ],
    ]
)

print("\nBiogeographic QC status:")
print(
    occ_thinned["qc_biogeographic_status"]
    .value_counts(dropna=False)
)

occ_thinned.to_csv(
    "../data/processed/occurrences_biogeographic_qc_checkpoint.csv",
    index=False
)

print("\nCheckpoint atualizado.")

In [ ]:
target_id = "5862715833"

mask = occ_thinned["gbifID"].astype(str) == target_id

# Atualizar status
occ_thinned.loc[
    mask,
    "qc_biogeographic_status"
] = "exclude_unverifiable_provenance"

# Registrar justificativa
occ_thinned.loc[
    mask,
    "qc_biogeographic_reason"
] = (
    "Material sample associated with a COI DNA sequence and identified "
    "as Sapajus libidinosus in GBIF. However, the primary BOLD Systems "
    "reference (GBAAY81450-24) could not be retrieved, preventing "
    "independent verification of specimen provenance, voucher information, "
    "and the geographic origin of the reported coordinates. The record was "
    "therefore excluded from SDM calibration because its geographic "
    "provenance could not be independently verified."
)

# Conferir registro
display(
    occ_thinned.loc[
        mask,
        [
            "gbifID",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "basisOfRecord",
            "institutionCode",
            "qc_biogeographic_status",
            "qc_biogeographic_reason",
        ],
    ]
)

print("\nBiogeographic QC status:")
print(
    occ_thinned["qc_biogeographic_status"]
    .value_counts(dropna=False)
)

# Salvar checkpoint
occ_thinned.to_csv(
    "../data/processed/occurrences_biogeographic_qc_checkpoint.csv",
    index=False
)

print("\nCheckpoint atualizado.")

In [ ]:
pending_priority = (
    occ_thinned.loc[
        occ_thinned["qc_biogeographic_status"].eq("pending")
    ]
    .sort_values(
        "nearest_neighbor_km",
        ascending=False
    )
)

display(
    pending_priority[
        [
            "gbifID",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "nearest_neighbor_km",
            "institutionCode",
            "basisOfRecord",
        ]
    ].head(10)
)

print(
    "High-priority pendentes:",
    len(pending_priority)
)

In [ ]:
# ============================================================
# HIGH-PRIORITY BIOGEOGRAPHIC QC — FINAL AUDIT TABLE
# ============================================================

high_priority_ids = [
    "1321834077",
    "6510070188",
    "1317493471",
    "1320820255",
    "3397816338",
    "6320291511",
    "6273800014",
    "6185940309",
    "5292552768",
    "5892444353",
    "6133042386",
    "4440816529",
    "3925438343",
    "6163007459",
    "3902295584",
    "6159632702",
    "4171964336",
    "6236293553",
    "6235417760",
    "5862715833",
]

audit_high = (
    occ_thinned.loc[
        occ_thinned["gbifID"].astype(str).isin(high_priority_ids),
        [
            "gbifID",
            "scientificName",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "year",
            "institutionCode",
            "basisOfRecord",
            "nearest_neighbor_km",
            "qc_biogeographic_status",
            "qc_biogeographic_reason",
        ],
    ]
    .copy()
)

# Ordenar do registro espacialmente mais isolado para o menos isolado
audit_high = (
    audit_high
    .sort_values("nearest_neighbor_km", ascending=False)
    .reset_index(drop=True)
)

print("High-priority records recovered:", len(audit_high))

print("\n=== FINAL STATUS ===")
print(
    audit_high["qc_biogeographic_status"]
    .value_counts(dropna=False)
)

display(audit_high)

In [ ]:
print(
    "\nStill pending:",
    audit_high["qc_biogeographic_status"]
    .eq("pending")
    .sum()
)

In [ ]:
# ============================================================
# EXPORT — HIGH-PRIORITY MANUAL BIOGEOGRAPHIC AUDIT
# ============================================================

audit_path = (
    "../data/processed/"
    "sapajus_libidinosus_high_priority_biogeographic_audit.csv"
)

audit_high.to_csv(
    audit_path,
    index=False,
    encoding="utf-8-sig"
)

print("Audit table exported:")
print(audit_path)

print("\nRecords:", len(audit_high))
print(
    "Pending:",
    audit_high["qc_biogeographic_status"].eq("pending").sum()
)

In [ ]:
final_qc_path = (
    "../data/processed/"
    "sapajus_libidinosus_occurrences_biogeographic_qc.csv"
)

occ_thinned.to_csv(
    final_qc_path,
    index=False,
    encoding="utf-8-sig"
)

print("Full QC dataset exported:")
print(final_qc_path)

In [ ]:
# ============================================================
# DIAGNOSTIC — REMAINING PENDING RECORDS
# ============================================================

pending = occ_thinned.loc[
    occ_thinned["qc_biogeographic_status"].eq("pending")
].copy()

print("Pending records:", len(pending))

print("\n=== UNCERTAINTY QC ===")
print(
    pending["qc_uncertainty"]
    .value_counts(dropna=False)
)

print("\n=== GEOGRAPHIC OUTLIER ===")
print(
    pending["qc_geographic_outlier"]
    .value_counts(dropna=False)
)

print("\n=== SPATIAL ISOLATION ===")
print(
    pending["qc_spatial_isolation"]
    .value_counts(dropna=False)
)

print("\n=== ORIGINAL QC KEEP ===")
print(
    pending["qc_keep"]
    .value_counts(dropna=False)
)

print("\n=== BASIS OF RECORD ===")
print(
    pending["basisOfRecord"]
    .value_counts(dropna=False)
)

print("\n=== INSTITUTION ===")
print(
    pending["institutionCode"]
    .value_counts(dropna=False)
    .head(15)
)

print("\n=== NN DISTANCE SUMMARY (km) ===")
print(
    pending["nearest_neighbor_km"]
    .describe()
)

print("\n=== COORDINATE UNCERTAINTY SUMMARY (m) ===")
print(
    pending["coordinateUncertaintyInMeters"]
    .describe()
)

In [ ]:
# ============================================================
# CROSS-TAB — REMAINING PENDING QC FLAGS
# ============================================================

pending = occ_thinned.loc[
    occ_thinned["qc_biogeographic_status"].eq("pending")
].copy()

# 1. Incerteza × isolamento
print("=== UNCERTAINTY × SPATIAL ISOLATION ===")

cross_qc = pd.crosstab(
    pending["qc_uncertainty"],
    pending["qc_spatial_isolation"],
    margins=True
)

display(cross_qc)


# 2. Criar classes diagnósticas — SEM alterar occ_thinned
pending["remaining_qc_class"] = "routine"

pending.loc[
    (pending["qc_uncertainty"] == "unknown") &
    (~pending["qc_spatial_isolation"]),
    "remaining_qc_class"
] = "unknown_uncertainty_only"

pending.loc[
    (pending["qc_uncertainty"] == "acceptable") &
    (pending["qc_spatial_isolation"]),
    "remaining_qc_class"
] = "spatial_isolation_only"

pending.loc[
    (pending["qc_uncertainty"] == "unknown") &
    (pending["qc_spatial_isolation"]),
    "remaining_qc_class"
] = "uncertainty_and_isolation"


print("\n=== REMAINING QC CLASSES ===")

print(
    pending["remaining_qc_class"]
    .value_counts()
)


# 3. Distância NN por classe
print("\n=== NN DISTANCE BY CLASS ===")

display(
    pending.groupby("remaining_qc_class")[
        "nearest_neighbor_km"
    ]
    .agg(
        n="size",
        median="median",
        mean="mean",
        maximum="max"
    )
    .sort_values("maximum", ascending=False)
)


# 4. Incerteza das coordenadas por classe
print("\n=== COORDINATE UNCERTAINTY BY CLASS ===")

display(
    pending.groupby("remaining_qc_class")[
        "coordinateUncertaintyInMeters"
    ]
    .agg(
        n_known="count",
        median="median",
        maximum="max"
    )
)

In [ ]:
# ============================================================
# COORDINATE UNCERTAINTY — THRESHOLD DIAGNOSTIC
# ============================================================

known_uncertainty = pending[
    pending["coordinateUncertaintyInMeters"].notna()
].copy()

thresholds = [100, 500, 1000, 2500, 5000, 10000]

print("Records with known uncertainty:", len(known_uncertainty))

print("\n=== UNCERTAINTY THRESHOLDS ===")

for threshold in thresholds:
    n = (
        known_uncertainty["coordinateUncertaintyInMeters"]
        <= threshold
    ).sum()

    print(
        f"≤ {threshold:>5} m: "
        f"{n:>3} / {len(known_uncertainty)} "
        f"({n / len(known_uncertainty) * 100:.1f}%)"
    )


print("\n=== QUANTILES ===")

print(
    known_uncertainty["coordinateUncertaintyInMeters"]
    .quantile([0, .25, .50, .75, .90, .95, .99, 1])
)


print("\n=== RECORDS > 5 km ===")

display(
    known_uncertainty.loc[
        known_uncertainty["coordinateUncertaintyInMeters"] > 5000,
        [
            "gbifID",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "coordinateUncertaintyInMeters",
            "nearest_neighbor_km",
            "institutionCode",
            "basisOfRecord",
        ]
    ]
    .sort_values(
        "coordinateUncertaintyInMeters",
        ascending=False
    )
)

In [ ]:
# ============================================================
# ENVIRONMENTAL RASTER OBJECTS — SAFE CHECK
# ============================================================

raster_objects = []

# list() cria uma cópia antes de iterar
for name, obj in list(globals().items()):
    try:
        if hasattr(obj, "res") and hasattr(obj, "crs"):
            raster_objects.append(name)
    except Exception:
        pass

print("Possible raster objects:")
print(raster_objects)

In [ ]:
from pathlib import Path

# ============================================================
# FIND ENVIRONMENTAL RASTERS
# ============================================================

project_root = Path("..")

raster_files = sorted(
    list(project_root.rglob("*.tif")) +
    list(project_root.rglob("*.tiff"))
)

print(f"Raster files found: {len(raster_files)}\n")

for path in raster_files:
    print(path)

In [ ]:
import rasterio
import pandas as pd
from pathlib import Path

# ============================================================
# CHECK ENVIRONMENTAL RASTER RESOLUTION
# ============================================================

env_dir = Path("../data/processed/environment")

raster_paths = [
    env_dir / "wc2.1_30s_bio_1.tif",
    env_dir / "wc2.1_30s_bio_4.tif",
    env_dir / "wc2.1_30s_bio_12.tif",
    env_dir / "wc2.1_30s_bio_15.tif",
]

raster_info = []

for path in raster_paths:
    with rasterio.open(path) as src:
        raster_info.append({
            "variable": path.stem,
            "crs": str(src.crs),
            "resolution_x_deg": abs(src.res[0]),
            "resolution_y_deg": abs(src.res[1]),
            "width": src.width,
            "height": src.height,
            "left": src.bounds.left,
            "bottom": src.bounds.bottom,
            "right": src.bounds.right,
            "top": src.bounds.top,
        })

raster_info = pd.DataFrame(raster_info)

display(raster_info)

print("\nUnique resolutions:")
print(
    raster_info[
        ["resolution_x_deg", "resolution_y_deg"]
    ].drop_duplicates()
)

In [ ]:
# ============================================================
# COORDINATE UNCERTAINTY × ISOLATION — FINAL DIAGNOSTIC
# ============================================================

pending_test = pending.copy()

def uncertainty_level(x):
    if pd.isna(x):
        return "unknown"
    elif x <= 1000:
        return "high_precision"
    elif x <= 5000:
        return "moderate_precision"
    else:
        return "low_precision"

pending_test["coordinate_precision"] = (
    pending_test["coordinateUncertaintyInMeters"]
    .apply(uncertainty_level)
)

print("=== COORDINATE PRECISION ===")
print(
    pending_test["coordinate_precision"]
    .value_counts()
)

print("\n=== PRECISION × SPATIAL ISOLATION ===")
display(
    pd.crosstab(
        pending_test["coordinate_precision"],
        pending_test["qc_spatial_isolation"],
        margins=True
    )
)

print("\n=== LOW-PRECISION RECORDS (>5 km) ===")

display(
    pending_test.loc[
        pending_test["coordinate_precision"].eq("low_precision"),
        [
            "gbifID",
            "stateProvince",
            "coordinateUncertaintyInMeters",
            "nearest_neighbor_km",
            "qc_spatial_isolation",
            "institutionCode",
            "basisOfRecord",
        ]
    ]
    .sort_values(
        ["qc_spatial_isolation", "coordinateUncertaintyInMeters"],
        ascending=[False, False]
    )
)

In [ ]:
# ============================================================
# QC RULE — EXCLUDE LOW-PRECISION COORDINATES (> 5 km)
# ============================================================

low_precision_mask = (
    occ_thinned["qc_biogeographic_status"].eq("pending")
    &
    occ_thinned["coordinateUncertaintyInMeters"].notna()
    &
    (occ_thinned["coordinateUncertaintyInMeters"] > 5000)
)

print(
    "Records to exclude for low coordinate precision:",
    low_precision_mask.sum()
)

# Guardar classe de precisão no dataframe completo
def classify_coordinate_precision(x):
    if pd.isna(x):
        return "unknown"
    elif x <= 1000:
        return "high_precision"
    elif x <= 5000:
        return "moderate_precision"
    else:
        return "low_precision"

occ_thinned["coordinate_precision"] = (
    occ_thinned["coordinateUncertaintyInMeters"]
    .apply(classify_coordinate_precision)
)

# Atualizar somente os registros ainda pending
occ_thinned.loc[
    low_precision_mask,
    "qc_biogeographic_status"
] = "exclude_low_coordinate_precision"

occ_thinned.loc[
    low_precision_mask,
    "qc_biogeographic_reason"
] = (
    "Reported coordinate uncertainty exceeds 5 km. "
    "Given the approximately 1-km spatial resolution of the environmental "
    "predictor layers, the occurrence location is considered insufficiently "
    "precise for reliable environmental attribution and was excluded from "
    "SDM calibration."
)

print("\n=== UPDATED BIOGEOGRAPHIC QC ===")

print(
    occ_thinned["qc_biogeographic_status"]
    .value_counts(dropna=False)
)

print("\nRemaining pending:")
print(
    occ_thinned["qc_biogeographic_status"]
    .eq("pending")
    .sum()
)

# Checkpoint
occ_thinned.to_csv(
    "../data/processed/"
    "sapajus_libidinosus_occurrences_biogeographic_qc.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nQC dataset updated.")

In [ ]:
# ============================================================
# UNKNOWN COORDINATE UNCERTAINTY — SOURCE AUDIT
# ============================================================

unknown_pending = occ_thinned.loc[
    occ_thinned["qc_biogeographic_status"].eq("pending")
    &
    occ_thinned["coordinate_precision"].eq("unknown")
].copy()

print("Pending records with unknown uncertainty:",
      len(unknown_pending))


# ------------------------------------------------------------
# 1. Institution × basis of record
# ------------------------------------------------------------

source_summary = (
    unknown_pending
    .groupby(
        ["institutionCode", "basisOfRecord"],
        dropna=False
    )
    .agg(
        n=("gbifID", "size"),
        median_nn_km=("nearest_neighbor_km", "median"),
        max_nn_km=("nearest_neighbor_km", "max"),
        isolated=("qc_spatial_isolation", "sum")
    )
    .sort_values("n", ascending=False)
    .reset_index()
)

print("\n=== SOURCE SUMMARY ===")
display(source_summary)


# ------------------------------------------------------------
# 2. Dataset summary
# ------------------------------------------------------------

dataset_summary = (
    unknown_pending
    .groupby(
        ["datasetKey", "institutionCode"],
        dropna=False
    )
    .agg(
        n=("gbifID", "size"),
        isolated=("qc_spatial_isolation", "sum"),
        median_nn_km=("nearest_neighbor_km", "median"),
        max_nn_km=("nearest_neighbor_km", "max")
    )
    .sort_values("n", ascending=False)
    .reset_index()
)

print("\n=== DATASET SUMMARY ===")
display(dataset_summary.head(20))


# ------------------------------------------------------------
# 3. Basis of record
# ------------------------------------------------------------

print("\n=== BASIS OF RECORD ===")

print(
    unknown_pending["basisOfRecord"]
    .value_counts(dropna=False)
)


# ------------------------------------------------------------
# 4. How many are isolated?
# ------------------------------------------------------------

print("\n=== SPATIAL ISOLATION ===")

print(
    unknown_pending["qc_spatial_isolation"]
    .value_counts(dropna=False)
)

In [ ]:
# ============================================================
# UNKNOWN UNCERTAINTY + SPATIAL ISOLATION
# ============================================================

unknown_isolated = (
    unknown_pending.loc[
        unknown_pending["qc_spatial_isolation"].eq(True),
        [
            "gbifID",
            "scientificName",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "year",
            "institutionCode",
            "basisOfRecord",
            "nearest_neighbor_km",
            "datasetKey",
        ]
    ]
    .sort_values("nearest_neighbor_km", ascending=False)
    .reset_index(drop=True)
)

print(
    "Unknown uncertainty + spatial isolation:",
    len(unknown_isolated)
)

display(unknown_isolated)

print("\n=== BY SOURCE ===")

print(
    unknown_isolated[
        ["institutionCode", "basisOfRecord"]
    ]
    .value_counts(dropna=False)
)

In [ ]:
# ============================================================
# ICMBio / SISBio — ISOLATED UNKNOWN-UNCERTAINTY AUDIT
# ============================================================

icmbio_isolated = unknown_isolated.loc[
    unknown_isolated["institutionCode"].eq("ICMBio")
].copy()

icmbio_ids = icmbio_isolated["gbifID"].astype(str)

# Recuperar todas as colunas disponíveis no dataframe principal
icmbio_full = occ_thinned.loc[
    occ_thinned["gbifID"].astype(str).isin(icmbio_ids)
].copy()

print("ICMBio isolated records:", len(icmbio_full))

# Campos potencialmente úteis que realmente existem
candidate_columns = [
    "gbifID",
    "scientificName",
    "stateProvince",
    "municipality",
    "locality",
    "decimalLongitude",
    "decimalLatitude",
    "year",
    "basisOfRecord",
    "institutionCode",
    "datasetKey",
    "coordinateUncertaintyInMeters",
    "nearest_neighbor_km",
    "qc_spatial_isolation",
    "occurrenceStatus",
    "issue",
    "issues",
    "georeferenceRemarks",
    "locationRemarks",
    "recordedBy",
]

available_columns = [
    col for col in candidate_columns
    if col in icmbio_full.columns
]

print("\nAvailable diagnostic columns:")
print(available_columns)

display(
    icmbio_full[available_columns]
    .sort_values("nearest_neighbor_km", ascending=False)
)

In [ ]:
import requests
import pandas as pd
import time

# ============================================================
# GBIF API — ICMBio ISOLATED RECORD METADATA
# ============================================================

icmbio_ids = (
    icmbio_full["gbifID"]
    .astype(str)
    .tolist()
)

records = []

for gbif_id in icmbio_ids:
    
    url = f"https://api.gbif.org/v1/occurrence/{gbif_id}"
    
    response = requests.get(url, timeout=30)
    response.raise_for_status()
    
    r = response.json()
    
    records.append({
        "gbifID": gbif_id,
        "scientificName": r.get("scientificName"),
        "stateProvince": r.get("stateProvince"),
        "municipality": r.get("municipality"),
        "locality": r.get("locality"),
        "decimalLongitude": r.get("decimalLongitude"),
        "decimalLatitude": r.get("decimalLatitude"),
        "coordinateUncertaintyInMeters":
            r.get("coordinateUncertaintyInMeters"),
        "geodeticDatum": r.get("geodeticDatum"),
        "georeferenceRemarks": r.get("georeferenceRemarks"),
        "locationRemarks": r.get("locationRemarks"),
        "recordedBy": r.get("recordedBy"),
        "eventDate": r.get("eventDate"),
        "occurrenceID": r.get("occurrenceID"),
        "catalogNumber": r.get("catalogNumber"),
        "datasetName": r.get("datasetTitle") or r.get("datasetName"),
        "issues": "; ".join(r.get("issues", [])),
    })
    
    time.sleep(0.1)

icmbio_gbif_audit = pd.DataFrame(records)

print("Records retrieved:", len(icmbio_gbif_audit))

display(icmbio_gbif_audit)

In [ ]:
display(
    icmbio_gbif_audit[
        [
            "gbifID",
            "stateProvince",
            "municipality",
            "locality",
            "georeferenceRemarks",
            "locationRemarks",
            "recordedBy",
            "eventDate",
        ]
    ]
)

In [ ]:
# ============================================================
# ICMBio / SISBio — GEOREFERENCE QUALITY DECISIONS
# ============================================================

# Coordenada exata e geograficamente consistente
exact_retain_ids = [
    "6510896727",  # Betânia, PE
]

# Coordenada aproximada — magnitude da incerteza não informada
approximate_ids = [
    "6510102232",
    "6510068097",
    "6510680227",
    "6510453717",
    "6510831505",
    "6510103563",
    "6510461987",
]

# Coordenada representa a área amostrada, não necessariamente
# a posição exata do indivíduo
sampling_area_ids = [
    "6510845047",
    "6510483497",
    "6511168066",
]

# ------------------------------------------------------------
# 1. Retain exact occurrence
# ------------------------------------------------------------

mask_exact = (
    occ_thinned["gbifID"].astype(str).isin(exact_retain_ids)
    &
    occ_thinned["qc_biogeographic_status"].eq("pending")
)

occ_thinned.loc[
    mask_exact,
    "qc_biogeographic_status"
] = "retain_native_verified_georeference"

occ_thinned.loc[
    mask_exact,
    "qc_biogeographic_reason"
] = (
    "ICMBio/SISBio record explicitly documented as 'Coordenada exata "
    "da ocorrência'. The record was therefore retained as geographically "
    "suitable for SDM calibration."
)


# ------------------------------------------------------------
# 2. Exclude approximate coordinates
# ------------------------------------------------------------

mask_approx = (
    occ_thinned["gbifID"].astype(str).isin(approximate_ids)
    &
    occ_thinned["qc_biogeographic_status"].eq("pending")
)

occ_thinned.loc[
    mask_approx,
    "qc_biogeographic_status"
] = "exclude_unquantified_coordinate_precision"

occ_thinned.loc[
    mask_approx,
    "qc_biogeographic_reason"
] = (
    "ICMBio/SISBio metadata describe the location as 'Coordenada aproximada', "
    "but no coordinate uncertainty was quantified. Because compatibility "
    "with the approximately 1-km environmental grid cannot be established, "
    "the record was excluded from the primary SDM calibration dataset."
)


# ------------------------------------------------------------
# 3. Exclude sampling-area reference coordinates
# ------------------------------------------------------------

mask_area = (
    occ_thinned["gbifID"].astype(str).isin(sampling_area_ids)
    &
    occ_thinned["qc_biogeographic_status"].eq("pending")
)

occ_thinned.loc[
    mask_area,
    "qc_biogeographic_status"
] = "exclude_sampling_area_coordinate"

occ_thinned.loc[
    mask_area,
    "qc_biogeographic_reason"
] = (
    "ICMBio/SISBio metadata identify the coordinate as a reference point "
    "for the sampled area ('Coordenada de referência da área amostrada'), "
    "rather than an explicitly exact occurrence location. The record was "
    "excluded from the primary SDM calibration dataset."
)


# ------------------------------------------------------------
# Check
# ------------------------------------------------------------

icmbio_ids = [
    "6510102232", "6510680227", "6510896727",
    "6510483497", "6510068097", "6510461987",
    "6510845047", "6510453717", "6510103563",
    "6510831505", "6510188564", "6511168066"
]

display(
    occ_thinned.loc[
        occ_thinned["gbifID"].astype(str).isin(icmbio_ids),
        [
            "gbifID",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "qc_biogeographic_status",
            "qc_biogeographic_reason",
        ]
    ]
)

print("\nRemaining pending:")
print(
    occ_thinned["qc_biogeographic_status"]
    .eq("pending")
    .sum()
)

# checkpoint
occ_thinned.to_csv(
    "../data/processed/"
    "sapajus_libidinosus_occurrences_biogeographic_qc.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nCheckpoint atualizado.")

In [ ]:
# ============================================================
# GBIF 6510188564 — INTERNAL GEOGRAPHIC METADATA CONFLICT
# ============================================================

target_id = "6510188564"

mask = (
    occ_thinned["gbifID"].astype(str).eq(target_id)
    &
    occ_thinned["qc_biogeographic_status"].eq("pending")
)

occ_thinned.loc[
    mask,
    "qc_biogeographic_status"
] = "exclude_geographic_metadata_conflict"

occ_thinned.loc[
    mask,
    "qc_biogeographic_reason"
] = (
    "ICMBio/SISBio record excluded because of internally inconsistent "
    "geographic metadata. The record is reported as Ananindeua, Pará, "
    "while the supplied coordinates (-48.380833, -13.838890) indicate "
    "a substantially different geographic location. Although the source "
    "describes the coordinates as an exact occurrence location, the "
    "conflicting administrative geography prevents reliable assignment "
    "of the occurrence for SDM calibration."
)

# Conferir
display(
    occ_thinned.loc[
        mask,
        [
            "gbifID",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "qc_biogeographic_status",
            "qc_biogeographic_reason",
        ]
    ]
)

print("\nRemaining pending:")
print(
    occ_thinned["qc_biogeographic_status"]
    .eq("pending")
    .sum()
)

# Salvar checkpoint
occ_thinned.to_csv(
    "../data/processed/"
    "sapajus_libidinosus_occurrences_biogeographic_qc.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\nCheckpoint atualizado.")

In [ ]:
# ============================================================
# NEXT BATCH — ISOLATED iNATURALIST WITH UNKNOWN UNCERTAINTY
# ============================================================

inat_isolated = occ_thinned.loc[
    occ_thinned["qc_biogeographic_status"].eq("pending")
    &
    occ_thinned["coordinate_precision"].eq("unknown")
    &
    occ_thinned["qc_spatial_isolation"].eq(True)
    &
    occ_thinned["institutionCode"].eq("iNaturalist")
].copy()

inat_isolated = inat_isolated.sort_values(
    "nearest_neighbor_km",
    ascending=False
)

print("iNaturalist isolated records:", len(inat_isolated))

display(
    inat_isolated[
        [
            "gbifID",
            "scientificName",
            "stateProvince",
            "decimalLongitude",
            "decimalLatitude",
            "year",
            "nearest_neighbor_km",
            "basisOfRecord",
        ]
    ]
)

In [ ]:
# ============================================================
# UPDATE — USNM ISOLATED RECORDS VERIFIED IN PRIMARY SOURCE
# ============================================================

usnm_verified = {
    "1317480884": (
        "Primary-source verification in the Smithsonian NMNH mammal "
        "collection confirms a preserved specimen of Sapajus libidinosus "
        "from Pernambuco. Museum locality and coordinates are consistent "
        "with the GBIF occurrence; spatial isolation alone does not justify exclusion."
    ),

    "1320458686": (
        "Primary-source verification in the Smithsonian NMNH mammal "
        "collection confirms a preserved specimen of Sapajus libidinosus "
        "from Goiás. Museum locality and coordinates are consistent "
        "with the GBIF occurrence; spatial isolation alone does not justify exclusion."
    ),

    "1322442680": (
        "Primary-source verification in the Smithsonian NMNH mammal "
        "collection confirms a preserved specimen of Sapajus libidinosus "
        "from Minas Gerais. Museum locality and coordinates are consistent "
        "with the GBIF occurrence; spatial isolation alone does not justify exclusion."
    ),

    "1318994983": (
        "Primary-source verification in the Smithsonian NMNH mammal "
        "collection confirms preserved specimen USNM 518418 from Araguari, "
        "Piracaiba, Minas Gerais. Museum locality and coordinates are consistent "
        "with the GBIF occurrence; spatial isolation alone does not justify exclusion."
    ),
}


# ------------------------------------------------------------
# Apply decisions
# ------------------------------------------------------------

for gbif_id, reason in usnm_verified.items():

    mask = (
        occ_thinned["gbifID"].astype(str).eq(gbif_id)
        &
        occ_thinned["qc_biogeographic_status"].eq("pending")
    )

    occ_thinned.loc[
        mask,
        "qc_biogeographic_status"
    ] = "retain_native_verified_specimen"

    occ_thinned.loc[
        mask,
        "qc_biogeographic_reason"
    ] = reason


# ------------------------------------------------------------
# Check updated records
# ------------------------------------------------------------

check_usnm = occ_thinned.loc[
    occ_thinned["gbifID"].astype(str).isin(usnm_verified.keys()),
    [
        "gbifID",
        "stateProvince",
        "decimalLongitude",
        "decimalLatitude",
        "nearest_neighbor_km",
        "qc_biogeographic_status",
        "qc_biogeographic_reason",
    ]
]

display(check_usnm)


# ------------------------------------------------------------
# Remaining pending
# ------------------------------------------------------------

n_pending = (
    occ_thinned["qc_biogeographic_status"]
    .eq("pending")
    .sum()
)

print("\nRemaining pending:", n_pending)


# ------------------------------------------------------------
# Save checkpoint
# ------------------------------------------------------------

output_path = (
    "../data/processed/"
    "sapajus_libidinosus_occurrences_biogeographic_qc.csv"
)

occ_thinned.to_csv(
    output_path,
    index=False
)

print("Checkpoint atualizado.")

In [ ]:
# ============================================================
# FINAL UNKNOWN-UNCERTAINTY + ISOLATED RECORDS
# ============================================================

remaining_isolated_unknown = occ_thinned.loc[
    occ_thinned["qc_biogeographic_status"].eq("pending")
    &
    occ_thinned["coordinate_precision"].eq("unknown")
    &
    occ_thinned["qc_spatial_isolation"].eq(True),
    [
        "gbifID",
        "scientificName",
        "stateProvince",
        "decimalLongitude",
        "decimalLatitude",
        "year",
        "institutionCode",
        "basisOfRecord",
        "nearest_neighbor_km",
        "datasetKey",
    ]
].copy()

remaining_isolated_unknown = (
    remaining_isolated_unknown
    .sort_values("nearest_neighbor_km", ascending=False)
)

print(
    "Remaining unknown-uncertainty + isolated records:",
    len(remaining_isolated_unknown)
)

display(remaining_isolated_unknown)

In [ ]:
# ============================================================
# UPDATE — iNATURALIST ISOLATED RECORDS VERIFIED BY PHOTOGRAPH
# ============================================================

inat_verified = [
    "5868150587",
    "6178771434",
    "5230211588",
    "5838037692",
    "6133027127",
]

mask = (
    occ_thinned["gbifID"].astype(str).isin(inat_verified)
    &
    occ_thinned["qc_biogeographic_status"].eq("pending")
)

occ_thinned.loc[
    mask,
    "qc_biogeographic_status"
] = "retain_native_verified_observation"

occ_thinned.loc[
    mask,
    "qc_biogeographic_reason"
] = (
    "Research-grade iNaturalist occurrence independently verified "
    "through photographic evidence associated with the original record. "
    "The photographic documentation supports the taxonomic identification; "
    "spatial isolation alone does not justify exclusion."
)

print("iNaturalist records updated:", mask.sum())

display(
    occ_thinned.loc[
        occ_thinned["gbifID"].astype(str).isin(inat_verified),
        [
            "gbifID",
            "stateProvince",
            "nearest_neighbor_km",
            "qc_biogeographic_status",
            "qc_biogeographic_reason",
        ]
    ]
)

print(
    "\nRemaining pending:",
    occ_thinned["qc_biogeographic_status"].eq("pending").sum()
)

# checkpoint
occ_thinned.to_csv(
    "../data/processed/"
    "sapajus_libidinosus_occurrences_biogeographic_qc.csv",
    index=False
)

print("Checkpoint atualizado.")

In [ ]:
# ============================================================
# FINAL 4 — UNKNOWN UNCERTAINTY + SPATIAL ISOLATION
# ============================================================

final_isolated = occ_thinned.loc[
    occ_thinned["qc_biogeographic_status"].eq("pending")
    &
    occ_thinned["coordinate_precision"].eq("unknown")
    &
    occ_thinned["qc_spatial_isolation"].eq(True)
].copy()

print("Remaining isolated records:", len(final_isolated))

# Campos potencialmente úteis que realmente existem no dataframe
wanted_cols = [
    "gbifID",
    "scientificName",
    "stateProvince",
    "municipality",
    "locality",
    "decimalLongitude",
    "decimalLatitude",
    "year",
    "institutionCode",
    "basisOfRecord",
    "datasetKey",
    "occurrenceID",
    "catalogNumber",
    "recordedBy",
    "nearest_neighbor_km",
    "issue",
]

available_cols = [
    col for col in wanted_cols
    if col in final_isolated.columns
]

display(
    final_isolated[available_cols]
    .sort_values("nearest_neighbor_km", ascending=False)
)

print("\n=== BY SOURCE ===")
print(
    final_isolated[
        ["institutionCode", "basisOfRecord"]
    ].value_counts(dropna=False)
)

In [ ]:
# ============================================================
# UnB 1988590681 — VERIFIED INSTITUTIONAL OCCURRENCE
# ============================================================

gbif_id = "1988590681"

mask = (
    occ_thinned["gbifID"].astype(str).eq(gbif_id)
    &
    occ_thinned["qc_biogeographic_status"].eq("pending")
)

occ_thinned.loc[
    mask,
    "qc_biogeographic_status"
] = "retain_native_verified_observation"

occ_thinned.loc[
    mask,
    "qc_biogeographic_reason"
] = (
    "Institutional UnB mastofauna record documents Sapajus libidinosus "
    "as present in the Médio Araguaia region, Cocalinho, Mato Grosso. "
    "The occurrence has an institutional occurrence identifier and "
    "explicit occurrence remarks reporting individuals. Spatial isolation "
    "alone does not justify exclusion. The published coordinate represents "
    "the centroid of the sampled perimeter and is therefore retained with "
    "a spatial-representativeness flag."
)

# Create flag if necessary
if "qc_coordinate_representativeness" not in occ_thinned.columns:
    occ_thinned["qc_coordinate_representativeness"] = "point_or_unspecified"

occ_thinned.loc[
    mask,
    "qc_coordinate_representativeness"
] = "centroid_of_sampling_perimeter"


# Check
display(
    occ_thinned.loc[
        occ_thinned["gbifID"].astype(str).eq(gbif_id),
        [
            "gbifID",
            "stateProvince",
            "nearest_neighbor_km",
            "qc_biogeographic_status",
            "qc_coordinate_representativeness",
            "qc_biogeographic_reason",
        ]
    ]
)

print(
    "\nRemaining pending:",
    occ_thinned["qc_biogeographic_status"].eq("pending").sum()
)

# Save checkpoint
occ_thinned.to_csv(
    "../data/processed/"
    "sapajus_libidinosus_occurrences_biogeographic_qc.csv",
    index=False
)

print("Checkpoint atualizado.")

In [ ]:
import pandas as pd

occ_thinned = pd.read_csv(
    "../data/processed/sapajus_libidinosus_occurrences_biogeographic_qc.csv"
)

print("Registros:", len(occ_thinned))
print(
    "Remaining pending:",
    occ_thinned["qc_biogeographic_status"].eq("pending").sum()
)

In [ ]:
gbif_id = "5863633723"

mask = occ_thinned["gbifID"].astype(str).eq(gbif_id)

occ_thinned.loc[
    mask, "qc_biogeographic_status"
] = "retain_native_verified_genetic_sample"

occ_thinned.loc[
    mask, "qc_biogeographic_reason"
] = (
    "Genetic sample independently verified in the primary phylogenomic "
    "study of Lima et al. (2018), which reports Sapajus libidinosus at "
    "the exact coordinates -14.14, -48.17."
)

# Conferir
occ_thinned.loc[
    mask,
    [
        "gbifID",
        "scientificName",
        "stateProvince",
        "decimalLatitude",
        "decimalLongitude",
        "nearest_neighbor_km",
        "qc_biogeographic_status",
        "qc_biogeographic_reason",
    ],
]

In [ ]:
# Save checkpoint
occ_thinned.to_csv(
    "../data/processed/sapajus_libidinosus_occurrences_biogeographic_qc.csv",
    index=False
)

print(
    "Remaining pending:",
    occ_thinned["qc_biogeographic_status"].eq("pending").sum()
)

print("Checkpoint atualizado.")

In [ ]:
xeno_ids = ["5995532354", "5995531874"]

occ_thinned.loc[
    occ_thinned["gbifID"].astype(str).isin(xeno_ids),
    [
        "gbifID",
        "stateProvince",
        "locality",
        "decimalLatitude",
        "decimalLongitude",
        "nearest_neighbor_km",
        "qc_biogeographic_status",
        "qc_biogeographic_reason",
    ],
]

In [ ]:
xeno_ids = ["5995532354", "5995531874"]

mask_xeno = occ_thinned["gbifID"].astype(str).isin(xeno_ids)

occ_thinned.loc[
    mask_xeno, "qc_biogeographic_status"
] = "retain_native_verified_acoustic_record"

occ_thinned.loc[
    mask_xeno, "qc_biogeographic_reason"
] = (
    "Primary-source Xeno-canto acoustic record independently verifies "
    "Sapajus libidinosus at the reported locality, date, and coordinates; "
    "retained despite spatial isolation and unreported coordinate uncertainty."
)

# Conferência
occ_thinned.loc[
    mask_xeno,
    [
        "gbifID",
        "locality",
        "decimalLatitude",
        "decimalLongitude",
        "nearest_neighbor_km",
        "qc_biogeographic_status",
        "qc_biogeographic_reason",
    ],
]

In [ ]:
occ_thinned.to_csv(
    "../data/processed/sapajus_libidinosus_occurrences_biogeographic_qc.csv",
    index=False
)

print(
    "Remaining pending:",
    occ_thinned["qc_biogeographic_status"].eq("pending").sum()
)

print("Checkpoint atualizado.")

In [ ]:
pending = occ_thinned.loc[
    occ_thinned["qc_biogeographic_status"].eq("pending")
].copy()

print("Total pending:", len(pending))

summary_pending = (
    pending
    .groupby(
        [
            "coordinate_precision",
            "qc_spatial_isolation",
            "institutionCode",
            "basisOfRecord"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="n")
    .sort_values(
        ["coordinate_precision", "qc_spatial_isolation", "n"],
        ascending=[True, True, False]
    )
)

summary_pending

In [ ]:
pending_known_precision = pending.loc[
    pending["coordinate_precision"].isin(
        ["high_precision", "moderate_precision"]
    )
].copy()

print("Known precision pending:", len(pending_known_precision))

(
    pending_known_precision
    .groupby("coordinate_precision")["coordinateUncertaintyInMeters"]
    .agg(["count", "min", "median", "max"])
)

In [ ]:
(
    pending_known_precision
    .groupby(
        [
            "coordinate_precision",
            "qc_spatial_isolation",
            "institutionCode",
            "basisOfRecord"
        ],
        dropna=False
    )
    .size()
    .reset_index(name="n")
)

In [ ]:
known_isolated = pending.loc[
    pending["coordinate_precision"].isin(
        ["high_precision", "moderate_precision"]
    )
    & pending["qc_spatial_isolation"].eq(True)
].copy()

print("Known-precision isolated:", len(known_isolated))

known_isolated[
    [
        "gbifID",
        "institutionCode",
        "basisOfRecord",
        "stateProvince",
        "locality",
        "decimalLatitude",
        "decimalLongitude",
        "coordinateUncertaintyInMeters",
        "coordinate_precision",
        "nearest_neighbor_km",
    ]
].sort_values(
    "nearest_neighbor_km",
    ascending=False
)

In [ ]:
cols_check = [
    "gbifID",
    "institutionCode",
    "basisOfRecord",
    "year",
    "occurrenceStatus",
    "coordinateUncertaintyInMeters",
    "coordinate_precision",
    "nearest_neighbor_km",
    "issue",
]

known_isolated[cols_check].sort_values(
    "nearest_neighbor_km",
    ascending=False
)

In [ ]:
print("Total:", len(known_isolated))

print("\nPor fonte:")
print(
    known_isolated["institutionCode"]
    .fillna("NA")
    .value_counts()
)

print("\nOccurrence status:")
print(
    known_isolated["occurrenceStatus"]
    .fillna("NA")
    .value_counts()
)

print("\nPrecision:")
print(
    known_isolated["coordinate_precision"]
    .value_counts()
)

In [ ]:
mask_known_isolated = (
    occ_thinned["qc_biogeographic_status"].eq("pending")
    & occ_thinned["coordinate_precision"].isin(
        ["high_precision", "moderate_precision"]
    )
    & occ_thinned["qc_spatial_isolation"].eq(True)
)

print("Registros selecionados:", mask_known_isolated.sum())

In [ ]:
occ_thinned.loc[
    mask_known_isolated,
    "qc_biogeographic_status"
] = "retain_native_known_precision"

occ_thinned.loc[
    mask_known_isolated,
    "qc_biogeographic_reason"
] = (
    "Native-range occurrence with quantified coordinate uncertainty "
    "(<= 5 km) and no critical spatial QC issue; retained despite "
    "spatial isolation."
)

In [ ]:
occ_thinned.to_csv(
    "../data/processed/sapajus_libidinosus_occurrences_biogeographic_qc.csv",
    index=False
)

print("Checkpoint atualizado.")

In [ ]:
occ_thinned.loc[
    mask_known_isolated,
    "qc_biogeographic_status"
] = "retain_native_known_precision"

occ_thinned.loc[
    mask_known_isolated,
    "qc_biogeographic_reason"
] = (
    "Native-range occurrence with quantified coordinate uncertainty "
    "(<= 5 km) and no critical spatial QC issue; retained despite "
    "spatial isolation."
)

In [ ]:
print(
    occ_thinned.loc[
        mask_known_isolated,
        "qc_biogeographic_status"
    ].value_counts()
)

print(
    "\nRemaining pending:",
    occ_thinned["qc_biogeographic_status"].eq("pending").sum()
)

In [ ]:
occ_thinned.to_csv(
    "../data/processed/sapajus_libidinosus_occurrences_biogeographic_qc.csv",
    index=False
)

print("Checkpoint atualizado.")

In [ ]:
known_nonisolated = occ_thinned.loc[
    occ_thinned["qc_biogeographic_status"].eq("pending")
    & occ_thinned["coordinate_precision"].isin(
        ["high_precision", "moderate_precision"]
    )
    & occ_thinned["qc_spatial_isolation"].eq(False)
].copy()

print("Known-precision non-isolated:", len(known_nonisolated))

print("\nPor fonte:")
print(
    known_nonisolated["institutionCode"]
    .fillna("NA")
    .value_counts()
)

print("\nOccurrence status:")
print(
    known_nonisolated["occurrenceStatus"]
    .fillna("NA")
    .value_counts()
)

print("\nPrecision:")
print(
    known_nonisolated["coordinate_precision"]
    .value_counts()
)

print("\nIssues:")
print(
    known_nonisolated["issue"]
    .fillna("NO_ISSUE")
    .value_counts()
)

In [ ]:
mask_known_nonisolated = (
    occ_thinned["qc_biogeographic_status"].eq("pending")
    & occ_thinned["coordinate_precision"].isin(
        ["high_precision", "moderate_precision"]
    )
    & occ_thinned["qc_spatial_isolation"].eq(False)
)

print("Registros selecionados:", mask_known_nonisolated.sum())

In [ ]:
occ_thinned.loc[
    mask_known_nonisolated,
    "qc_biogeographic_status"
] = "retain_native_known_precision"

occ_thinned.loc[
    mask_known_nonisolated,
    "qc_biogeographic_reason"
] = (
    "Native-range occurrence with quantified coordinate uncertainty "
    "(<= 5 km), no critical spatial QC issue, and spatial support from "
    "neighboring occurrence records."
)

In [ ]:
print(
    occ_thinned.loc[
        mask_known_nonisolated,
        "qc_biogeographic_status"
    ].value_counts()
)

print(
    "\nRemaining pending:",
    occ_thinned["qc_biogeographic_status"].eq("pending").sum()
)

In [ ]:
occ_thinned.to_csv(
    "../data/processed/sapajus_libidinosus_occurrences_biogeographic_qc.csv",
    index=False
)

print("Checkpoint atualizado.")

In [ ]:
icmbio_pending = occ_thinned.loc[
    occ_thinned["qc_biogeographic_status"].eq("pending")
    & occ_thinned["institutionCode"].eq("ICMBio")
].copy()

print("ICMBio pending:", len(icmbio_pending))

# Colunas relacionadas à georreferência
[
    col for col in icmbio_pending.columns
    if any(
        term in col.lower()
        for term in [
            "georef",
            "coordinate",
            "locality",
            "location",
            "decimal"
        ]
    )
]

In [ ]:
icmbio_pending["georeferenceRemarks"].value_counts(dropna=False)

In [ ]:
import requests

print(requests.__version__)

In [ ]:
import requests
import pandas as pd
import time

records = []

for gbif_id in icmbio_pending["gbifID"].astype(str):

    url = f"https://api.gbif.org/v1/occurrence/{gbif_id}"

    try:
        response = requests.get(url, timeout=20)
        response.raise_for_status()
        data = response.json()

        records.append({
            "gbifID": gbif_id,
            "georeferenceRemarks": data.get("georeferenceRemarks"),
            "georeferenceProtocol": data.get("georeferenceProtocol"),
            "georeferencedBy": data.get("georeferencedBy"),
            "coordinateUncertaintyInMeters": data.get(
                "coordinateUncertaintyInMeters"
            ),
            "locality": data.get("locality"),
            "stateProvince": data.get("stateProvince"),
        })

    except requests.RequestException as e:
        print(f"Erro no GBIF {gbif_id}: {e}")

    time.sleep(0.1)

icmbio_metadata = pd.DataFrame(records)

print("Registros recuperados:", len(icmbio_metadata))
print("\nGeoreference remarks:")
print(
    icmbio_metadata["georeferenceRemarks"]
    .fillna("NA")
    .value_counts()
)

In [ ]:
# Criar mapa gbifID -> georeferenceRemarks
remarks_map = dict(
    zip(
        icmbio_metadata["gbifID"].astype(str),
        icmbio_metadata["georeferenceRemarks"]
    )
)

# Criar coluna de auditoria, caso ainda não exista
if "qc_georeference_remarks" not in occ_thinned.columns:
    occ_thinned["qc_georeference_remarks"] = pd.NA

mask_icmbio_pending = (
    occ_thinned["qc_biogeographic_status"].eq("pending")
    & occ_thinned["institutionCode"].eq("ICMBio")
)

occ_thinned.loc[
    mask_icmbio_pending,
    "qc_georeference_remarks"
] = (
    occ_thinned.loc[mask_icmbio_pending, "gbifID"]
    .astype(str)
    .map(remarks_map)
)

print(
    occ_thinned.loc[
        mask_icmbio_pending,
        "qc_georeference_remarks"
    ].value_counts(dropna=False)
)

In [ ]:
# 1. Coordenada exata -> RETAIN
mask_exact = (
    mask_icmbio_pending
    & occ_thinned["qc_georeference_remarks"]
        .eq("Coordenada exata da ocorrência")
)

occ_thinned.loc[
    mask_exact,
    "qc_biogeographic_status"
] = "retain_native_verified_georeference"

occ_thinned.loc[
    mask_exact,
    "qc_biogeographic_reason"
] = (
    "ICMBio source metadata explicitly identifies the coordinate "
    "as the exact occurrence location."
)


# 2. Coordenada aproximada -> EXCLUDE
mask_approx = (
    mask_icmbio_pending
    & occ_thinned["qc_georeference_remarks"]
        .eq("Coordenada aproximada")
)

occ_thinned.loc[
    mask_approx,
    "qc_biogeographic_status"
] = "exclude_unquantified_coordinate_precision"

occ_thinned.loc[
    mask_approx,
    "qc_biogeographic_reason"
] = (
    "ICMBio source metadata identifies the coordinate as approximate, "
    "but no quantitative coordinate uncertainty is provided."
)


# 3. Referência da área amostrada -> EXCLUDE
mask_sampling = (
    mask_icmbio_pending
    & occ_thinned["qc_georeference_remarks"]
        .eq("Coordenada de referência da área amostrada")
)

occ_thinned.loc[
    mask_sampling,
    "qc_biogeographic_status"
] = "exclude_sampling_area_coordinate"

occ_thinned.loc[
    mask_sampling,
    "qc_biogeographic_reason"
] = (
    "ICMBio source metadata identifies the coordinate as a reference "
    "for the sampled area rather than the exact occurrence location."
)

In [ ]:
print("Exact retained:", mask_exact.sum())
print("Approximate excluded:", mask_approx.sum())
print("Sampling-area excluded:", mask_sampling.sum())

print(
    "\nRemaining pending:",
    occ_thinned["qc_biogeographic_status"].eq("pending").sum()
)

In [ ]:
occ_thinned.to_csv(
    "../data/processed/sapajus_libidinosus_occurrences_biogeographic_qc.csv",
    index=False
)

print("Checkpoint atualizado.")

In [ ]:
remaining = occ_thinned.loc[
    occ_thinned["qc_biogeographic_status"].eq("pending")
].copy()

print("Remaining pending:", len(remaining))

remaining_summary = (
    remaining
    .groupby(
        ["institutionCode", "basisOfRecord"],
        dropna=False
    )
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)

remaining_summary

In [ ]:
inat_pending = occ_thinned.loc[
    occ_thinned["qc_biogeographic_status"].eq("pending")
    & occ_thinned["institutionCode"].eq("iNaturalist")
].copy()

print("iNaturalist pending:", len(inat_pending))

In [ ]:
inat_records = []

for gbif_id in inat_pending["gbifID"].astype(str):

    url = f"https://api.gbif.org/v1/occurrence/{gbif_id}"

    try:
        response = requests.get(url, timeout=20)
        response.raise_for_status()
        data = response.json()

        inat_records.append({
            "gbifID": gbif_id,
            "occurrenceID": data.get("occurrenceID"),
            "references": data.get("references"),
            "eventDate": data.get("eventDate"),
            "decimalLatitude": data.get("decimalLatitude"),
            "decimalLongitude": data.get("decimalLongitude"),
            "coordinateUncertaintyInMeters":
                data.get("coordinateUncertaintyInMeters"),
            "georeferenceRemarks": data.get("georeferenceRemarks"),
            "identificationVerificationStatus":
                data.get("identificationVerificationStatus"),
            "occurrenceStatus": data.get("occurrenceStatus"),
            "issue": ";".join(data.get("issues", []))
                if isinstance(data.get("issues"), list)
                else data.get("issues"),
        })

    except requests.RequestException as e:
        print(f"Erro no GBIF {gbif_id}: {e}")

    time.sleep(0.1)

inat_metadata = pd.DataFrame(inat_records)

print("Registros recuperados:", len(inat_metadata))

In [ ]:
inat_metadata[
    [
        "gbifID",
        "occurrenceID",
        "coordinateUncertaintyInMeters",
        "georeferenceRemarks",
        "identificationVerificationStatus",
        "occurrenceStatus",
    ]
]

In [ ]:
mask_inat_unknown = (
    occ_thinned["qc_biogeographic_status"].eq("pending")
    & occ_thinned["institutionCode"].eq("iNaturalist")
    & occ_thinned["coordinate_precision"].eq("unknown")
    & occ_thinned["qc_spatial_isolation"].eq(False)
)

print("Selecionados:", mask_inat_unknown.sum())

In [ ]:
occ_thinned.loc[
    mask_inat_unknown,
    "qc_biogeographic_status"
] = "retain_native_source_supported"

occ_thinned.loc[
    mask_inat_unknown,
    "qc_biogeographic_reason"
] = (
    "Traceable iNaturalist observation within the native-range occurrence "
    "cluster; retained despite unreported coordinate uncertainty because "
    "the record is not spatially isolated."
)

In [ ]:
print(
    occ_thinned.loc[
        mask_inat_unknown,
        "qc_biogeographic_status"
    ].value_counts()
)

print(
    "\nRemaining pending:",
    occ_thinned["qc_biogeographic_status"].eq("pending").sum()
)

In [ ]:
occ_thinned.to_csv(
    "../data/processed/sapajus_libidinosus_occurrences_biogeographic_qc.csv",
    index=False
)

print("Checkpoint atualizado.")

In [ ]:
usnm_pending = occ_thinned.loc[
    occ_thinned["qc_biogeographic_status"].eq("pending")
    & occ_thinned["institutionCode"].eq("USNM")
].copy()

print("USNM pending:", len(usnm_pending))

usnm_records = []

for gbif_id in usnm_pending["gbifID"].astype(str):

    url = f"https://api.gbif.org/v1/occurrence/{gbif_id}"

    try:
        response = requests.get(url, timeout=20)
        response.raise_for_status()
        data = response.json()

        usnm_records.append({
            "gbifID": gbif_id,
            "catalogNumber": data.get("catalogNumber"),
            "occurrenceID": data.get("occurrenceID"),
            "scientificName": data.get("scientificName"),
            "stateProvince": data.get("stateProvince"),
            "locality": data.get("locality"),
            "decimalLatitude": data.get("decimalLatitude"),
            "decimalLongitude": data.get("decimalLongitude"),
            "coordinateUncertaintyInMeters":
                data.get("coordinateUncertaintyInMeters"),
            "year": data.get("year"),
            "recordedBy": data.get("recordedBy"),
            "references": data.get("references"),
        })

    except requests.RequestException as e:
        print(f"Erro no GBIF {gbif_id}: {e}")

    time.sleep(0.1)

usnm_metadata = pd.DataFrame(usnm_records)

print("Registros recuperados:", len(usnm_metadata))

usnm_metadata[
    [
        "gbifID",
        "catalogNumber",
        "stateProvince",
        "locality",
        "decimalLatitude",
        "decimalLongitude",
        "year",
        "recordedBy",
    ]
]

In [ ]:
mask_usnm = (
    occ_thinned["qc_biogeographic_status"].eq("pending")
    & occ_thinned["institutionCode"].eq("USNM")
)

print("Selecionados:", mask_usnm.sum())

In [ ]:
occ_thinned.loc[
    mask_usnm,
    "qc_biogeographic_status"
] = "retain_native_source_supported"

occ_thinned.loc[
    mask_usnm,
    "qc_biogeographic_reason"
] = (
    "Traceable USNM preserved specimen with individual catalog number, "
    "specific collection locality, and coordinates consistent with the "
    "native range; retained despite unreported coordinate uncertainty."
)

In [ ]:
print(
    occ_thinned.loc[
        mask_usnm,
        "qc_biogeographic_status"
    ].value_counts()
)

print(
    "\nRemaining pending:",
    occ_thinned["qc_biogeographic_status"].eq("pending").sum()
)

In [ ]:
occ_thinned.to_csv(
    "../data/processed/sapajus_libidinosus_occurrences_biogeographic_qc.csv",
    index=False
)

print("Checkpoint atualizado.")

In [ ]:
genbank_pending = occ_thinned.loc[
    occ_thinned["qc_biogeographic_status"].eq("pending")
    & occ_thinned["institutionCode"].eq("Mined from GenBank, NCBI")
].copy()

print("GenBank pending:", len(genbank_pending))

genbank_records = []

for gbif_id in genbank_pending["gbifID"].astype(str):

    url = f"https://api.gbif.org/v1/occurrence/{gbif_id}"

    try:
        response = requests.get(url, timeout=20)
        response.raise_for_status()
        data = response.json()

        genbank_records.append({
            "gbifID": gbif_id,
            "catalogNumber": data.get("catalogNumber"),
            "occurrenceID": data.get("occurrenceID"),
            "otherCatalogNumbers": data.get("otherCatalogNumbers"),
            "associatedSequences": data.get("associatedSequences"),
            "references": data.get("references"),
            "scientificName": data.get("scientificName"),
            "stateProvince": data.get("stateProvince"),
            "locality": data.get("locality"),
            "decimalLatitude": data.get("decimalLatitude"),
            "decimalLongitude": data.get("decimalLongitude"),
            "year": data.get("year"),
            "recordedBy": data.get("recordedBy"),
            "identifiedBy": data.get("identifiedBy"),
        })

    except requests.RequestException as e:
        print(f"Erro no GBIF {gbif_id}: {e}")

    time.sleep(0.1)

genbank_metadata = pd.DataFrame(genbank_records)

print("Registros recuperados:", len(genbank_metadata))

genbank_metadata[
    [
        "gbifID",
        "catalogNumber",
        "occurrenceID",
        "otherCatalogNumbers",
        "associatedSequences",
        "stateProvince",
        "locality",
        "decimalLatitude",
        "decimalLongitude",
        "year",
    ]
]

In [ ]:
mask_genbank = (
    occ_thinned["qc_biogeographic_status"].eq("pending")
    & occ_thinned["institutionCode"].eq("Mined from GenBank, NCBI")
)

print("Selecionados:", mask_genbank.sum())

In [ ]:
occ_thinned.loc[
    mask_genbank,
    "qc_biogeographic_status"
] = "retain_native_verified_genetic_sample"

occ_thinned.loc[
    mask_genbank,
    "qc_biogeographic_reason"
] = (
    "Genetic sample independently verified in the primary phylogenomic "
    "study of Lima et al. (2018), which reports Sapajus libidinosus "
    "at the exact coordinates associated with the GBIF record."
)

In [ ]:
print(
    occ_thinned.loc[
        mask_genbank,
        "qc_biogeographic_status"
    ].value_counts()
)

print(
    "\nRemaining pending:",
    occ_thinned["qc_biogeographic_status"].eq("pending").sum()
)

In [ ]:
occ_thinned.to_csv(
    "../data/processed/sapajus_libidinosus_occurrences_biogeographic_qc.csv",
    index=False
)

print("Checkpoint atualizado.")

In [ ]:
brandt_pending = occ_thinned.loc[
    occ_thinned["qc_biogeographic_status"].eq("pending")
    & occ_thinned["institutionCode"].eq("Brandt Meio Ambiente")
].copy()

print("Brandt pending:", len(brandt_pending))

brandt_pending[
    [
        "gbifID",
        "basisOfRecord",
        "stateProvince",
        "locality",
        "decimalLatitude",
        "decimalLongitude",
        "year",
        "nearest_neighbor_km",
        "qc_spatial_isolation",
        "issue",
    ]
]

In [ ]:
brandt_records = []

for gbif_id in brandt_pending["gbifID"].astype(str):

    url = f"https://api.gbif.org/v1/occurrence/{gbif_id}"

    try:
        response = requests.get(url, timeout=20)
        response.raise_for_status()
        data = response.json()

        brandt_records.append({
            "gbifID": gbif_id,
            "datasetTitle": data.get("datasetTitle"),
            "datasetKey": data.get("datasetKey"),
            "institutionCode": data.get("institutionCode"),
            "collectionCode": data.get("collectionCode"),
            "catalogNumber": data.get("catalogNumber"),
            "occurrenceID": data.get("occurrenceID"),
            "recordedBy": data.get("recordedBy"),
            "identifiedBy": data.get("identifiedBy"),
            "eventDate": data.get("eventDate"),
            "samplingProtocol": data.get("samplingProtocol"),
            "eventRemarks": data.get("eventRemarks"),
            "occurrenceRemarks": data.get("occurrenceRemarks"),
            "locality": data.get("locality"),
            "verbatimLocality": data.get("verbatimLocality"),
            "decimalLatitude": data.get("decimalLatitude"),
            "decimalLongitude": data.get("decimalLongitude"),
            "coordinateUncertaintyInMeters":
                data.get("coordinateUncertaintyInMeters"),
            "georeferenceRemarks": data.get("georeferenceRemarks"),
            "references": data.get("references"),
        })

    except requests.RequestException as e:
        print(f"Erro no GBIF {gbif_id}: {e}")

    time.sleep(0.1)

brandt_metadata = pd.DataFrame(brandt_records)

print("Registros recuperados:", len(brandt_metadata))

In [ ]:
brandt_metadata[
    [
        "gbifID",
        "datasetTitle",
        "catalogNumber",
        "recordedBy",
        "eventDate",
        "samplingProtocol",
        "locality",
        "coordinateUncertaintyInMeters",
    ]
]

In [ ]:
brandt_metadata[
    [
        "gbifID",
        "occurrenceID",
        "eventRemarks",
        "occurrenceRemarks",
        "georeferenceRemarks",
        "references",
    ]
]

In [ ]:
gbif_id = "4112448514"

response = requests.get(
    f"https://api.gbif.org/v1/occurrence/{gbif_id}",
    timeout=20
)
response.raise_for_status()

brandt_example = response.json()

for field in [
    "datasetKey",
    "datasetTitle",
    "publishingOrgKey",
    "publishingCountry",
    "institutionCode",
    "collectionCode",
    "ownerInstitutionCode",
    "rightsHolder",
    "license",
    "type",
    "occurrenceID",
    "samplingProtocol",
    "recordedBy",
]:
    print(f"{field}: {brandt_example.get(field)}")

In [ ]:
print("\n--- URLs / identifiers ---")

for field in [
    "references",
    "identifier",
    "occurrenceID",
    "datasetID",
    "institutionID",
    "collectionID",
]:
    print(f"{field}: {brandt_example.get(field)}")

In [ ]:
mask_brandt = (
    occ_thinned["qc_biogeographic_status"].eq("pending")
    & occ_thinned["institutionCode"].eq("Brandt Meio Ambiente")
)

print("Selecionados:", mask_brandt.sum())

In [ ]:
occ_thinned.loc[
    mask_brandt,
    "qc_biogeographic_status"
] = "retain_native_source_supported"

occ_thinned.loc[
    mask_brandt,
    "qc_biogeographic_reason"
] = (
    "Traceable record from a structured fauna monitoring program, "
    "with sampling station, event date, sampling protocol, observer, "
    "and institutional provenance; retained despite unreported "
    "coordinate uncertainty."
)

In [ ]:
print(
    occ_thinned["qc_biogeographic_status"]
    .value_counts(dropna=False)
)

print(
    "\nRemaining pending:",
    occ_thinned["qc_biogeographic_status"].eq("pending").sum()
)

In [ ]:
occ_thinned.loc[
    occ_thinned["institutionCode"].eq("Brandt Meio Ambiente"),
    ["gbifID", "qc_biogeographic_status", "qc_biogeographic_reason"]
]

In [ ]:
occ_thinned.to_csv(
    "../data/processed/sapajus_libidinosus_occurrences_biogeographic_qc.csv",
    index=False
)

print("Checkpoint atualizado.")

In [ ]:
ueg_pending = occ_thinned.loc[
    occ_thinned["qc_biogeographic_status"].eq("pending")
].copy()

ueg_pending[
    [
        "gbifID",
        "institutionCode",
        "basisOfRecord",
        "stateProvince",
        "locality",
        "decimalLatitude",
        "decimalLongitude",
        "year",
        "nearest_neighbor_km",
        "qc_spatial_isolation",
        "issue",
    ]
]

In [ ]:
gbif_id = str(ueg_pending.iloc[0]["gbifID"])

response = requests.get(
    f"https://api.gbif.org/v1/occurrence/{gbif_id}",
    timeout=20
)
response.raise_for_status()

ueg_data = response.json()

fields = [
    "key",
    "scientificName",
    "acceptedScientificName",
    "basisOfRecord",
    "institutionCode",
    "collectionCode",
    "catalogNumber",
    "occurrenceID",
    "recordNumber",
    "recordedBy",
    "identifiedBy",
    "eventDate",
    "year",
    "stateProvince",
    "municipality",
    "locality",
    "verbatimLocality",
    "decimalLatitude",
    "decimalLongitude",
    "coordinateUncertaintyInMeters",
    "georeferenceRemarks",
    "georeferenceProtocol",
    "datasetTitle",
    "datasetKey",
    "publishingOrgKey",
    "references",
]

for field in fields:
    print(f"{field}: {ueg_data.get(field)}")

In [ ]:
mask_ueg = (
    occ_thinned["qc_biogeographic_status"].eq("pending")
    & occ_thinned["institutionCode"].eq("UEG")
)

print("Selecionados:", mask_ueg.sum())

occ_thinned.loc[
    mask_ueg,
    "qc_biogeographic_status"
] = "retain_native_verified_collection"

occ_thinned.loc[
    mask_ueg,
    "qc_biogeographic_reason"
] = (
    "UEG record independently verified in the primary publication: "
    "Sapajus libidinosus voucher CMUEG-200 was documented at the "
    "Serra da Fortaleza Wildlife Refuge on 18 November 2022; retained "
    "despite unreported coordinate uncertainty."
)

In [ ]:
print(
    occ_thinned["qc_biogeographic_status"]
    .value_counts(dropna=False)
)

print(
    "\nRemaining pending:",
    occ_thinned["qc_biogeographic_status"].eq("pending").sum()
)

In [ ]:
occ_thinned.to_csv(
    "../data/processed/sapajus_libidinosus_occurrences_biogeographic_qc.csv",
    index=False
)

print("FINAL BIOGEOGRAPHIC QC SAVED.")

In [ ]:
# ============================================================
# FINAL BIOGEOGRAPHIC QC AUDIT
# ============================================================

print("Total records:", len(occ_thinned))
print(
    "Pending:",
    occ_thinned["qc_biogeographic_status"].eq("pending").sum()
)
print(
    "Missing QC status:",
    occ_thinned["qc_biogeographic_status"].isna().sum()
)

print("\nQC status distribution:")
print(
    occ_thinned["qc_biogeographic_status"]
    .value_counts(dropna=False)
)

In [ ]:
import numpy as np

occ_thinned["qc_final_decision"] = np.select(
    [
        occ_thinned["qc_biogeographic_status"].str.startswith(
            "retain_", na=False
        ),
        occ_thinned["qc_biogeographic_status"].str.startswith(
            "exclude_", na=False
        ),
        occ_thinned["qc_biogeographic_status"].str.startswith(
            "review_", na=False
        ),
    ],
    [
        "retain",
        "exclude",
        "review",
    ],
    default="unclassified"
)

print(
    occ_thinned["qc_final_decision"]
    .value_counts(dropna=False)
)

print(
    "\nUnclassified:",
    occ_thinned["qc_final_decision"].eq("unclassified").sum()
)

In [ ]:
taxonomic_review = occ_thinned.loc[
    occ_thinned["qc_final_decision"].eq("review")
].copy()

taxonomic_review[
    [
        "gbifID",
        "scientificName",
        "institutionCode",
        "basisOfRecord",
        "stateProvince",
        "locality",
        "decimalLatitude",
        "decimalLongitude",
        "year",
        "coordinateUncertaintyInMeters",
        "nearest_neighbor_km",
        "qc_biogeographic_reason",
    ]
]

In [ ]:
pd.set_option("display.max_colwidth", None)

taxonomic_review[
    [
        "gbifID",
        "stateProvince",
        "decimalLatitude",
        "decimalLongitude",
        "nearest_neighbor_km",
        "qc_biogeographic_reason",
    ]
].to_string(index=False)

In [ ]:
[c for c in occ_thinned.columns if "tax" in c.lower() or "bio" in c.lower()]

In [ ]:
for _, row in taxonomic_review.iterrows():
    print("=" * 80)
    print("GBIF:", row["gbifID"])
    print("State:", row["stateProvince"])
    print(
        "Coordinates:",
        row["decimalLatitude"],
        row["decimalLongitude"]
    )
    print("Nearest neighbor (km):", row["nearest_neighbor_km"])
    print("Reason:")
    print(row["qc_biogeographic_reason"])
    print()

In [ ]:
mask_review = (
    occ_thinned["qc_biogeographic_status"]
    .eq("review_taxonomic_boundary")
)

print("Registros a resolver:", mask_review.sum())

occ_thinned.loc[
    mask_review,
    "qc_biogeographic_status"
] = "exclude_taxonomic_boundary"

occ_thinned.loc[
    mask_review,
    "qc_biogeographic_reason"
] = (
    "Occurrence located in a taxonomically and biogeographically "
    "sensitive region involving other Sapajus taxa; available evidence "
    "was insufficient to confirm species-level identity as "
    "Sapajus libidinosus for SDM calibration."
)

In [ ]:
mask_sp = (
    occ_thinned["gbifID"].astype(str).eq("6510070188")
)

occ_thinned.loc[
    mask_sp,
    "qc_biogeographic_reason"
] = (
    "Occurrence from a taxonomically sensitive region involving other "
    "Sapajus taxa, without individual-level verification; additionally, "
    "the reported coordinate represents the sampled area rather than "
    "the exact occurrence location. Excluded from SDM calibration."
)

In [ ]:
occ_thinned["qc_final_decision"] = np.select(
    [
        occ_thinned["qc_biogeographic_status"].str.startswith(
            "retain_", na=False
        ),
        occ_thinned["qc_biogeographic_status"].str.startswith(
            "exclude_", na=False
        ),
    ],
    [
        "retain",
        "exclude",
    ],
    default="unclassified"
)

print(
    occ_thinned["qc_final_decision"]
    .value_counts()
)

print(
    "\nUnclassified:",
    occ_thinned["qc_final_decision"]
    .eq("unclassified").sum()
)

In [ ]:
occ_thinned.to_csv(
    "../data/processed/sapajus_libidinosus_occurrences_biogeographic_qc.csv",
    index=False
)

print("Final QC checkpoint saved.")

In [ ]:
occ_final = (
    occ_thinned.loc[
        occ_thinned["qc_final_decision"].eq("retain")
    ]
    .copy()
    .reset_index(drop=True)
)

print("Final SDM occurrences:", len(occ_final))

In [ ]:
assert len(occ_final) == 252
assert occ_final["qc_final_decision"].eq("retain").all()
assert occ_final["decimalLongitude"].notna().all()
assert occ_final["decimalLatitude"].notna().all()

print("✓ Final occurrence dataset validated.")

In [ ]:
occ_final.to_csv(
    "../data/processed/sapajus_libidinosus_occurrences_final.csv",
    index=False
)

print("Final occurrence dataset saved.")

In [ ]:
import geopandas as gpd

occ_final_gdf = gpd.GeoDataFrame(
    occ_final.copy(),
    geometry=gpd.points_from_xy(
        occ_final["decimalLongitude"],
        occ_final["decimalLatitude"]
    ),
    crs="EPSG:4326"
)

print(occ_final_gdf.crs)
print("Occurrences:", len(occ_final_gdf))

In [ ]:
print("Geographic extent")

print(
    "Longitude:",
    occ_final_gdf["decimalLongitude"].min(),
    "to",
    occ_final_gdf["decimalLongitude"].max()
)

print(
    "Latitude:",
    occ_final_gdf["decimalLatitude"].min(),
    "to",
    occ_final_gdf["decimalLatitude"].max()
)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 8))

occ_final_gdf.plot(
    ax=ax,
    markersize=18,
    alpha=0.7
)

ax.set_title(
    "Final occurrences of Sapajus libidinosus",
    fontsize=13
)

ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")

plt.show()

In [ ]:
from pathlib import Path

spatial_dir = Path("../data")

spatial_files = list(spatial_dir.rglob("*.shp"))
spatial_files += list(spatial_dir.rglob("*.gpkg"))
spatial_files += list(spatial_dir.rglob("*.geojson"))

print("Spatial files found:", len(spatial_files))

for f in spatial_files:
    print(f)

In [ ]:
from pathlib import Path

raw_spatial = Path("../data/raw/spatial")
raw_spatial.mkdir(parents=True, exist_ok=True)

print(raw_spatial.resolve())

In [ ]:
from pathlib import Path
import zipfile

zip_path = Path("../data/raw/spatial/Ecoregions2017.zip")
extract_dir = Path("../data/raw/spatial/Ecoregions2017")

print("ZIP exists:", zip_path.exists())

if not extract_dir.exists():
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)

    print("Ecoregions extracted.")
else:
    print("Extraction directory already exists.")

In [ ]:
shapefiles = list(extract_dir.rglob("*.shp"))

print("Shapefiles found:", len(shapefiles))

for shp in shapefiles:
    print(shp)

In [ ]:
ecoregions = gpd.read_file(shapefiles[0])

print("Ecoregions:", len(ecoregions))
print("CRS:", ecoregions.crs)
print("\nColumns:")
print(ecoregions.columns.tolist())

In [ ]:
ecoregions.head()

In [ ]:
print("Ecoregions CRS:", ecoregions.crs)
print("Occurrences CRS:", occ_final_gdf.crs)

if ecoregions.crs != occ_final_gdf.crs:
    occ_ecoreg_crs = occ_final_gdf.to_crs(ecoregions.crs)
else:
    occ_ecoreg_crs = occ_final_gdf.copy()

In [ ]:
occ_ecoreg = gpd.sjoin(
    occ_ecoreg_crs,
    ecoregions[
        [
            "ECO_ID",
            "ECO_NAME",
            "BIOME_NUM",
            "BIOME_NAME",
            "REALM",
            "geometry"
        ]
    ],
    how="left",
    predicate="within"
)

print("Occurrences:", len(occ_ecoreg))
print("Without ecoregion:", occ_ecoreg["ECO_NAME"].isna().sum())
print("Ecoregions occupied:", occ_ecoreg["ECO_NAME"].nunique())

In [ ]:
ecoregion_summary = (
    occ_ecoreg
    .groupby(
        ["ECO_ID", "ECO_NAME", "BIOME_NAME", "REALM"],
        dropna=False
    )
    .size()
    .reset_index(name="n_occurrences")
    .sort_values("n_occurrences", ascending=False)
)

ecoregion_summary["percent"] = (
    ecoregion_summary["n_occurrences"]
    / len(occ_ecoreg)
    * 100
).round(2)

ecoregion_summary

In [ ]:
biome_summary = (
    occ_ecoreg["BIOME_NAME"]
    .value_counts(dropna=False)
    .rename_axis("BIOME_NAME")
    .reset_index(name="n_occurrences")
)

biome_summary["percent"] = (
    biome_summary["n_occurrences"]
    / len(occ_ecoreg)
    * 100
).round(2)

biome_summary

In [ ]:
import matplotlib.pyplot as plt

# Ecoregions containing at least one final occurrence
occupied_ids = occ_ecoreg["ECO_ID"].dropna().unique()

occupied_ecoregions = ecoregions[
    ecoregions["ECO_ID"].isin(occupied_ids)
].copy()

fig, ax = plt.subplots(figsize=(11, 11))

occupied_ecoregions.plot(
    ax=ax,
    column="ECO_NAME",
    categorical=True,
    alpha=0.45,
    edgecolor="black",
    linewidth=0.6,
    legend=True
)

occ_ecoreg_crs.plot(
    ax=ax,
    markersize=12,
    edgecolor="black",
    linewidth=0.2
)

ax.set_title(
    "Final Sapajus libidinosus occurrences and occupied ecoregions",
    fontsize=14
)

ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
bounds = occ_final_gdf.total_bounds

print(f"West:  {bounds[0]:.3f}")
print(f"South: {bounds[1]:.3f}")
print(f"East:  {bounds[2]:.3f}")
print(f"North:  {bounds[3]:.3f}")

In [ ]:
ecoregion_summary[
    ["ECO_NAME", "n_occurrences", "percent"]
].reset_index(drop=True)

In [ ]:
# Project occupied ecoregions to an equal/metric CRS suitable for Brazil
occupied_proj = occupied_ecoregions.to_crs("EPSG:5880").copy()

occupied_proj["area_km2"] = (
    occupied_proj.geometry.area / 1_000_000
)

ecoregion_area = (
    occupied_proj[
        ["ECO_ID", "ECO_NAME", "BIOME_NAME", "area_km2"]
    ]
    .merge(
        ecoregion_summary[
            ["ECO_ID", "n_occurrences", "percent"]
        ],
        on="ECO_ID",
        how="left"
    )
    .sort_values("area_km2", ascending=False)
)

ecoregion_area["occ_per_10000_km2"] = (
    ecoregion_area["n_occurrences"]
    / ecoregion_area["area_km2"]
    * 10_000
)

ecoregion_area[
    [
        "ECO_NAME",
        "n_occurrences",
        "percent",
        "area_km2",
        "occ_per_10000_km2"
    ]
].round(2)

In [ ]:
rare_ecoregions = ecoregion_summary.loc[
    ecoregion_summary["n_occurrences"] <= 5,
    "ECO_NAME"
]

rare_occ = (
    occ_ecoreg.loc[
        occ_ecoreg["ECO_NAME"].isin(rare_ecoregions),
        [
            "gbifID",
            "decimalLongitude",
            "decimalLatitude",
            "ECO_NAME",
            "BIOME_NAME"
        ]
    ]
    .sort_values(["ECO_NAME", "gbifID"])
)

rare_occ

In [ ]:
print("Rare-ecoregion occurrences:", len(rare_occ))

In [ ]:
import numpy as np
from sklearn.neighbors import BallTree

coords_rad = np.radians(
    occ_final_gdf[
        ["decimalLatitude", "decimalLongitude"]
    ].to_numpy()
)

tree = BallTree(coords_rad, metric="haversine")

dist, ind = tree.query(coords_rad, k=2)

# k=2 because the first neighbour is the point itself
nn_km = dist[:, 1] * 6371.0088

occ_final_gdf["nearest_neighbor_km_final"] = nn_km

print("Nearest-neighbour distance (km)")
print("--------------------------------")
print(f"Minimum : {np.min(nn_km):.2f}")
print(f"Median  : {np.median(nn_km):.2f}")
print(f"Mean    : {np.mean(nn_km):.2f}")
print(f"P75     : {np.percentile(nn_km, 75):.2f}")
print(f"P90     : {np.percentile(nn_km, 90):.2f}")
print(f"P95     : {np.percentile(nn_km, 95):.2f}")
print(f"P99     : {np.percentile(nn_km, 99):.2f}")
print(f"Maximum : {np.max(nn_km):.2f}")

In [ ]:
nn_diagnostic = (
    occ_final_gdf[
        [
            "gbifID",
            "decimalLongitude",
            "decimalLatitude",
            "nearest_neighbor_km_final"
        ]
    ]
    .sort_values(
        "nearest_neighbor_km_final",
        ascending=False
    )
    .head(20)
)

nn_diagnostic

In [ ]:
# Project final occurrences to metric CRS
occ_proj = occ_final_gdf.to_crs("EPSG:5880").copy()

buffers = {}

for km in [50, 75, 100]:
    geom = occ_proj.geometry.buffer(km * 1000).union_all()

    buffers[km] = gpd.GeoDataFrame(
        {"buffer_km": [km]},
        geometry=[geom],
        crs="EPSG:5880"
    )

    area = geom.area / 1_000_000

    print(f"{km:>3} km buffer: {area:,.0f} km²")

In [ ]:
def count_components(geom):
    if geom.geom_type == "Polygon":
        return 1
    elif geom.geom_type == "MultiPolygon":
        return len(geom.geoms)
    return np.nan


for km, gdf in buffers.items():
    geom = gdf.geometry.iloc[0]

    print(
        f"{km:>3} km | "
        f"area = {geom.area / 1_000_000:,.0f} km² | "
        f"components = {count_components(geom)}"
    )

In [ ]:
for km, buffer_gdf in buffers.items():

    fig, ax = plt.subplots(figsize=(9, 9))

    buffer_gdf.plot(
        ax=ax,
        alpha=0.35,
        edgecolor="black"
    )

    occ_proj.plot(
        ax=ax,
        markersize=8
    )

    ax.set_title(
        f"Sapajus libidinosus — {km} km occurrence buffer"
    )

    ax.set_axis_off()
    plt.tight_layout()
    plt.show()

In [ ]:
rows = []

for km, buffer_gdf in buffers.items():

    buffer_geom = buffer_gdf.geometry.iloc[0]

    for _, eco in occupied_proj.iterrows():

        intersection = eco.geometry.intersection(buffer_geom)

        eco_area = eco.geometry.area / 1_000_000
        overlap_area = intersection.area / 1_000_000

        rows.append({
            "buffer_km": km,
            "ECO_ID": eco["ECO_ID"],
            "ECO_NAME": eco["ECO_NAME"],
            "ecoregion_area_km2": eco_area,
            "overlap_area_km2": overlap_area,
            "percent_ecoregion_covered": (
                overlap_area / eco_area * 100
            )
        })

buffer_ecoreg_overlap = pd.DataFrame(rows)

buffer_ecoreg_overlap = (
    buffer_ecoreg_overlap
    .merge(
        ecoregion_summary[
            ["ECO_ID", "n_occurrences"]
        ],
        on="ECO_ID",
        how="left"
    )
)

buffer_ecoreg_overlap[
    [
        "buffer_km",
        "ECO_NAME",
        "n_occurrences",
        "overlap_area_km2",
        "percent_ecoregion_covered"
    ]
].round(2)

In [ ]:
overlap_pivot = (
    buffer_ecoreg_overlap
    .pivot(
        index=["ECO_NAME", "n_occurrences"],
        columns="buffer_km",
        values="percent_ecoregion_covered"
    )
    .rename(columns={
        50: "covered_50km_pct",
        75: "covered_75km_pct",
        100: "covered_100km_pct"
    })
    .reset_index()
    .sort_values(
        "n_occurrences",
        ascending=False
    )
)

overlap_pivot.round(2)

In [ ]:
from sklearn.metrics.pairwise import haversine_distances
from scipy.sparse.csgraph import connected_components
from scipy.sparse import csr_matrix

coords_rad = np.radians(
    occ_final_gdf[
        ["decimalLatitude", "decimalLongitude"]
    ].to_numpy()
)

dist_matrix_km = haversine_distances(coords_rad) * 6371.0088

thresholds = [50, 75, 100, 125, 150, 175, 200]

network_results = []

for threshold in thresholds:

    adjacency = (
        (dist_matrix_km <= threshold) &
        (dist_matrix_km > 0)
    )

    graph = csr_matrix(adjacency)

    n_components, labels = connected_components(
        graph,
        directed=False
    )

    component_sizes = np.bincount(labels)

    largest_component = component_sizes.max()
    isolated_points = np.sum(component_sizes == 1)

    network_results.append({
        "threshold_km": threshold,
        "n_components": n_components,
        "largest_component_n": largest_component,
        "largest_component_pct":
            largest_component / len(occ_final_gdf) * 100,
        "isolated_occurrences": isolated_points
    })

network_summary = pd.DataFrame(network_results)

network_summary.round(2)

In [ ]:
# Work in projected CRS
all_eco_proj = ecoregions.to_crs("EPSG:5880").copy()

occupied_ids = set(
    occ_ecoreg["ECO_ID"].dropna().astype(int)
)

occupied_union = (
    all_eco_proj[
        all_eco_proj["ECO_ID"].isin(occupied_ids)
    ]
    .geometry
    .union_all()
)

# Candidate ecoregions not currently occupied
unoccupied = all_eco_proj[
    ~all_eco_proj["ECO_ID"].isin(occupied_ids)
].copy()

# Select ecoregions touching/intersecting occupied ecoregions
adjacent = unoccupied[
    unoccupied.geometry.intersects(occupied_union)
].copy()

print("Occupied ecoregions:", len(occupied_ids))
print("Adjacent unoccupied ecoregions:", len(adjacent))

adjacent[
    ["ECO_ID", "ECO_NAME", "BIOME_NAME", "REALM"]
].sort_values("ECO_NAME")

In [ ]:
occ_union = occ_proj.geometry.union_all()

adjacent["distance_to_occ_km"] = (
    adjacent.geometry.distance(occ_union) / 1000
)

adjacent[
    [
        "ECO_NAME",
        "BIOME_NAME",
        "distance_to_occ_km"
    ]
].sort_values("distance_to_occ_km").round(2)

In [ ]:
# Regional search domain: 500 km around all validated occurrences
regional_domain = (
    occ_proj.geometry
    .union_all()
    .convex_hull
    .buffer(500_000)
)

# Keep only ecoregion geometries that intersect this regional domain
regional_eco = all_eco_proj[
    all_eco_proj.geometry.intersects(regional_domain)
].copy()

print("Global ecoregions:", len(all_eco_proj))
print("Regional ecoregions:", len(regional_eco))

In [ ]:
regional_eco = regional_eco[
    regional_eco["REALM"].eq("Neotropic")
].copy()

print("Neotropical regional ecoregions:", len(regional_eco))

In [ ]:
occupied_regional = regional_eco[
    regional_eco["ECO_ID"].isin(occupied_ids)
].copy()

occupied_union = occupied_regional.geometry.union_all()

unoccupied_regional = regional_eco[
    ~regional_eco["ECO_ID"].isin(occupied_ids)
].copy()

adjacent_corrected = unoccupied_regional[
    unoccupied_regional.geometry.intersects(occupied_union)
].copy()

occ_union = occ_proj.geometry.union_all()

adjacent_corrected["distance_to_occ_km"] = (
    adjacent_corrected.geometry.distance(occ_union) / 1000
)

adjacent_corrected[
    [
        "ECO_ID",
        "ECO_NAME",
        "BIOME_NAME",
        "distance_to_occ_km"
    ]
].sort_values("distance_to_occ_km").round(2)

In [ ]:
rows = []

for _, candidate in adjacent_corrected.iterrows():

    shared = candidate.geometry.boundary.intersection(
        occupied_union.boundary
    )

    shared_km = shared.length / 1000

    rows.append({
        "ECO_ID": candidate["ECO_ID"],
        "ECO_NAME": candidate["ECO_NAME"],
        "BIOME_NAME": candidate["BIOME_NAME"],
        "distance_to_occ_km": candidate["distance_to_occ_km"],
        "shared_boundary_km": shared_km
    })

adjacency_diagnostic = (
    pd.DataFrame(rows)
    .sort_values(
        ["distance_to_occ_km", "shared_boundary_km"],
        ascending=[True, False]
    )
)

adjacency_diagnostic.round(2)

In [ ]:
# Adjacent ecoregions sufficiently close to validated occurrences
adjacent_near = adjacent_corrected[
    adjacent_corrected["distance_to_occ_km"] <= 100
].copy()

print("Nearby adjacent ecoregions:", len(adjacent_near))

adjacent_near[
    ["ECO_NAME", "BIOME_NAME", "distance_to_occ_km"]
].sort_values("distance_to_occ_km").round(2)

In [ ]:
biogeographic_domain = pd.concat(
    [
        occupied_regional,
        adjacent_near
    ],
    ignore_index=True
)

# Avoid any duplicated ECO_ID
biogeographic_domain = (
    biogeographic_domain
    .drop_duplicates(subset="ECO_ID")
    .copy()
)

bio_union = biogeographic_domain.geometry.union_all()

print(
    "Ecoregions in candidate biogeographic domain:",
    biogeographic_domain["ECO_ID"].nunique()
)

In [ ]:
M_candidates = {}

for km in [50, 75, 100]:

    buffer_geom = buffers[km].geometry.iloc[0]

    M_geom = buffer_geom.intersection(bio_union)

    M_candidates[km] = gpd.GeoDataFrame(
        {"candidate": [f"M{km}"]},
        geometry=[M_geom],
        crs="EPSG:5880"
    )

In [ ]:
candidate_results = []

for km, gdf in M_candidates.items():

    geom = gdf.geometry.iloc[0]

    area_km2 = geom.area / 1_000_000

    if geom.geom_type == "Polygon":
        components = 1
    elif geom.geom_type == "MultiPolygon":
        components = len(geom.geoms)
    else:
        components = np.nan

    # All final occurrences should be covered
    covered = occ_proj.geometry.intersects(geom).sum()

    candidate_results.append({
        "candidate": f"M{km}",
        "buffer_km": km,
        "area_km2": area_km2,
        "components": components,
        "occurrences_covered": covered,
        "occurrences_pct":
            covered / len(occ_proj) * 100
    })

candidate_summary = pd.DataFrame(candidate_results)

candidate_summary.round(2)

In [ ]:
# Primary calibration area
M_final = M_candidates[75].copy()

M_final["candidate"] = "M_primary"
M_final["buffer_km"] = 75
M_final["area_km2"] = (
    M_final.geometry.area / 1_000_000
)

# Save projected version
M_final.to_file(
    "../data/processed/sapajus_calibration_area_M.gpkg",
    layer="M_primary",
    driver="GPKG"
)

# Also save geographic version
M_final_wgs84 = M_final.to_crs("EPSG:4326")

M_final_wgs84.to_file(
    "../data/processed/sapajus_calibration_area_M_wgs84.gpkg",
    layer="M_primary",
    driver="GPKG"
)

print("Primary calibration area saved.")
print(f"Area: {M_final['area_km2'].iloc[0]:,.0f} km²")
print(f"Occurrences covered: {occ_proj.geometry.intersects(M_final.geometry.iloc[0]).sum()}/252")

In [ ]:
for km, label in [
    (50, "M_sensitivity_low"),
    (100, "M_sensitivity_high")
]:

    out = M_candidates[km].copy()
    out["candidate"] = label
    out["buffer_km"] = km
    out["area_km2"] = out.geometry.area / 1_000_000

    out.to_file(
        f"../data/processed/sapajus_calibration_area_{label}.gpkg",
        layer=label,
        driver="GPKG"
    )

print("Sensitivity calibration areas saved.")

In [ ]:
import matplotlib.pyplot as plt

# Convert candidates to the same CRS used for plotting
M50_plot = M_candidates[50]
M75_plot = M_candidates[75]
M100_plot = M_candidates[100]

fig, axes = plt.subplots(
    1, 3,
    figsize=(18, 7),
    sharex=True,
    sharey=True
)

candidates_plot = [
    (M50_plot, "M50", "732,290 km²"),
    (M75_plot, "M75 — Primary", "1,248,926 km²"),
    (M100_plot, "M100", "1,738,966 km²")
]

for ax, (M, title, area) in zip(axes, candidates_plot):

    # Calibration area
    M.plot(
        ax=ax,
        facecolor="lightgray",
        edgecolor="black",
        linewidth=0.7
    )

    # Final validated occurrences
    occ_proj.plot(
        ax=ax,
        markersize=8,
        color="black",
        alpha=0.75
    )

    ax.set_title(
        f"{title}\n{area}",
        fontsize=12
    )

    ax.set_axis_off()

fig.suptitle(
    "Candidate calibration areas (M) for Sapajus libidinosus",
    fontsize=15,
    y=0.96
)

plt.tight_layout()

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

M_final.plot(
    ax=ax,
    facecolor="lightgray",
    edgecolor="black",
    linewidth=0.8,
    alpha=0.7
)

occ_proj.plot(
    ax=ax,
    color="black",
    markersize=12,
    alpha=0.8,
    label="Validated occurrences"
)

ax.set_title(
    r"Calibration area ($M$) for $\it{Sapajus\ libidinosus}$",
    fontsize=15,
    pad=15
)

ax.legend(
    frameon=False,
    loc="lower left"
)

ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
# Plotting layers in the same CRS
bio_plot = biogeographic_domain.to_crs(M_final.crs)
occ_plot = occ_final_gdf.to_crs(M_final.crs)

fig, ax = plt.subplots(figsize=(10, 10))

# Biogeographic domain
bio_plot.plot(
    ax=ax,
    facecolor="whitesmoke",
    edgecolor="0.70",
    linewidth=0.5,
    alpha=0.7,
    zorder=1
)

# Primary calibration area
M_final.plot(
    ax=ax,
    facecolor="lightgray",
    edgecolor="0.20",
    linewidth=0.8,
    alpha=0.85,
    zorder=2
)

# Validated occurrences
occ_plot.plot(
    ax=ax,
    color="black",
    markersize=12,
    alpha=0.85,
    label="Validated occurrences",
    zorder=3
)

ax.set_title(
    r"Calibration area ($M$) for $\it{Sapajus\ libidinosus}$",
    fontsize=15,
    pad=15
)

ax.legend(
    frameon=False,
    loc="lower left"
)

ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path
import rasterio
import pandas as pd

raster_dir = Path("../data/raw/environmental")

raster_files = sorted(
    list(raster_dir.glob("*.tif")) +
    list(raster_dir.glob("*.tiff"))
)

print("Raster files found:", len(raster_files))

raster_info = []

for path in raster_files:

    with rasterio.open(path) as src:

        raster_info.append({
            "file": path.name,
            "crs": str(src.crs),
            "width": src.width,
            "height": src.height,
            "resolution_x": abs(src.res[0]),
            "resolution_y": abs(src.res[1]),
            "nodata": src.nodata,
            "xmin": src.bounds.left,
            "ymin": src.bounds.bottom,
            "xmax": src.bounds.right,
            "ymax": src.bounds.top
        })

raster_info = pd.DataFrame(raster_info)

raster_info

In [ ]:
from pathlib import Path

data_dir = Path("../data")

for path in sorted(data_dir.rglob("*")):
    if path.is_file():
        print(path)

In [ ]:
import pandas as pd

audit_env = pd.read_csv(
    "../data/processed/audit_02_environment.csv"
)

print("Shape:", audit_env.shape)
print("\nColumns:")
print(audit_env.columns.tolist())

audit_env

In [ ]:
from pathlib import Path
import rasterio
import pandas as pd

raster_dir = Path("../data/processed/environment")

raster_files = sorted(raster_dir.glob("*.tif"))

print("Raster files found:", len(raster_files))

raster_info = []

for path in raster_files:
    with rasterio.open(path) as src:
        raster_info.append({
            "file": path.name,
            "crs": str(src.crs),
            "width": src.width,
            "height": src.height,
            "resolution_x": abs(src.res[0]),
            "resolution_y": abs(src.res[1]),
            "nodata": src.nodata,
            "xmin": src.bounds.left,
            "ymin": src.bounds.bottom,
            "xmax": src.bounds.right,
            "ymax": src.bounds.top
        })

raster_info = pd.DataFrame(raster_info)

raster_info

In [ ]:
import geopandas as gpd
import rasterio
from rasterio.mask import mask
from pathlib import Path

M_wgs84 = gpd.read_file(
    "../data/processed/sapajus_calibration_area_M_wgs84.gpkg"
)

raster_dir = Path("../data/processed/environment")
output_dir = Path("../data/processed/environment_M75")
output_dir.mkdir(parents=True, exist_ok=True)

raster_files = sorted(raster_dir.glob("*.tif"))

for raster_path in raster_files:

    with rasterio.open(raster_path) as src:

        # Ensure M has exactly the raster CRS
        M_raster_crs = M_wgs84.to_crs(src.crs)

        shapes = [
            geom.__geo_interface__
            for geom in M_raster_crs.geometry
        ]

        out_image, out_transform = mask(
            src,
            shapes,
            crop=True,
            nodata=src.nodata
        )

        out_meta = src.meta.copy()

        out_meta.update({
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform,
            "nodata": src.nodata
        })

        output_path = output_dir / raster_path.name

        with rasterio.open(
            output_path,
            "w",
            **out_meta
        ) as dst:
            dst.write(out_image)

        print(f"Saved: {output_path.name}")

In [ ]:
import numpy as np
import pandas as pd
import rasterio

cropped_files = sorted(output_dir.glob("*.tif"))

m75_raster_info = []

for path in cropped_files:

    with rasterio.open(path) as src:

        arr = src.read(1)

        valid = (
            np.isfinite(arr) &
            (arr != src.nodata)
        )

        m75_raster_info.append({
            "file": path.name,
            "crs": str(src.crs),
            "width": src.width,
            "height": src.height,
            "resolution_x": abs(src.res[0]),
            "resolution_y": abs(src.res[1]),
            "valid_cells": int(valid.sum()),
            "nodata_cells": int((~valid).sum()),
            "min": float(arr[valid].min()),
            "max": float(arr[valid].max()),
            "mean": float(arr[valid].mean())
        })

m75_raster_info = pd.DataFrame(m75_raster_info)

m75_raster_info.round(3)

In [ ]:
reference = cropped_files[0]

with rasterio.open(reference) as ref:
    ref_crs = ref.crs
    ref_transform = ref.transform
    ref_shape = ref.shape

for path in cropped_files[1:]:

    with rasterio.open(path) as src:

        assert src.crs == ref_crs, f"CRS mismatch: {path.name}"
        assert src.transform == ref_transform, f"Transform mismatch: {path.name}"
        assert src.shape == ref_shape, f"Shape mismatch: {path.name}"

print("✓ All M75 environmental rasters are perfectly aligned.")

In [ ]:
import numpy as np
import pandas as pd
import rasterio

# Occurrences must be in the raster CRS
occ_env = occ_final_gdf.to_crs("EPSG:4326").copy()

coords = list(
    zip(
        occ_env.geometry.x,
        occ_env.geometry.y
    )
)

env_columns = []

for path in cropped_files:

    # BIO variable name
    bio_name = (
        path.stem
        .replace("wc2.1_30s_", "")
        .upper()
    )

    with rasterio.open(path) as src:

        values = np.array(
            [v[0] for v in src.sample(coords)],
            dtype=float
        )

        # Convert raster NoData to NaN
        if src.nodata is not None:
            values[np.isclose(values, src.nodata)] = np.nan

        occ_env[bio_name] = values
        env_columns.append(bio_name)

print("Environmental variables:", env_columns)

occ_env[
    ["gbifID"] + env_columns
].head()

In [ ]:
presence_env_audit = pd.DataFrame({
    "variable": env_columns,
    "n_valid": [
        occ_env[col].notna().sum()
        for col in env_columns
    ],
    "n_missing": [
        occ_env[col].isna().sum()
        for col in env_columns
    ]
})

presence_env_audit["valid_pct"] = (
    presence_env_audit["n_valid"] /
    len(occ_env) * 100
)

presence_env_audit

In [ ]:
assert occ_env[env_columns].notna().all().all(), (
    "At least one final occurrence has missing environmental data."
)

print(
    f"✓ All {len(occ_env)} final occurrences have "
    f"complete environmental data for {len(env_columns)} predictors."
)

In [ ]:
import matplotlib.pyplot as plt
import rasterio
import numpy as np

# Friendly names
bio_labels = {
    "BIO1": "Annual Mean Temperature",
    "BIO4": "Temperature Seasonality",
    "BIO12": "Annual Precipitation",
    "BIO15": "Precipitation Seasonality"
}

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 12)
)

axes = axes.flatten()

for ax, path in zip(axes, cropped_files):

    bio_name = (
        path.stem
        .replace("wc2.1_30s_", "")
        .upper()
    )

    with rasterio.open(path) as src:

        arr = src.read(1).astype(float)

        # NoData -> NaN
        if src.nodata is not None:
            arr[np.isclose(arr, src.nodata)] = np.nan

        extent = [
            src.bounds.left,
            src.bounds.right,
            src.bounds.bottom,
            src.bounds.top
        ]

        im = ax.imshow(
            arr,
            extent=extent,
            origin="upper"
        )

    # Final occurrences
    occ_env.plot(
        ax=ax,
        color="black",
        markersize=4,
        alpha=0.65
    )

    ax.set_title(
        f"{bio_name} — {bio_labels.get(bio_name, bio_name)}",
        fontsize=11
    )

    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")

    plt.colorbar(
        im,
        ax=ax,
        shrink=0.75
    )

fig.suptitle(
    r"Environmental predictors within calibration area ($M$)",
    fontsize=15,
    y=0.98
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import rasterio

arrays = {}
valid_mask = None

for path in cropped_files:

    bio_name = (
        path.stem
        .replace("wc2.1_30s_", "")
        .upper()
        .replace("_", "")
    )

    with rasterio.open(path) as src:

        arr = src.read(1).astype(float)

        valid = np.isfinite(arr)

        if src.nodata is not None:
            valid &= ~np.isclose(arr, src.nodata)

        arrays[bio_name] = arr

        if valid_mask is None:
            valid_mask = valid.copy()
        else:
            valid_mask &= valid

print("Joint valid cells:", valid_mask.sum())

In [ ]:
rng = np.random.default_rng(42)

valid_indices = np.flatnonzero(valid_mask)

n_sample = min(
    100_000,
    len(valid_indices)
)

sample_idx = rng.choice(
    valid_indices,
    size=n_sample,
    replace=False
)

env_sample = pd.DataFrame({
    name: arr.ravel()[sample_idx]
    for name, arr in arrays.items()
})

print("Environmental sample:", len(env_sample))

env_sample.describe().round(2)

In [ ]:
pearson_corr = env_sample.corr(
    method="pearson"
)

spearman_corr = env_sample.corr(
    method="spearman"
)

print("Pearson correlation")
display(pearson_corr.round(3))

print("\nSpearman correlation")
display(spearman_corr.round(3))

In [ ]:
corr = pearson_corr.abs()

upper = corr.where(
    np.triu(
        np.ones(corr.shape),
        k=1
    ).astype(bool)
)

high_corr_pairs = (
    upper.stack()
    .reset_index()
)

high_corr_pairs.columns = [
    "variable_1",
    "variable_2",
    "abs_r"
]

high_corr_pairs = (
    high_corr_pairs
    .sort_values("abs_r", ascending=False)
    .reset_index(drop=True)
)

high_corr_pairs

In [ ]:
import sys
!{sys.executable} -m pip install statsmodels

In [ ]:
import statsmodels

print(statsmodels.__version__)

from statsmodels.stats.outliers_influence import variance_inflation_factor

print("✓ statsmodels loaded successfully.")

In [ ]:
X = env_sample[
    ["BIO1", "BIO4", "BIO12", "BIO15"]
].copy()

vif_results = pd.DataFrame({
    "variable": X.columns,
    "VIF": [
        variance_inflation_factor(
            X.values,
            i
        )
        for i in range(X.shape[1])
    ]
})

vif_results.sort_values(
    "VIF",
    ascending=False
).round(3)

In [ ]:
from sklearn.preprocessing import StandardScaler

X_scaled = StandardScaler().fit_transform(X)

vif_scaled = pd.DataFrame({
    "variable": X.columns,
    "VIF": [
        variance_inflation_factor(X_scaled, i)
        for i in range(X_scaled.shape[1])
    ]
})

vif_scaled.sort_values(
    "VIF",
    ascending=False
).round(3)

In [ ]:
env_qc_summary = pd.DataFrame({
    "variable": ["BIO1", "BIO4", "BIO12", "BIO15"],
    "VIF": [2.443, 2.599, 1.605, 1.544],
    "decision": ["retain"] * 4
})

env_qc_summary.to_csv(
    "../data/processed/environment_M75/environmental_predictor_qc.csv",
    index=False
)

print("✓ Environmental predictor QC saved.")

In [ ]:
from sklearn.neighbors import KernelDensity
import numpy as np
import matplotlib.pyplot as plt

# Coordinates in meters
xy = np.column_stack([
    occ_proj.geometry.x,
    occ_proj.geometry.y
])

# 50-km bandwidth as a diagnostic scale
bandwidth = 50_000

kde = KernelDensity(
    bandwidth=bandwidth,
    kernel="gaussian"
)

kde.fit(xy)

print("✓ KDE fitted to", len(xy), "occurrences.")

In [ ]:
# M75 bounds
xmin, ymin, xmax, ymax = M_final.total_bounds

grid_size = 300

xx, yy = np.meshgrid(
    np.linspace(xmin, xmax, grid_size),
    np.linspace(ymin, ymax, grid_size)
)

grid_xy = np.column_stack([
    xx.ravel(),
    yy.ravel()
])

log_density = kde.score_samples(grid_xy)

density = np.exp(log_density).reshape(xx.shape)

In [ ]:
import geopandas as gpd

grid_points = gpd.GeoDataFrame(
    geometry=gpd.points_from_xy(
        grid_xy[:, 0],
        grid_xy[:, 1]
    ),
    crs=M_final.crs
)

inside_M = grid_points.geometry.within(
    M_final.geometry.iloc[0]
).to_numpy()

density_masked = density.ravel()
density_masked[~inside_M] = np.nan
density_masked = density_masked.reshape(xx.shape)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 10))

M_final.boundary.plot(
    ax=ax,
    linewidth=0.7
)

im = ax.imshow(
    density_masked,
    extent=[xmin, xmax, ymin, ymax],
    origin="lower",
    alpha=0.8
)

occ_proj.plot(
    ax=ax,
    color="black",
    markersize=8,
    alpha=0.7,
    label="Validated occurrences"
)

plt.colorbar(
    im,
    ax=ax,
    shrink=0.7,
    label="Occurrence density (KDE)"
)

ax.set_title(
    r"Spatial sampling density within calibration area ($M$)",
    fontsize=14
)

ax.legend(
    frameon=False,
    loc="lower left"
)

ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
from scipy.ndimage import distance_transform_edt
import rasterio
import numpy as np
import pandas as pd

# Reference raster
ref_path = cropped_files[0]

with rasterio.open(ref_path) as src:
    transform = src.transform
    raster_crs = src.crs
    height, width = src.shape

# Presence cells
presence_mask = np.zeros(
    (height, width),
    dtype=bool
)

for geom in occ_env.geometry:

    row, col = rasterio.transform.rowcol(
        transform,
        geom.x,
        geom.y
    )

    if (
        0 <= row < height and
        0 <= col < width
    ):
        presence_mask[row, col] = True

print("Presence cells:", presence_mask.sum())

In [ ]:
from sklearn.neighbors import BallTree

rows, cols = np.where(valid_mask)

xs, ys = rasterio.transform.xy(
    transform,
    rows,
    cols,
    offset="center"
)

cell_coords_rad = np.radians(
    np.column_stack([
        ys,  # latitude
        xs   # longitude
    ])
)

presence_coords_rad = np.radians(
    np.column_stack([
        occ_env.geometry.y,
        occ_env.geometry.x
    ])
)

tree = BallTree(
    presence_coords_rad,
    metric="haversine"
)

dist_rad, _ = tree.query(
    cell_coords_rad,
    k=1
)

distance_to_presence_km = (
    dist_rad[:, 0] * 6371.0088
)

In [ ]:
distance_summary = []

for threshold in [0, 5, 10, 20, 50]:

    eligible = (
        distance_to_presence_km >= threshold
    )

    distance_summary.append({
        "min_distance_km": threshold,
        "eligible_cells": int(eligible.sum()),
        "eligible_pct": (
            eligible.mean() * 100
        )
    })

distance_summary = pd.DataFrame(
    distance_summary
)

distance_summary.round(2)

In [ ]:
eligible_10km = (
    distance_to_presence_km >= 10
)

eligible_rows = rows[eligible_10km]
eligible_cols = cols[eligible_10km]

eligible_lon = np.asarray(xs)[eligible_10km]
eligible_lat = np.asarray(ys)[eligible_10km]

eligible_distance = (
    distance_to_presence_km[eligible_10km]
)

print(
    "Eligible background cells:",
    len(eligible_rows)
)

In [ ]:
n_presences = len(occ_env)

pseudoabsence_ratio = 5
n_pseudoabsence = (
    n_presences * pseudoabsence_ratio
)

seeds = [101, 202, 303, 404, 505]

pseudoabsence_replicates = {}

for rep, seed in enumerate(seeds, start=1):

    rng = np.random.default_rng(seed)

    selected = rng.choice(
        len(eligible_rows),
        size=n_pseudoabsence,
        replace=False
    )

    pa = pd.DataFrame({
        "replicate": rep,
        "row": eligible_rows[selected],
        "col": eligible_cols[selected],
        "longitude": eligible_lon[selected],
        "latitude": eligible_lat[selected],
        "distance_to_presence_km":
            eligible_distance[selected],
        "class": 0
    })

    pseudoabsence_replicates[rep] = pa

    print(
        f"Replicate {rep}: "
        f"{len(pa)} pseudoabsences"
    )

In [ ]:
for rep, pa in pseudoabsence_replicates.items():

    rr = pa["row"].to_numpy()
    cc = pa["col"].to_numpy()

    for variable, arr in arrays.items():
        pa[variable] = arr[rr, cc]

    pseudoabsence_replicates[rep] = pa

In [ ]:
for rep, pa in pseudoabsence_replicates.items():

    assert len(pa) == 1260

    assert (
        pa[["BIO1", "BIO4", "BIO12", "BIO15"]]
        .notna()
        .all()
        .all()
    )

    assert (
        pa["distance_to_presence_km"] >= 10
    ).all()

    assert (
        ~pa.duplicated(["row", "col"])
    ).all()

print(
    "✓ All pseudoabsence replicates validated."
)

In [ ]:
pa_dir = Path(
    "../data/processed/pseudoabsences"
)

pa_dir.mkdir(
    parents=True,
    exist_ok=True
)

for rep, pa in pseudoabsence_replicates.items():

    path = (
        pa_dir /
        f"sapajus_pseudoabsence_rep{rep}.csv"
    )

    pa.to_csv(
        path,
        index=False
    )

print(
    "✓ Five pseudoabsence replicates saved."
)

In [ ]:
pa_audit = []

for rep, pa in pseudoabsence_replicates.items():

    pa_audit.append({
        "replicate": rep,
        "n_pseudoabsences": len(pa),
        "unique_cells":
            pa[["row", "col"]]
            .drop_duplicates()
            .shape[0],
        "min_distance_km":
            pa["distance_to_presence_km"].min(),
        "median_distance_km":
            pa["distance_to_presence_km"].median(),
        "mean_distance_km":
            pa["distance_to_presence_km"].mean(),
        "max_distance_km":
            pa["distance_to_presence_km"].max()
    })

pa_audit = pd.DataFrame(pa_audit)

pa_audit.round(2)

In [ ]:
pa1 = pseudoabsence_replicates[1].copy()

pa1_gdf = gpd.GeoDataFrame(
    pa1,
    geometry=gpd.points_from_xy(
        pa1["longitude"],
        pa1["latitude"]
    ),
    crs="EPSG:4326"
).to_crs(M_final.crs)

In [ ]:
fig, ax = plt.subplots(
    figsize=(10, 10)
)

# M75
M_final.plot(
    ax=ax,
    facecolor="0.94",
    edgecolor="0.25",
    linewidth=0.7,
    zorder=1
)

# Pseudoabsences
pa1_gdf.plot(
    ax=ax,
    markersize=5,
    alpha=0.45,
    label="Pseudoabsences",
    zorder=2
)

# Presences
occ_proj.plot(
    ax=ax,
    color="black",
    markersize=18,
    label="Presences",
    zorder=3
)

ax.set_title(
    "Presence–pseudoabsence sampling design\n"
    r"$M_{75}$ — pseudoabsence replicate 1",
    fontsize=14
)

ax.legend(
    frameon=False,
    loc="lower left"
)

ax.set_axis_off()

plt.tight_layout()
plt.show()

In [ ]:
presence_data = occ_env.copy()

presence_data["class"] = 1
presence_data["replicate"] = 1

presence_model = presence_data[
    [
        "BIO1",
        "BIO4",
        "BIO12",
        "BIO15",
        "class",
        "geometry"
    ]
].copy()

In [ ]:
pa_model = pa1_gdf[
    [
        "BIO1",
        "BIO4",
        "BIO12",
        "BIO15",
        "class",
        "geometry"
    ]
].copy()

In [ ]:
presence_model = presence_model.to_crs(
    "EPSG:5880"
)

pa_model = pa_model.to_crs(
    "EPSG:5880"
)

In [ ]:
model_data_rep1 = pd.concat(
    [
        presence_model,
        pa_model
    ],
    ignore_index=True
)

model_data_rep1 = gpd.GeoDataFrame(
    model_data_rep1,
    geometry="geometry",
    crs="EPSG:5880"
)

print(model_data_rep1["class"].value_counts())
print("Total:", len(model_data_rep1))

In [ ]:
from sklearn.neighbors import BallTree
import numpy as np
import pandas as pd

rng = np.random.default_rng(42)

n_spatial = min(5000, len(valid_indices))

idx = rng.choice(
    valid_indices,
    size=n_spatial,
    replace=False
)

# Convert flattened indices back to raster row/column
sample_rows, sample_cols = np.unravel_index(
    idx,
    valid_mask.shape
)

# Cell-center coordinates
sample_x, sample_y = rasterio.transform.xy(
    transform,
    sample_rows,
    sample_cols,
    offset="center"
)

spatial_env = pd.DataFrame({
    "longitude": sample_x,
    "latitude": sample_y
})

for variable, arr in arrays.items():
    spatial_env[variable] = arr[
        sample_rows,
        sample_cols
    ]

spatial_env.head()

In [ ]:
from scipy.sparse import csr_matrix

coords_rad = np.radians(
    spatial_env[
        ["latitude", "longitude"]
    ].to_numpy()
)

tree = BallTree(
    coords_rad,
    metric="haversine"
)

distance_classes = [
    (0, 25),
    (25, 50),
    (50, 75),
    (75, 100),
    (100, 125),
    (125, 150),
    (150, 200),
    (200, 250),
    (250, 300)
]

variables = [
    "BIO1",
    "BIO4",
    "BIO12",
    "BIO15"
]

moran_results = []

earth_radius = 6371.0088

for lower_km, upper_km in distance_classes:

    # Neighbors within upper limit
    upper_rad = upper_km / earth_radius
    lower_rad = lower_km / earth_radius

    ind_upper, dist_upper = tree.query_radius(
        coords_rad,
        r=upper_rad,
        return_distance=True
    )

    rows_w = []
    cols_w = []

    for i, (neighbors, distances) in enumerate(
        zip(ind_upper, dist_upper)
    ):

        dist_km = distances * earth_radius

        keep = (
            (dist_km > lower_km) &
            (dist_km <= upper_km)
        )

        neighbors = neighbors[keep]

        rows_w.extend([i] * len(neighbors))
        cols_w.extend(neighbors.tolist())

    # Binary spatial weights
    W = csr_matrix(
        (
            np.ones(len(rows_w)),
            (rows_w, cols_w)
        ),
        shape=(n_spatial, n_spatial)
    )

    S0 = W.sum()

    for variable in variables:

        x = spatial_env[variable].to_numpy(
            dtype=float
        )

        z = x - x.mean()

        numerator = z @ (W @ z)
        denominator = z @ z

        moran_I = (
            n_spatial / S0 *
            numerator / denominator
        )

        moran_results.append({
            "variable": variable,
            "distance_min_km": lower_km,
            "distance_max_km": upper_km,
            "moran_I": moran_I,
            "n_links": int(S0)
        })

moran_df = pd.DataFrame(moran_results)

In [ ]:
moran_table = moran_df.pivot(
    index="distance_max_km",
    columns="variable",
    values="moran_I"
)

moran_table.round(3)

In [ ]:
import numpy as np
import geopandas as gpd
from shapely.geometry import box

def create_spatial_blocks(gdf, block_size_km):

    block_size = block_size_km * 1000

    xmin, ymin, xmax, ymax = gdf.total_bounds

    xs = np.arange(
        xmin,
        xmax + block_size,
        block_size
    )

    ys = np.arange(
        ymin,
        ymax + block_size,
        block_size
    )

    polygons = []

    for x in xs[:-1]:
        for y in ys[:-1]:

            polygons.append(
                box(
                    x,
                    y,
                    x + block_size,
                    y + block_size
                )
            )

    blocks = gpd.GeoDataFrame(
        geometry=polygons,
        crs=gdf.crs
    )

    blocks["block_id"] = np.arange(
        len(blocks)
    )

    return blocks

In [ ]:
block_diagnostics = []

for size in [100, 150, 200]:

    blocks = create_spatial_blocks(
        model_data_rep1,
        size
    )

    joined = gpd.sjoin(
        model_data_rep1,
        blocks[
            ["block_id", "geometry"]
        ],
        how="left",
        predicate="within"
    )

    occupied_blocks = (
        joined["block_id"]
        .nunique()
    )

    presence_blocks = (
        joined.loc[
            joined["class"] == 1,
            "block_id"
        ]
        .nunique()
    )

    pa_blocks = (
        joined.loc[
            joined["class"] == 0,
            "block_id"
        ]
        .nunique()
    )

    block_diagnostics.append({
        "block_size_km": size,
        "occupied_blocks": occupied_blocks,
        "presence_blocks": presence_blocks,
        "pseudoabsence_blocks": pa_blocks
    })

block_diagnostics = pd.DataFrame(
    block_diagnostics
)

block_diagnostics

In [ ]:
presence_block_summary = []

for size in [100, 150, 200]:

    blocks = create_spatial_blocks(
        model_data_rep1,
        size
    )

    joined = gpd.sjoin(
        model_data_rep1,
        blocks[
            ["block_id", "geometry"]
        ],
        how="left",
        predicate="within"
    )

    counts = (
        joined[joined["class"] == 1]
        .groupby("block_id")
        .size()
    )

    presence_block_summary.append({
        "block_size_km": size,
        "n_presence_blocks": len(counts),
        "min_presences": counts.min(),
        "median_presences": counts.median(),
        "mean_presences": counts.mean(),
        "max_presences": counts.max(),
        "blocks_1_presence": (counts == 1).sum(),
        "blocks_2_or_less": (counts <= 2).sum(),
        "blocks_10_or_more": (counts >= 10).sum()
    })

presence_block_summary = pd.DataFrame(
    presence_block_summary
)

presence_block_summary.round(2)

In [ ]:
concentration = []

for size in [100, 150, 200]:

    blocks = create_spatial_blocks(
        model_data_rep1,
        size
    )

    joined = gpd.sjoin(
        model_data_rep1,
        blocks[
            ["block_id", "geometry"]
        ],
        how="left",
        predicate="within"
    )

    counts = (
        joined[joined["class"] == 1]
        .groupby("block_id")
        .size()
        .sort_values(ascending=False)
    )

    concentration.append({
        "block_size_km": size,
        "largest_block_n": counts.iloc[0],
        "largest_block_pct":
            counts.iloc[0] / 252 * 100,
        "top5_blocks_n":
            counts.head(5).sum(),
        "top5_blocks_pct":
            counts.head(5).sum() / 252 * 100
    })

pd.DataFrame(concentration).round(2)

In [ ]:
block_size_km = 150

blocks_150 = create_spatial_blocks(
    model_data_rep1,
    block_size_km
)

model_blocks = gpd.sjoin(
    model_data_rep1,
    blocks_150[
        ["block_id", "geometry"]
    ],
    how="left",
    predicate="within"
)

assert model_blocks["block_id"].notna().all()

print(
    "Occupied blocks:",
    model_blocks["block_id"].nunique()
)

In [ ]:
missing_blocks = model_blocks[
    model_blocks["block_id"].isna()
].copy()

print(
    "Points without block:",
    len(missing_blocks)
)

missing_blocks[
    ["class", "geometry"]
].head(20)

In [ ]:
block_size_km = 150
block_size = block_size_km * 1000

# Origin of the grid
xmin, ymin, xmax, ymax = model_data_rep1.total_bounds

model_blocks = model_data_rep1.copy()

model_blocks["block_col"] = np.floor(
    (model_blocks.geometry.x - xmin) /
    block_size
).astype(int)

model_blocks["block_row"] = np.floor(
    (model_blocks.geometry.y - ymin) /
    block_size
).astype(int)

# Unique block identifier
model_blocks["block_id"] = (
    model_blocks["block_row"].astype(str)
    + "_"
    + model_blocks["block_col"].astype(str)
)

print(
    "Occupied blocks:",
    model_blocks["block_id"].nunique()
)

print(
    "Missing block IDs:",
    model_blocks["block_id"].isna().sum()
)

In [ ]:
from shapely.geometry import box

block_indices = (
    model_blocks[
        ["block_row", "block_col", "block_id"]
    ]
    .drop_duplicates()
    .copy()
)

geometries = []

for _, row in block_indices.iterrows():

    x0 = xmin + row["block_col"] * block_size
    y0 = ymin + row["block_row"] * block_size

    geometries.append(
        box(
            x0,
            y0,
            x0 + block_size,
            y0 + block_size
        )
    )

blocks_150 = gpd.GeoDataFrame(
    block_indices,
    geometry=geometries,
    crs=model_data_rep1.crs
)

print(
    "Spatial blocks:",
    len(blocks_150)
)

In [ ]:
assert model_blocks["block_id"].notna().all()

assert (
    model_blocks[
        ["block_row", "block_col"]
    ]
    .notna()
    .all()
    .all()
)

print(
    "✓ Every observation assigned to a 150-km spatial block."
)

In [ ]:
block_counts = (
    model_blocks
    .groupby(["block_id", "class"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={
        0: "n_pseudoabsence",
        1: "n_presence"
    })
    .reset_index()
)

for col in [
    "n_presence",
    "n_pseudoabsence"
]:
    if col not in block_counts.columns:
        block_counts[col] = 0

block_counts["n_total"] = (
    block_counts["n_presence"] +
    block_counts["n_pseudoabsence"]
)

print(
    "Occupied blocks:",
    len(block_counts)
)

block_counts.sort_values(
    "n_presence",
    ascending=False
).head(10)

In [ ]:
n_folds = 5

# Blocks with the largest number of presences first.
# Pseudoabsences are used as secondary ordering criterion.
block_order = (
    block_counts
    .sort_values(
        ["n_presence", "n_pseudoabsence"],
        ascending=False
    )
    .reset_index(drop=True)
)

fold_stats = {
    fold: {
        "presence": 0,
        "pseudoabsence": 0,
        "blocks": []
    }
    for fold in range(1, n_folds + 1)
}

for _, row in block_order.iterrows():

    # Assign the next block to the fold
    # currently containing fewer presences.
    # Pseudoabsences break ties.
    target_fold = min(
        fold_stats,
        key=lambda f: (
            fold_stats[f]["presence"],
            fold_stats[f]["pseudoabsence"]
        )
    )

    fold_stats[target_fold]["presence"] += int(
        row["n_presence"]
    )

    fold_stats[target_fold]["pseudoabsence"] += int(
        row["n_pseudoabsence"]
    )

    # block_id is now a string
    fold_stats[target_fold]["blocks"].append(
        row["block_id"]
    )

In [ ]:
block_to_fold = {}

for fold, stats in fold_stats.items():
    for block_id in stats["blocks"]:
        block_to_fold[block_id] = fold

model_blocks["fold"] = (
    model_blocks["block_id"]
    .map(block_to_fold)
)

In [ ]:
fold_audit = (
    model_blocks
    .groupby(["fold", "class"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={
        0: "pseudoabsences",
        1: "presences"
    })
)

fold_audit["total"] = (
    fold_audit["presences"] +
    fold_audit["pseudoabsences"]
)

fold_audit["presence_pct"] = (
    fold_audit["presences"] /
    model_blocks["class"].sum()
    * 100
)

fold_audit["pseudoabsence_pct"] = (
    fold_audit["pseudoabsences"] /
    (model_blocks["class"] == 0).sum()
    * 100
)

fold_audit.round(2)

In [ ]:
# Every observation must have a fold
assert model_blocks["fold"].notna().all()

# A spatial block can belong to only one fold
assert (
    model_blocks
    .groupby("block_id")["fold"]
    .nunique()
    .max()
    == 1
)

# All five folds must exist
assert model_blocks["fold"].nunique() == 5

# Every fold must contain both classes
class_check = (
    model_blocks
    .groupby("fold")["class"]
    .nunique()
)

assert (class_check == 2).all()

print(
    "✓ Five spatial folds created."
)
print(
    "✓ No spatial block is shared between folds."
)
print(
    "✓ Every fold contains presences and pseudoabsences."
)

In [ ]:
block_fold_table = (
    model_blocks[
        ["block_id", "fold"]
    ]
    .drop_duplicates()
)

blocks_150_folds = blocks_150.merge(
    block_fold_table,
    on="block_id",
    how="left"
)

assert blocks_150_folds["fold"].notna().all()

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(
    figsize=(12, 12)
)

# Calibration area
M_final.plot(
    ax=ax,
    facecolor="lightgrey",
    edgecolor="grey",
    linewidth=0.5,
    alpha=0.35
)

# Spatial folds
blocks_150_folds.plot(
    ax=ax,
    column="fold",
    categorical=True,
    cmap="tab10",
    alpha=0.45,
    edgecolor="black",
    linewidth=0.4,
    legend=True
)

# Presences
model_blocks[
    model_blocks["class"] == 1
].plot(
    ax=ax,
    color="black",
    markersize=14,
    alpha=0.8,
    label="Presence"
)

ax.set_title(
    "Spatial cross-validation blocks (150 km)"
)

ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")

ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import numpy as np

# One row per occupied spatial block
block_spatial = blocks_150.copy()

# Centroids of the 150-km blocks
block_spatial["centroid_x"] = (
    block_spatial.geometry.centroid.x
)

block_spatial["centroid_y"] = (
    block_spatial.geometry.centroid.y
)

XY = block_spatial[
    ["centroid_x", "centroid_y"]
].to_numpy()

# Scaling makes clustering depend on spatial geometry
# rather than coordinate magnitude
XY_scaled = StandardScaler().fit_transform(XY)

kmeans = KMeans(
    n_clusters=5,
    random_state=42,
    n_init=50
)

block_spatial["fold"] = (
    kmeans.fit_predict(XY_scaled) + 1
)

block_spatial[
    ["block_id", "fold"]
].head()

In [ ]:
spatial_fold_map = dict(
    zip(
        block_spatial["block_id"],
        block_spatial["fold"]
    )
)

model_spatial = model_blocks.copy()

model_spatial["fold"] = (
    model_spatial["block_id"]
    .map(spatial_fold_map)
)

assert model_spatial["fold"].notna().all()

In [ ]:
spatial_fold_audit = (
    model_spatial
    .groupby(["fold", "class"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={
        0: "pseudoabsences",
        1: "presences"
    })
)

spatial_fold_audit["total"] = (
    spatial_fold_audit["pseudoabsences"] +
    spatial_fold_audit["presences"]
)

spatial_fold_audit["presence_pct"] = (
    spatial_fold_audit["presences"] /
    (model_spatial["class"] == 1).sum()
    * 100
)

spatial_fold_audit["pseudoabsence_pct"] = (
    spatial_fold_audit["pseudoabsences"] /
    (model_spatial["class"] == 0).sum()
    * 100
)

spatial_fold_audit.round(2)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(
    figsize=(12, 12)
)

M_final.plot(
    ax=ax,
    facecolor="lightgrey",
    edgecolor="grey",
    linewidth=0.5,
    alpha=0.30
)

block_spatial.plot(
    ax=ax,
    column="fold",
    categorical=True,
    cmap="tab10",
    alpha=0.50,
    edgecolor="black",
    linewidth=0.5,
    legend=True
)

model_spatial[
    model_spatial["class"] == 1
].plot(
    ax=ax,
    color="black",
    markersize=14,
    alpha=0.8,
    label="Presence"
)

ax.set_title(
    "Spatially grouped cross-validation folds (150-km blocks)"
)

ax.set_xlabel("Easting (m)")
ax.set_ylabel("Northing (m)")

ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.neighbors import BallTree
import numpy as np
import pandas as pd

cv_distance_results = []

for fold in sorted(model_spatial["fold"].unique()):

    test_pres = model_spatial[
        (model_spatial["fold"] == fold) &
        (model_spatial["class"] == 1)
    ].copy()

    train_pres = model_spatial[
        (model_spatial["fold"] != fold) &
        (model_spatial["class"] == 1)
    ].copy()

    # Coordinates already in EPSG:5880 → metres
    train_xy = np.column_stack([
        train_pres.geometry.x,
        train_pres.geometry.y
    ])

    test_xy = np.column_stack([
        test_pres.geometry.x,
        test_pres.geometry.y
    ])

    tree = BallTree(
        train_xy,
        metric="euclidean"
    )

    distances, _ = tree.query(
        test_xy,
        k=1
    )

    distances_km = distances[:, 0] / 1000

    cv_distance_results.append({
        "fold": fold,
        "n_test_presences": len(test_pres),
        "min_km": distances_km.min(),
        "p25_km": np.percentile(distances_km, 25),
        "median_km": np.median(distances_km),
        "mean_km": distances_km.mean(),
        "p75_km": np.percentile(distances_km, 75),
        "max_km": distances_km.max()
    })

cv_distance_audit = pd.DataFrame(
    cv_distance_results
)

cv_distance_audit.round(2)

In [ ]:
blocks_150_folds_final = block_spatial[
    ["block_id", "fold", "geometry"]
].copy()

blocks_150_folds_final.to_file(
    "../data/processed/"
    "sapajus_spatial_cv_blocks_150km.gpkg",
    layer="spatial_cv_5fold",
    driver="GPKG"
)

In [ ]:
model_spatial.to_csv(
    "../data/processed/"
    "sapajus_model_data_rep1_spatial_folds.csv",
    index=False
)

In [ ]:
spatial_fold_audit.to_csv(
    "../data/processed/"
    "sapajus_spatial_cv_fold_audit.csv"
)

cv_distance_audit.to_csv(
    "../data/processed/"
    "sapajus_spatial_cv_distance_audit.csv",
    index=False
)

print("✓ Spatial CV design saved.")

In [ ]:
cv_config = pd.DataFrame({
    "parameter": [
        "block_size_km",
        "n_folds",
        "clustering_method",
        "random_state",
        "kmeans_n_init",
        "crs"
    ],
    "value": [
        150,
        5,
        "KMeans on occupied block centroids",
        42,
        50,
        str(model_spatial.crs)
    ]
})

cv_config.to_csv(
    "../data/processed/"
    "sapajus_spatial_cv_config.csv",
    index=False
)

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import numpy as np

predictors = [
    "BIO1",
    "BIO4",
    "BIO12",
    "BIO15"
]

fold_blocks = blocks_150_folds_final.copy()

def build_model_replicate(rep):

    # -------------------------
    # PRESENCES
    # -------------------------

    pres = occ_env.copy()

    pres["class"] = 1
    pres["source"] = "presence"

    pres = pres.to_crs("EPSG:5880")

    # -------------------------
    # PSEUDOABSENCES
    # -------------------------

    pa_path = (
        "../data/processed/pseudoabsences/"
        f"sapajus_pseudoabsence_rep{rep}.csv"
    )

    pa = pd.read_csv(pa_path)

    pa_gdf = gpd.GeoDataFrame(
        pa,
        geometry=gpd.points_from_xy(
            pa["longitude"],
            pa["latitude"]
        ),
        crs="EPSG:4326"
    ).to_crs("EPSG:5880")

    pa_gdf["class"] = 0
    pa_gdf["source"] = "pseudoabsence"

    # -------------------------
    # COMBINE
    # -------------------------

    cols = (
        predictors +
        ["class", "source", "geometry"]
    )

    data = pd.concat(
        [
            pres[cols],
            pa_gdf[cols]
        ],
        ignore_index=True
    )

    data = gpd.GeoDataFrame(
        data,
        geometry="geometry",
        crs="EPSG:5880"
    )

    return data

In [ ]:
test_rep1 = build_model_replicate(1)

print(test_rep1.shape)
print()
print(test_rep1["class"].value_counts())
print()
print(test_rep1[predictors].isna().sum())

In [ ]:
block_size = 150_000

grid_xmin = xmin
grid_ymin = ymin

def assign_fixed_folds(data):

    data = data.copy()

    data["block_col"] = np.floor(
        (data.geometry.x - grid_xmin) /
        block_size
    ).astype(int)

    data["block_row"] = np.floor(
        (data.geometry.y - grid_ymin) /
        block_size
    ).astype(int)

    data["block_id"] = (
        data["block_row"].astype(str)
        + "_"
        + data["block_col"].astype(str)
    )

    fold_map = dict(
        zip(
            fold_blocks["block_id"],
            fold_blocks["fold"]
        )
    )

    data["fold"] = (
        data["block_id"]
        .map(fold_map)
    )

    return data

In [ ]:
test_rep1 = assign_fixed_folds(
    test_rep1
)

print(
    "Missing folds:",
    test_rep1["fold"].isna().sum()
)

In [ ]:
replicate_audit = []
model_replicates = {}

for rep in range(1, 6):

    data = build_model_replicate(rep)

    data = assign_fixed_folds(data)

    model_replicates[rep] = data

    replicate_audit.append({
        "replicate": rep,
        "n": len(data),
        "presences":
            (data["class"] == 1).sum(),
        "pseudoabsences":
            (data["class"] == 0).sum(),
        "missing_fold":
            data["fold"].isna().sum(),
        "n_blocks":
            data["block_id"].nunique()
    })

replicate_audit = pd.DataFrame(
    replicate_audit
)

replicate_audit

In [ ]:
new_blocks = set()

for rep, data in model_replicates.items():

    missing = data.loc[
        data["fold"].isna(),
        "block_id"
    ]

    new_blocks.update(
        missing.unique()
    )

print(
    "Unique new blocks:",
    len(new_blocks)
)

print(
    sorted(new_blocks)
)

In [ ]:
from shapely.geometry import box

new_block_records = []

for block_id in sorted(new_blocks):

    row, col = map(
        int,
        block_id.split("_")
    )

    x0 = grid_xmin + col * block_size
    y0 = grid_ymin + row * block_size

    new_block_records.append({
        "block_id": block_id,
        "block_row": row,
        "block_col": col,
        "geometry": box(
            x0,
            y0,
            x0 + block_size,
            y0 + block_size
        )
    })

new_blocks_gdf = gpd.GeoDataFrame(
    new_block_records,
    geometry="geometry",
    crs="EPSG:5880"
)

new_blocks_gdf

In [ ]:
from sklearn.neighbors import BallTree

reference_blocks = (
    fold_blocks[
        ["block_id", "fold", "geometry"]
    ]
    .copy()
)

reference_blocks["cx"] = (
    reference_blocks.geometry.centroid.x
)

reference_blocks["cy"] = (
    reference_blocks.geometry.centroid.y
)

new_blocks_gdf["cx"] = (
    new_blocks_gdf.geometry.centroid.x
)

new_blocks_gdf["cy"] = (
    new_blocks_gdf.geometry.centroid.y
)

tree = BallTree(
    reference_blocks[
        ["cx", "cy"]
    ].to_numpy(),
    metric="euclidean"
)

distances, indices = tree.query(
    new_blocks_gdf[
        ["cx", "cy"]
    ].to_numpy(),
    k=1
)

new_blocks_gdf["fold"] = (
    reference_blocks.iloc[
        indices[:, 0]
    ]["fold"]
    .to_numpy()
)

new_blocks_gdf[
    "distance_to_reference_block_km"
] = distances[:, 0] / 1000

new_blocks_gdf[
    [
        "block_id",
        "fold",
        "distance_to_reference_block_km"
    ]
]

In [ ]:
fold_blocks_complete = pd.concat(
    [
        fold_blocks,
        new_blocks_gdf[
            ["block_id", "fold", "geometry"]
        ]
    ],
    ignore_index=True
)

fold_blocks_complete = gpd.GeoDataFrame(
    fold_blocks_complete,
    geometry="geometry",
    crs="EPSG:5880"
)

assert (
    fold_blocks_complete["block_id"]
    .duplicated()
    .sum()
    == 0
)

In [ ]:
final_replicate_audit = []

for rep, data in model_replicates.items():

    final_replicate_audit.append({
        "replicate": rep,
        "n": len(data),
        "presences":
            (data["class"] == 1).sum(),
        "pseudoabsences":
            (data["class"] == 0).sum(),
        "missing_fold":
            data["fold"].isna().sum(),
        "n_folds":
            data["fold"].nunique(),
        "n_blocks":
            data["block_id"].nunique()
    })

final_replicate_audit = pd.DataFrame(
    final_replicate_audit
)

final_replicate_audit

In [ ]:
for rep, data in model_replicates.items():

    missing = data.loc[
        data["fold"].isna(),
        ["block_id", "class"]
    ]

    print(f"\n--- Replicate {rep} ---")
    print("Missing rows:", len(missing))
    print("Missing block IDs:", missing["block_id"].unique())

In [ ]:
# ============================================================
# COMPLETE THE FIXED SPATIAL-FOLD MAP
# ============================================================

# 1. Combine original blocks + newly encountered blocks
fold_blocks_complete = pd.concat(
    [
        fold_blocks[
            ["block_id", "fold", "geometry"]
        ],
        new_blocks_gdf[
            ["block_id", "fold", "geometry"]
        ]
    ],
    ignore_index=True
)

fold_blocks_complete = gpd.GeoDataFrame(
    fold_blocks_complete,
    geometry="geometry",
    crs="EPSG:5880"
)

# 2. Standardize block IDs
fold_blocks_complete["block_id"] = (
    fold_blocks_complete["block_id"]
    .astype(str)
)

# 3. Remove accidental duplicates, if any
fold_blocks_complete = (
    fold_blocks_complete
    .drop_duplicates(
        subset="block_id",
        keep="first"
    )
    .reset_index(drop=True)
)

# 4. Create complete block -> fold dictionary
complete_fold_map = dict(
    zip(
        fold_blocks_complete["block_id"],
        fold_blocks_complete["fold"]
    )
)

print(
    "Blocks in complete fold map:",
    len(complete_fold_map)
)

print(
    "Blocks without fold:",
    fold_blocks_complete["fold"].isna().sum()
)

In [ ]:
for rep, data in model_replicates.items():

    data["block_id"] = (
        data["block_id"]
        .astype(str)
    )

    data["fold"] = (
        data["block_id"]
        .map(complete_fold_map)
    )

In [ ]:
final_replicate_audit = []

for rep, data in model_replicates.items():

    final_replicate_audit.append({
        "replicate": rep,
        "n": len(data),
        "presences":
            (data["class"] == 1).sum(),
        "pseudoabsences":
            (data["class"] == 0).sum(),
        "missing_fold":
            data["fold"].isna().sum(),
        "n_folds":
            data["fold"].nunique(),
        "n_blocks":
            data["block_id"].nunique()
    })

final_replicate_audit = pd.DataFrame(
    final_replicate_audit
)

final_replicate_audit

In [ ]:
assert (
    final_replicate_audit["missing_fold"] == 0
).all(), "Some observations still have no spatial fold."

assert (
    final_replicate_audit["n_folds"] == 5
).all(), "Not all replicates contain the five spatial folds."

print(
    "✓ All five pseudoabsence replicates use the fixed spatial CV design."
)

In [ ]:
from sklearn.preprocessing import StandardScaler
import numpy as np

predictors = [
    "BIO1",
    "BIO4",
    "BIO12",
    "BIO15"
]

rep = 1
test_fold = 1

data = model_replicates[rep].copy()

train = data[
    data["fold"] != test_fold
].copy()

test = data[
    data["fold"] == test_fold
].copy()

X_train = train[predictors].to_numpy(
    dtype=np.float32
)

y_train = train["class"].to_numpy(
    dtype=np.float32
)

X_test = test[predictors].to_numpy(
    dtype=np.float32
)

y_test = test["class"].to_numpy(
    dtype=np.float32
)

print("Train:", X_train.shape)
print("Test :", X_test.shape)

print(
    "\nTrain classes:",
    np.unique(y_train, return_counts=True)
)

print(
    "Test classes:",
    np.unique(y_test, return_counts=True)
)

In [ ]:
scaler = StandardScaler()

# FIT ONLY ON TRAINING DATA
X_train_scaled = scaler.fit_transform(
    X_train
).astype(np.float32)

# Test receives the transformation learned from training
X_test_scaled = scaler.transform(
    X_test
).astype(np.float32)

In [ ]:
print("TRAIN")
print(
    "Mean:",
    np.round(
        X_train_scaled.mean(axis=0),
        3
    )
)

print(
    "SD:",
    np.round(
        X_train_scaled.std(axis=0),
        3
    )
)

print("\nTEST")
print(
    "Mean:",
    np.round(
        X_test_scaled.mean(axis=0),
        3
    )
)

print(
    "SD:",
    np.round(
        X_test_scaled.std(axis=0),
        3
    )
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train.astype(int)
)

class_weight = {
    0: weights[0],
    1: weights[1]
}

class_weight

In [ ]:
import sys

print(sys.executable)
print(sys.version)

In [ ]:
import sys
print(sys.version_info)

In [ ]:
%pip install tensorflow

In [ ]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("Python:", sys.version)
print("Devices:", tf.config.list_physical_devices())

In [ ]:
print(
    "GPU:",
    tf.config.list_physical_devices("GPU")
)

In [ ]:
fold_blocks_complete.to_file(
    "../data/processed/sapajus_spatial_cv_blocks_150km_complete.gpkg",
    layer="spatial_cv_5fold",
    driver="GPKG"
)

In [ ]:
from pathlib import Path

output_dir = Path(
    "../data/processed/neural_sdm"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

for rep, data in model_replicates.items():

    output = (
        output_dir /
        f"sapajus_neural_sdm_rep{rep}.csv"
    )

    # CSV does not need the projected geometry;
    # retain coordinates explicitly instead.
    export = data.copy()

    export["x_5880"] = export.geometry.x
    export["y_5880"] = export.geometry.y

    export = export.drop(
        columns="geometry"
    )

    export.to_csv(
        output,
        index=False
    )

print(
    "✓ Five Neural SDM datasets saved."
)

In [ ]:
for rep in range(1, 6):

    path = (
        output_dir /
        f"sapajus_neural_sdm_rep{rep}.csv"
    )

    check = pd.read_csv(path)

    print(
        f"Rep {rep}:",
        len(check),
        "rows |",
        check["fold"].nunique(),
        "folds |",
        check["fold"].isna().sum(),
        "missing folds"
    )